<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/11_Privacy_Accounting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

print("\n" + "=" * 100)
print("1. HEADER & SCOPE")
print("=" * 100)

print("\nNOTEBOOK 11 — SPP-GAN PRIVACY ACCOUNTING")
print("-" * 100)

print("Framework                       : SPP-GAN")
print("Purpose                         : Formal privacy accounting for DP-SGD")
print("Protected component             : SPP-GAN discriminator")
print("Privacy mechanism               : Differentially Private SGD (DP-SGD)")
print("Gradient representation         : Per-example discriminator gradients")
print("Gradient clipping               : Flat L2")
print("Noise mechanism                 : Gaussian")
print("Sampling mechanism              : Poisson")
print("Privacy accountant              : Rényi Differential Privacy (RDP)")

print("\n" + "-" * 100)
print("ACCOUNTING SCOPE")
print("-" * 100)

print("✓ Dataset-specific sampling rate")
print("✓ Dataset-specific noise multiplier")
print("✓ Dataset-specific training population")
print("✓ Gradient clipping norm")
print("✓ Configured training epochs")
print("✓ Dataset-specific delta")
print("✓ RDP composition over the configured DP-SGD schedule")

print("\n" + "-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

print("✓ Discriminator DP-SGD mechanism is the accounted component.")
print("✗ Generator update is not privatized.")
print("✗ Statistical guidance is not privatized.")
print("✗ Preprocessing is not privatized.")
print("✗ End-to-end SPP-GAN privacy is not established.")

print("\n" + "-" * 100)
print("SOURCE / DOWNSTREAM")
print("-" * 100)

print("Source Notebook 10             : DP-SGD mechanism and parameters")
print("Notebook 12                    : Actual SPP-GAN DP training")
print("Notebook 13                    : Synthetic data generation")

print("\n" + "-" * 100)
print("ACCOUNTING INTERPRETATION")
print("-" * 100)

print("Target epsilon                 : Configured privacy budget")
print("Accounted epsilon              : Formal RDP result for configured schedule")
print("Observed training epsilon      : To be verified during actual training")

print("\n" + "=" * 100)
print("SECTION 1 STATUS: PASS")
print("=" * 100)


1. HEADER & SCOPE

NOTEBOOK 11 — SPP-GAN PRIVACY ACCOUNTING
----------------------------------------------------------------------------------------------------
Framework                       : SPP-GAN
Purpose                         : Formal privacy accounting for DP-SGD
Protected component             : SPP-GAN discriminator
Privacy mechanism               : Differentially Private SGD (DP-SGD)
Gradient representation         : Per-example discriminator gradients
Gradient clipping               : Flat L2
Noise mechanism                 : Gaussian
Sampling mechanism              : Poisson
Privacy accountant              : Rényi Differential Privacy (RDP)

----------------------------------------------------------------------------------------------------
ACCOUNTING SCOPE
----------------------------------------------------------------------------------------------------
✓ Dataset-specific sampling rate
✓ Dataset-specific noise multiplier
✓ Dataset-specific training population
✓ Gradi

In [2]:
# ==================================================================================================
# 2. LOAD PRIVACY CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("2. LOAD PRIVACY CONFIGURATION")
print("=" * 100)

from pathlib import Path
import json
import os
import subprocess

# -----------------------------------------------------------------------------------------------
# 1. Google Drive Mount Verification
# -----------------------------------------------------------------------------------------------

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"


def is_drive_mounted(path):
    """
    Verify that the path is an actual mounted filesystem,
    rather than merely an existing local directory.
    """

    try:

        result = subprocess.run(
            ["mountpoint", "-q", str(path)],
            check=False,
        )

        return result.returncode == 0

    except Exception:

        return os.path.ismount(path)


if not is_drive_mounted(DRIVE_ROOT):

    print("Google Drive is not currently mounted.")
    print("Mounting Google Drive...")

    drive.mount(
        str(DRIVE_ROOT),
        force_remount=False,
    )


if not is_drive_mounted(DRIVE_ROOT):

    raise RuntimeError(
        "Google Drive mount verification failed.\n"
        f"Expected mounted path: {DRIVE_ROOT}"
    )


if not MYDRIVE_ROOT.exists():

    raise RuntimeError(
        "Google Drive MyDrive is unavailable after mount.\n"
        f"Expected path: {MYDRIVE_ROOT}"
    )


print(
    f"✓ Google Drive mounted  : {DRIVE_ROOT}"
)

print(
    f"✓ MyDrive available     : {MYDRIVE_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 2. Canonical Project Root
# -----------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    MYDRIVE_ROOT /
    "SPP_GAN_Research"
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "Canonical SPP-GAN project root not found:\n"
        f"{PROJECT_ROOT}"
    )


if not PROJECT_ROOT.is_dir():

    raise RuntimeError(
        "Canonical SPP-GAN project root is not a directory:\n"
        f"{PROJECT_ROOT}"
    )


print(
    f"✓ Project root          : {PROJECT_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 3. Notebook Identity
# -----------------------------------------------------------------------------------------------

NOTEBOOK_ID = "11"
NOTEBOOK_NAME = "SPP-GAN Privacy Accounting"
FRAMEWORK_NAME = "SPP-GAN"

NB10_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_10"
)

NB11_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_11"
)


if not NB10_ROOT.exists():

    raise FileNotFoundError(
        "Notebook 10 result root not found:\n"
        f"{NB10_ROOT}"
    )


print(
    f"✓ Notebook 10 root     : {NB10_ROOT}"
)

print(
    f"✓ Notebook 11 root     : {NB11_ROOT}"
)

# -----------------------------------------------------------------------------------------------
# 4. Notebook 11 Artifact Directories
# -----------------------------------------------------------------------------------------------

DIRS = {

    "root":
        NB11_ROOT,

    "configuration":
        NB11_ROOT / "configuration",

    "metadata":
        NB11_ROOT / "metadata",

    "accounting":
        NB11_ROOT / "accounting",

    "audit":
        NB11_ROOT / "audit",

    "validation":
        NB11_ROOT / "validation",

    "manifests":
        NB11_ROOT / "manifests",
}


for path in DIRS.values():

    path.mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "✓ Notebook 11 artifact directories ready"
)

# -----------------------------------------------------------------------------------------------
# 5. Notebook 10 Privacy Configuration
# -----------------------------------------------------------------------------------------------

NB10_CONFIGURATION_PATH = (
    NB10_ROOT /
    "configuration" /
    "sppgan_privacy_configuration.json"
)


if not NB10_CONFIGURATION_PATH.exists():

    raise FileNotFoundError(
        "Notebook 10 privacy configuration not found:\n"
        f"{NB10_CONFIGURATION_PATH}"
    )


if not NB10_CONFIGURATION_PATH.is_file():

    raise RuntimeError(
        "Notebook 10 privacy configuration is not a file:\n"
        f"{NB10_CONFIGURATION_PATH}"
    )


with open(
    NB10_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:

    PRIVACY_CONFIGURATION = json.load(f)


if not isinstance(
    PRIVACY_CONFIGURATION,
    dict,
):

    raise RuntimeError(
        "Notebook 10 privacy configuration must be a JSON object."
    )


print(
    f"✓ Notebook 10 configuration loaded:\n"
    f"  {NB10_CONFIGURATION_PATH}"
)

# -----------------------------------------------------------------------------------------------
# 6. Required Top-Level Configuration Schema
# -----------------------------------------------------------------------------------------------

REQUIRED_CONFIG_KEYS = {

    "configuration_version",

    "notebook",

    "name",

    "framework",

    "privacy_definition",

    "parameters",

    "dataset_parameters",

    "source_dependencies",

    "downstream",

    "created_utc",
}


missing_keys = (
    REQUIRED_CONFIG_KEYS
    -
    set(
        PRIVACY_CONFIGURATION.keys()
    )
)


if missing_keys:

    raise RuntimeError(
        "Notebook 10 privacy configuration is incomplete.\n"
        f"Missing keys: {sorted(missing_keys)}"
    )


print(
    "✓ Required top-level configuration keys present"
)

# -----------------------------------------------------------------------------------------------
# 7. Notebook Provenance Validation
# -----------------------------------------------------------------------------------------------

CONFIG_NOTEBOOK_ID = str(
    PRIVACY_CONFIGURATION[
        "notebook"
    ]
)


if CONFIG_NOTEBOOK_ID != "10":

    raise RuntimeError(
        "Notebook provenance mismatch.\n"
        f"Expected source notebook: 10\n"
        f"Recorded notebook      : {CONFIG_NOTEBOOK_ID}"
    )


CONFIG_FRAMEWORK = str(
    PRIVACY_CONFIGURATION[
        "framework"
    ]
)


if CONFIG_FRAMEWORK != FRAMEWORK_NAME:

    raise RuntimeError(
        "Framework mismatch in Notebook 10 privacy configuration.\n"
        f"Expected: {FRAMEWORK_NAME}\n"
        f"Found   : {CONFIG_FRAMEWORK}"
    )


CONFIG_NAME = str(
    PRIVACY_CONFIGURATION[
        "name"
    ]
)


if not CONFIG_NAME.strip():

    raise RuntimeError(
        "Notebook 10 configuration name is empty."
    )


CONFIG_VERSION = str(
    PRIVACY_CONFIGURATION[
        "configuration_version"
    ]
)


if not CONFIG_VERSION.strip():

    raise RuntimeError(
        "Notebook 10 configuration version is empty."
    )


print(
    "✓ Notebook 10 provenance validated"
)

print(
    f"  Source notebook       : {CONFIG_NOTEBOOK_ID}"
)

print(
    f"  Configuration version : {CONFIG_VERSION}"
)

print(
    f"  Configuration name    : {CONFIG_NAME}"
)

print(
    f"  Framework             : {CONFIG_FRAMEWORK}"
)

# -----------------------------------------------------------------------------------------------
# 8. Non-Restrictive Structural Validation
# -----------------------------------------------------------------------------------------------

# The exact container type of these fields is owned by Notebook 10.
# Notebook 11 therefore validates that they are present and non-null,
# without imposing an unsupported dictionary/list schema.

STRUCTURAL_FIELDS = {

    "privacy_definition":
        "privacy_definition",

    "parameters":
        "parameters",

    "dataset_parameters":
        "dataset_parameters",

    "source_dependencies":
        "source_dependencies",

    "downstream":
        "downstream",
}


for label, key in STRUCTURAL_FIELDS.items():

    value = PRIVACY_CONFIGURATION[key]

    if value is None:

        raise RuntimeError(
            f"Notebook 10 configuration field '{key}' is null."
        )

    if isinstance(
        value,
        (dict, list, tuple, str)
    ):

        if isinstance(
            value,
            str
        ):

            if not value.strip():

                raise RuntimeError(
                    f"Notebook 10 configuration field '{key}' is empty."
                )

        elif len(value) == 0:

            raise RuntimeError(
                f"Notebook 10 configuration field '{key}' is empty."
            )

    else:

        raise RuntimeError(
            f"Unsupported JSON structure for '{key}': "
            f"{type(value).__name__}"
        )


print(
    "✓ Notebook 10 configuration structure validated"
)

# -----------------------------------------------------------------------------------------------
# 9. Source Artifact Availability
# -----------------------------------------------------------------------------------------------

NB10_METADATA_PATH = (
    NB10_ROOT /
    "metadata" /
    "sppgan_privacy_metadata.csv"
)

NB10_REGISTRY_PATH = (
    NB10_ROOT /
    "metadata" /
    "sppgan_privacy_artifact_registry.csv"
)

NB10_COMPLETION_PATH = (
    NB10_ROOT /
    "validation" /
    "sppgan_notebook_10_completion.json"
)


SOURCE_ARTIFACT_CHECKS = {

    "privacy_configuration":
        NB10_CONFIGURATION_PATH,

    "privacy_metadata":
        NB10_METADATA_PATH,

    "privacy_artifact_registry":
        NB10_REGISTRY_PATH,

    "notebook_10_completion":
        NB10_COMPLETION_PATH,
}


missing_source_artifacts = [

    name

    for name, path
    in SOURCE_ARTIFACT_CHECKS.items()

    if not path.exists()
]


if missing_source_artifacts:

    raise FileNotFoundError(
        "Required Notebook 10 source artifacts are missing:\n"
        f"{missing_source_artifacts}"
    )


print(
    "✓ Notebook 10 source artifacts available"
)

# -----------------------------------------------------------------------------------------------
# 10. Notebook 10 Completion Provenance
# -----------------------------------------------------------------------------------------------

with open(
    NB10_COMPLETION_PATH,
    "r",
    encoding="utf-8",
) as f:

    NB10_COMPLETION = json.load(f)


if not isinstance(
    NB10_COMPLETION,
    dict,
):

    raise RuntimeError(
        "Notebook 10 completion artifact must be a JSON object."
    )


completion_framework = str(
    NB10_COMPLETION.get(
        "framework",
        "",
    )
)


if completion_framework != FRAMEWORK_NAME:

    raise RuntimeError(
        "Notebook 10 completion framework mismatch.\n"
        f"Expected: {FRAMEWORK_NAME}\n"
        f"Found   : {completion_framework}"
    )


completion_status = str(
    NB10_COMPLETION.get(
        "status",
        "",
    )
).upper()


if completion_status != "PASS":

    raise RuntimeError(
        "Notebook 10 completion status is not PASS.\n"
        f"Status: {completion_status}"
    )


print(
    "✓ Notebook 10 completion status validated: PASS"
)

# -----------------------------------------------------------------------------------------------
# 11. Configuration Summary
# -----------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LOADED CONFIGURATION SUMMARY")
print("-" * 100)

print(
    f"Source notebook               : "
    f"{CONFIG_NOTEBOOK_ID}"
)

print(
    f"Configuration version         : "
    f"{CONFIG_VERSION}"
)

print(
    f"Configuration name            : "
    f"{CONFIG_NAME}"
)

print(
    f"Framework                     : "
    f"{CONFIG_FRAMEWORK}"
)

print(
    f"Target Notebook               : "
    f"{NOTEBOOK_ID} — {NOTEBOOK_NAME}"
)

print(
    f"Privacy configuration source  : "
    f"Notebook 10"
)

# -----------------------------------------------------------------------------------------------
# 12. Structural Type Audit
# -----------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("NOTEBOOK 10 CONFIGURATION FIELD TYPES")
print("-" * 100)

for key in STRUCTURAL_FIELDS.values():

    print(
        f"{key:30s}: "
        f"{type(PRIVACY_CONFIGURATION[key]).__name__}"
    )

# -----------------------------------------------------------------------------------------------
# 13. Final Section Validation
# -----------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 2 VALIDATION")
print("-" * 100)

print(
    "✓ Actual Google Drive mount verified"
)

print(
    "✓ Canonical project root verified"
)

print(
    "✓ Notebook 10 root verified"
)

print(
    "✓ Notebook 11 root verified"
)

print(
    "✓ Notebook 10 configuration loaded"
)

print(
    "✓ Required configuration keys validated"
)

print(
    "✓ Notebook 10 provenance validated"
)

print(
    "✓ Framework identity validated"
)

print(
    "✓ Configuration field structures validated"
)

print(
    "✓ Notebook 10 source artifacts verified"
)

print(
    "✓ Notebook 10 completion status verified"
)

print(
    "✓ No privacy parameters reconstructed"
)

print(
    "✓ No training data loaded"
)

print(
    "✓ RAM-safe configuration-only operation"
)

print(
    "\nSECTION 2 STATUS: PASS"
)

print("=" * 100)


2. LOAD PRIVACY CONFIGURATION
Google Drive is not currently mounted.
Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive mounted  : /content/drive
✓ MyDrive available     : /content/drive/MyDrive
✓ Project root          : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 10 root     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10
✓ Notebook 11 root     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11
✓ Notebook 11 artifact directories ready
✓ Notebook 10 configuration loaded:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/configuration/sppgan_privacy_configuration.json
✓ Required top-level configuration keys present
✓ Notebook 10 provenance validated
  Source notebook       : 10
  Configuration version : 1.0
  Configuration name    : SPP-GAN Differential Privacy
  Framework             : SPP-GAN
✓ Notebook 10 configuration structure validated
✓ Notebook 10 source artifacts available
✓ Notebook 10 co

In [4]:
# ==================================================================================================
# 3. LOAD TRAINING METADATA
# ==================================================================================================

print("\n" + "=" * 100)
print("3. LOAD TRAINING METADATA")
print("=" * 100)

import pandas as pd
import numpy as np


# --------------------------------------------------------------------------------------------------
# 1. Notebook 10 Privacy Metadata Path
# --------------------------------------------------------------------------------------------------

NB10_METADATA_PATH = (
    NB10_ROOT /
    "metadata" /
    "sppgan_privacy_metadata.csv"
)

if not NB10_METADATA_PATH.exists():
    raise FileNotFoundError(
        "Notebook 10 privacy metadata not found:\n"
        f"{NB10_METADATA_PATH}"
    )

if not NB10_METADATA_PATH.is_file():
    raise RuntimeError(
        "Notebook 10 privacy metadata path is not a file:\n"
        f"{NB10_METADATA_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Load Metadata
# --------------------------------------------------------------------------------------------------

TRAINING_METADATA_DF = pd.read_csv(
    NB10_METADATA_PATH
)

if TRAINING_METADATA_DF.empty:
    raise RuntimeError(
        "Notebook 10 privacy metadata is empty."
    )


# --------------------------------------------------------------------------------------------------
# 3. Required Metadata Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_METADATA_COLUMNS = {
    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size",
    "sample_rate",
    "epochs",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
    "max_grad_norm",
    "noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "protected_component",
    "per_example_gradients",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "achieved_epsilon",
    "achieved_epsilon_status",
    "training_status",
    "synthetic_generation_status",
    "end_to_end_privacy_claim",
    "status",
}

missing_columns = (
    REQUIRED_METADATA_COLUMNS
    -
    set(TRAINING_METADATA_DF.columns)
)

if missing_columns:
    raise RuntimeError(
        "Notebook 10 privacy metadata is missing columns:\n"
        f"{sorted(missing_columns)}"
    )

print("✓ Required metadata columns present")


# --------------------------------------------------------------------------------------------------
# 4. Canonical Dataset Validation
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}

DATASET_IDS = (
    TRAINING_METADATA_DF["dataset"]
    .astype(str)
    .str.strip()
    .tolist()
)

if len(DATASET_IDS) != 3:
    raise RuntimeError(
        "Expected exactly three canonical dataset records.\n"
        f"Found: {len(DATASET_IDS)}"
    )

if len(set(DATASET_IDS)) != 3:
    raise RuntimeError(
        "Duplicate datasets detected:\n"
        f"{DATASET_IDS}"
    )

if set(DATASET_IDS) != EXPECTED_DATASETS:
    raise RuntimeError(
        "Canonical dataset registry mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(set(DATASET_IDS))}"
    )

print(
    f"✓ Canonical datasets validated: {DATASET_IDS}"
)


# --------------------------------------------------------------------------------------------------
# 5. Numeric Metadata Validation
# --------------------------------------------------------------------------------------------------

NUMERIC_COLUMNS = {
    "n_train": "positive",
    "target_epsilon": "positive",
    "delta": "unit_interval_open",
    "batch_size": "positive",
    "sample_rate": "unit_interval_open",
    "epochs": "positive",
    "nominal_steps_per_epoch": "positive",
    "nominal_total_steps": "positive",
    "max_grad_norm": "positive",
    "noise_multiplier": "positive",
}

for column, rule in NUMERIC_COLUMNS.items():

    numeric_values = pd.to_numeric(
        TRAINING_METADATA_DF[column],
        errors="coerce",
    )

    if numeric_values.isna().any():

        bad_rows = TRAINING_METADATA_DF.index[
            numeric_values.isna()
        ].tolist()

        raise RuntimeError(
            f"Metadata column '{column}' contains "
            f"non-numeric or missing values at rows: {bad_rows}"
        )

    if not np.isfinite(
        numeric_values.to_numpy()
    ).all():

        raise RuntimeError(
            f"Metadata column '{column}' contains "
            f"non-finite values."
        )

    if rule == "positive":

        if (numeric_values <= 0).any():
            raise RuntimeError(
                f"Metadata column '{column}' must contain "
                f"only positive values."
            )

    elif rule == "unit_interval_open":

        if (
            (numeric_values <= 0).any()
            or
            (numeric_values > 1).any()
        ):
            raise RuntimeError(
                f"Metadata column '{column}' must satisfy "
                f"0 < value <= 1."
            )

    TRAINING_METADATA_DF[column] = numeric_values

print("✓ Numeric metadata values validated")


# --------------------------------------------------------------------------------------------------
# 6. Training-Step Consistency Validation
# --------------------------------------------------------------------------------------------------

for _, row in TRAINING_METADATA_DF.iterrows():

    dataset = str(row["dataset"])

    expected_steps_per_epoch = int(
        np.ceil(
            float(row["n_train"])
            /
            float(row["batch_size"])
        )
    )

    recorded_steps_per_epoch = int(
        row["nominal_steps_per_epoch"]
    )

    if recorded_steps_per_epoch != expected_steps_per_epoch:

        raise RuntimeError(
            f"{dataset}: nominal_steps_per_epoch mismatch.\n"
            f"Expected: {expected_steps_per_epoch}\n"
            f"Recorded: {recorded_steps_per_epoch}"
        )

    expected_total_steps = (
        recorded_steps_per_epoch
        *
        int(row["epochs"])
    )

    recorded_total_steps = int(
        row["nominal_total_steps"]
    )

    if recorded_total_steps != expected_total_steps:

        raise RuntimeError(
            f"{dataset}: nominal_total_steps mismatch.\n"
            f"Expected: {expected_total_steps}\n"
            f"Recorded: {recorded_total_steps}"
        )

print(
    "✓ Training-step metadata consistency validated"
)


# --------------------------------------------------------------------------------------------------
# 7. DP Mechanism Contract Validation
# --------------------------------------------------------------------------------------------------

ACCOUNTANT_EXPECTED = "rdp"
SAMPLING_EXPECTED = "poisson"
CLIPPING_EXPECTED = "flat"
LOSS_REDUCTION_EXPECTED = "mean"
PROTECTED_COMPONENT_EXPECTED = "spp-gan discriminator"

for _, row in TRAINING_METADATA_DF.iterrows():

    dataset = str(row["dataset"])

    accountant = (
        str(row["accountant"])
        .strip()
        .lower()
    )

    sampling = (
        str(row["sampling"])
        .strip()
        .lower()
    )

    clipping = (
        str(row["clipping"])
        .strip()
        .lower()
    )

    loss_reduction = (
        str(row["loss_reduction"])
        .strip()
        .lower()
    )

    protected_component = (
        str(row["protected_component"])
        .strip()
        .lower()
    )

    if accountant != ACCOUNTANT_EXPECTED:

        raise RuntimeError(
            f"{dataset}: unsupported accountant.\n"
            f"Expected: {ACCOUNTANT_EXPECTED}\n"
            f"Found   : {accountant}"
        )

    if sampling != SAMPLING_EXPECTED:

        raise RuntimeError(
            f"{dataset}: unsupported sampling mechanism.\n"
            f"Expected: {SAMPLING_EXPECTED}\n"
            f"Found   : {sampling}"
        )

    if clipping != CLIPPING_EXPECTED:

        raise RuntimeError(
            f"{dataset}: unsupported clipping mechanism.\n"
            f"Expected: {CLIPPING_EXPECTED}\n"
            f"Found   : {clipping}"
        )

    if loss_reduction != LOSS_REDUCTION_EXPECTED:

        raise RuntimeError(
            f"{dataset}: unsupported loss reduction.\n"
            f"Expected: {LOSS_REDUCTION_EXPECTED}\n"
            f"Found   : {loss_reduction}"
        )

    if protected_component != PROTECTED_COMPONENT_EXPECTED:

        raise RuntimeError(
            f"{dataset}: incorrect protected component.\n"
            f"Expected: {PROTECTED_COMPONENT_EXPECTED}\n"
            f"Found   : {protected_component}"
        )

print("✓ DP mechanism contract validated")


# --------------------------------------------------------------------------------------------------
# 8. Privacy Boundary Validation
# --------------------------------------------------------------------------------------------------

BOOLEAN_COLUMNS = {
    "per_example_gradients",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
}

for column in BOOLEAN_COLUMNS:

    normalized = (
        TRAINING_METADATA_DF[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    invalid_values = (
        set(normalized)
        -
        {"true", "false", "1", "0"}
    )

    if invalid_values:

        raise RuntimeError(
            f"Metadata column '{column}' contains "
            f"invalid boolean values:\n"
            f"{sorted(invalid_values)}"
        )

    TRAINING_METADATA_DF[column] = normalized.map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        }
    )


if not TRAINING_METADATA_DF[
    "per_example_gradients"
].all():

    raise RuntimeError(
        "Privacy accounting requires per-example "
        "discriminator gradients."
    )


if TRAINING_METADATA_DF[
    "generator_private"
].any():

    raise RuntimeError(
        "Privacy boundary mismatch: "
        "generator_private must be False."
    )


if TRAINING_METADATA_DF[
    "statistical_guidance_private"
].any():

    raise RuntimeError(
        "Privacy boundary mismatch: "
        "statistical_guidance_private must be False."
    )


if TRAINING_METADATA_DF[
    "preprocessing_private"
].any():

    raise RuntimeError(
        "Privacy boundary mismatch: "
        "preprocessing_private must be False."
    )


if TRAINING_METADATA_DF[
    "end_to_end_privacy_claim"
].any():

    raise RuntimeError(
        "Privacy boundary violation: "
        "end_to_end_privacy_claim must be False."
    )

print("✓ Privacy boundary validated")


# --------------------------------------------------------------------------------------------------
# 9. Status Field Validation
# --------------------------------------------------------------------------------------------------

STATUS_COLUMNS = {
    "achieved_epsilon_status",
    "training_status",
    "synthetic_generation_status",
    "status",
}

for column in STATUS_COLUMNS:

    if TRAINING_METADATA_DF[column].isna().any():

        raise RuntimeError(
            f"Metadata status column '{column}' "
            f"contains missing values."
        )

    normalized = (
        TRAINING_METADATA_DF[column]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    if (normalized == "").any():

        raise RuntimeError(
            f"Metadata status column '{column}' "
            f"contains empty values."
        )

    TRAINING_METADATA_DF[column] = normalized


if (
    TRAINING_METADATA_DF[
        "synthetic_generation_status"
    ]
    .eq("RUNNING")
    .any()
):

    raise RuntimeError(
        "Synthetic generation is unexpectedly marked "
        "RUNNING in Notebook 10 metadata."
    )

print("✓ Metadata status fields validated")


# --------------------------------------------------------------------------------------------------
# 10. Observed Training Epsilon Validation
# --------------------------------------------------------------------------------------------------

# Notebook 10 may contain an empty achieved_epsilon field because actual
# DP training has not yet been performed. Notebook 11 must not require
# observed training epsilon at this stage.
#
# Formal configured-schedule epsilon is calculated independently by
# Notebook 11 using the validated DP configuration and accounting schedule.
#
# Actual observed training epsilon is verified later by Notebook 12
# during the real DP-SGD training run.

ACHIEVED_EPSILON_VALUES = pd.to_numeric(
    TRAINING_METADATA_DF["achieved_epsilon"],
    errors="coerce",
)

if ACHIEVED_EPSILON_VALUES.notna().any():

    valid_values = ACHIEVED_EPSILON_VALUES.dropna()

    if not np.isfinite(
        valid_values.to_numpy()
    ).all():

        raise RuntimeError(
            "Notebook 10 achieved_epsilon contains "
            "non-finite populated values."
        )

    if (valid_values < 0).any():

        raise RuntimeError(
            "Notebook 10 achieved_epsilon contains "
            "negative populated values."
        )

    TRAINING_METADATA_DF[
        "achieved_epsilon"
    ] = ACHIEVED_EPSILON_VALUES

    print(
        "✓ Populated achieved_epsilon values validated"
    )

else:

    print(
        "✓ achieved_epsilon is not yet populated; "
        "observed training epsilon is deferred to Notebook 12"
    )


# --------------------------------------------------------------------------------------------------
# 11. Dataset-Level Metadata Summary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("VALIDATED TRAINING METADATA")
print("-" * 100)

for _, row in TRAINING_METADATA_DF.iterrows():

    print(f"\nDataset                  : {row['dataset']}")
    print(f"Training rows            : {int(row['n_train'])}")
    print(f"Target epsilon           : {row['target_epsilon']:.6f}")
    print(f"Delta                    : {row['delta']:.2e}")
    print(f"Batch size               : {int(row['batch_size'])}")
    print(f"Sample rate              : {row['sample_rate']:.8f}")
    print(f"Epochs                   : {int(row['epochs'])}")

    print(
        f"Steps / epoch           : "
        f"{int(row['nominal_steps_per_epoch'])}"
    )

    print(
        f"Total DP steps          : "
        f"{int(row['nominal_total_steps'])}"
    )

    print(
        f"Max grad norm            : "
        f"{row['max_grad_norm']:.6f}"
    )

    print(
        f"Noise multiplier         : "
        f"{row['noise_multiplier']:.6f}"
    )

    print(
        f"Accountant               : "
        f"{row['accountant']}"
    )

    print(
        f"Sampling                 : "
        f"{row['sampling']}"
    )

    print(
        f"Clipping                 : "
        f"{row['clipping']}"
    )

    print(
        f"Loss reduction           : "
        f"{row['loss_reduction']}"
    )

    print(
        f"Protected component      : "
        f"{row['protected_component']}"
    )

    print(
        f"Per-example gradients   : "
        f"{bool(row['per_example_gradients'])}"
    )

    print(
        f"Generator private       : "
        f"{bool(row['generator_private'])}"
    )

    print(
        f"Statistical guidance    : "
        f"{bool(row['statistical_guidance_private'])}"
    )

    print(
        f"Preprocessing private   : "
        f"{bool(row['preprocessing_private'])}"
    )

    print(
        f"End-to-end claim        : "
        f"{bool(row['end_to_end_privacy_claim'])}"
    )

    print(
        f"Metadata status         : "
        f"{row['status']}"
    )


# --------------------------------------------------------------------------------------------------
# 12. Final Section Validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 3 VALIDATION")
print("-" * 100)

print(
    f"✓ Metadata rows                 : "
    f"{len(TRAINING_METADATA_DF)}"
)

print(
    f"✓ Canonical datasets             : "
    f"{DATASET_IDS}"
)

print("✓ Required metadata schema       : PASS")
print("✓ Numeric metadata values        : PASS")
print("✓ Training-step consistency      : PASS")
print("✓ DP mechanism contract          : PASS")
print("✓ Privacy boundary               : PASS")
print("✓ Status fields                  : PASS")
print("✓ Achieved epsilon fields        : PASS")
print("✓ No training data loaded")
print("✓ No privacy parameters reconstructed")
print("✓ Notebook 10 remains authoritative")

print("\nSECTION 3 STATUS: PASS")
print("=" * 100)


3. LOAD TRAINING METADATA
✓ Required metadata columns present
✓ Canonical datasets validated: ['adult_income', 'bank_marketing', 'diabetes_130us']
✓ Numeric metadata values validated
✓ Training-step metadata consistency validated
✓ DP mechanism contract validated
✓ Privacy boundary validated
✓ Metadata status fields validated
✓ achieved_epsilon is not yet populated; observed training epsilon is deferred to Notebook 12

----------------------------------------------------------------------------------------------------
VALIDATED TRAINING METADATA
----------------------------------------------------------------------------------------------------

Dataset                  : adult_income
Training rows            : 34189
Target epsilon           : 5.000000
Delta                    : 1.00e-05
Batch size               : 128
Sample rate              : 0.00374389
Epochs                   : 300
Steps / epoch           : 268
Total DP steps          : 80400
Max grad norm            : 1.000000
No

In [5]:
# ==================================================================================================
# 4. LOAD DP PARAMETERS
# ==================================================================================================

print("\n" + "=" * 100)
print("4. LOAD DP PARAMETERS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Load Authoritative Notebook 10 Privacy Parameters
# --------------------------------------------------------------------------------------------------

PRIVACY_PARAMETERS = (
    PRIVACY_CONFIGURATION[
        "parameters"
    ]
)

if not isinstance(PRIVACY_PARAMETERS, dict):
    raise RuntimeError(
        "Notebook 10 privacy parameters must be a dictionary."
    )


REQUIRED_GLOBAL_DP_PARAMETERS = {
    "target_epsilon",
    "max_grad_norm",
    "batch_size",
    "epochs",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "grad_sample_mode",
}

missing_global_parameters = (
    REQUIRED_GLOBAL_DP_PARAMETERS
    -
    set(PRIVACY_PARAMETERS.keys())
)

if missing_global_parameters:
    raise RuntimeError(
        "Notebook 10 privacy parameters are incomplete.\n"
        f"Missing: {sorted(missing_global_parameters)}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Read Global DP Parameters
# --------------------------------------------------------------------------------------------------

TARGET_EPSILON = float(
    PRIVACY_PARAMETERS[
        "target_epsilon"
    ]
)

MAX_GRAD_NORM = float(
    PRIVACY_PARAMETERS[
        "max_grad_norm"
    ]
)

DP_BATCH_SIZE = int(
    PRIVACY_PARAMETERS[
        "batch_size"
    ]
)

DP_EPOCHS = int(
    PRIVACY_PARAMETERS[
        "epochs"
    ]
)

ACCOUNTANT = (
    str(
        PRIVACY_PARAMETERS[
            "accountant"
        ]
    )
    .strip()
    .lower()
)

SAMPLING_MECHANISM = (
    str(
        PRIVACY_PARAMETERS[
            "sampling"
        ]
    )
    .strip()
    .lower()
)

CLIPPING_MECHANISM = (
    str(
        PRIVACY_PARAMETERS[
            "clipping"
        ]
    )
    .strip()
    .lower()
)

LOSS_REDUCTION = (
    str(
        PRIVACY_PARAMETERS[
            "loss_reduction"
        ]
    )
    .strip()
    .lower()
)

GRAD_SAMPLE_MODE = (
    str(
        PRIVACY_PARAMETERS[
            "grad_sample_mode"
        ]
    )
    .strip()
    .lower()
)


# --------------------------------------------------------------------------------------------------
# 3. Create DP Parameter DataFrame
# --------------------------------------------------------------------------------------------------

PARAMETER_DF = TRAINING_METADATA_DF.copy()

REQUIRED_PARAMETER_COLUMNS = {
    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size",
    "sample_rate",
    "epochs",
    "max_grad_norm",
    "noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
}

missing_parameter_columns = (
    REQUIRED_PARAMETER_COLUMNS
    -
    set(PARAMETER_DF.columns)
)

if missing_parameter_columns:
    raise RuntimeError(
        "Required DP parameter columns are missing:\n"
        f"{sorted(missing_parameter_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Validate Global DP Parameter Values
# --------------------------------------------------------------------------------------------------

if TARGET_EPSILON <= 0:
    raise ValueError(
        "Target epsilon must be positive."
    )

if MAX_GRAD_NORM <= 0:
    raise ValueError(
        "Maximum gradient norm must be positive."
    )

if DP_BATCH_SIZE <= 0:
    raise ValueError(
        "DP batch size must be positive."
    )

if DP_EPOCHS <= 0:
    raise ValueError(
        "DP epochs must be positive."
    )

if ACCOUNTANT != "rdp":
    raise RuntimeError(
        "Unsupported privacy accountant.\n"
        f"Expected: rdp\n"
        f"Found   : {ACCOUNTANT}"
    )

if SAMPLING_MECHANISM != "poisson":
    raise RuntimeError(
        "Unsupported sampling mechanism.\n"
        f"Expected: poisson\n"
        f"Found   : {SAMPLING_MECHANISM}"
    )

if CLIPPING_MECHANISM != "flat":
    raise RuntimeError(
        "Unsupported clipping mechanism.\n"
        f"Expected: flat\n"
        f"Found   : {CLIPPING_MECHANISM}"
    )

if LOSS_REDUCTION != "mean":
    raise RuntimeError(
        "Unsupported loss reduction.\n"
        f"Expected: mean\n"
        f"Found   : {LOSS_REDUCTION}"
    )

if not GRAD_SAMPLE_MODE.strip():
    raise RuntimeError(
        "Grad-sample mode cannot be empty."
    )


# --------------------------------------------------------------------------------------------------
# 5. Cross-Check Global Parameters Against Section 3 Metadata
# --------------------------------------------------------------------------------------------------

for _, row in PARAMETER_DF.iterrows():

    dataset = str(
        row["dataset"]
    )

    # ----------------------------------------------------------------------------------------------
    # Target epsilon
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        float(row["target_epsilon"]),
        TARGET_EPSILON,
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"{dataset}: target epsilon mismatch.\n"
            f"Notebook 10 parameters : {TARGET_EPSILON}\n"
            f"Metadata                : {row['target_epsilon']}"
        )

    # ----------------------------------------------------------------------------------------------
    # Batch size
    # ----------------------------------------------------------------------------------------------

    if int(row["batch_size"]) != DP_BATCH_SIZE:
        raise RuntimeError(
            f"{dataset}: batch size mismatch.\n"
            f"Notebook 10 parameters : {DP_BATCH_SIZE}\n"
            f"Metadata                : {int(row['batch_size'])}"
        )

    # ----------------------------------------------------------------------------------------------
    # Epochs
    # ----------------------------------------------------------------------------------------------

    if int(row["epochs"]) != DP_EPOCHS:
        raise RuntimeError(
            f"{dataset}: epoch mismatch.\n"
            f"Notebook 10 parameters : {DP_EPOCHS}\n"
            f"Metadata                : {int(row['epochs'])}"
        )

    # ----------------------------------------------------------------------------------------------
    # Maximum gradient norm
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        float(row["max_grad_norm"]),
        MAX_GRAD_NORM,
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError(
            f"{dataset}: max_grad_norm mismatch.\n"
            f"Notebook 10 parameters : {MAX_GRAD_NORM}\n"
            f"Metadata                : {row['max_grad_norm']}"
        )

    # ----------------------------------------------------------------------------------------------
    # Accountant
    # ----------------------------------------------------------------------------------------------

    metadata_accountant = (
        str(row["accountant"])
        .strip()
        .lower()
    )

    if metadata_accountant != ACCOUNTANT:
        raise RuntimeError(
            f"{dataset}: accountant mismatch.\n"
            f"Notebook 10 parameters : {ACCOUNTANT}\n"
            f"Metadata                : {metadata_accountant}"
        )

    # ----------------------------------------------------------------------------------------------
    # Sampling
    # ----------------------------------------------------------------------------------------------

    metadata_sampling = (
        str(row["sampling"])
        .strip()
        .lower()
    )

    if metadata_sampling != SAMPLING_MECHANISM:
        raise RuntimeError(
            f"{dataset}: sampling mechanism mismatch.\n"
            f"Notebook 10 parameters : {SAMPLING_MECHANISM}\n"
            f"Metadata                : {metadata_sampling}"
        )

    # ----------------------------------------------------------------------------------------------
    # Clipping
    # ----------------------------------------------------------------------------------------------

    metadata_clipping = (
        str(row["clipping"])
        .strip()
        .lower()
    )

    if metadata_clipping != CLIPPING_MECHANISM:
        raise RuntimeError(
            f"{dataset}: clipping mechanism mismatch.\n"
            f"Notebook 10 parameters : {CLIPPING_MECHANISM}\n"
            f"Metadata                : {metadata_clipping}"
        )

    # ----------------------------------------------------------------------------------------------
    # Loss reduction
    # ----------------------------------------------------------------------------------------------

    metadata_loss_reduction = (
        str(row["loss_reduction"])
        .strip()
        .lower()
    )

    if metadata_loss_reduction != LOSS_REDUCTION:
        raise RuntimeError(
            f"{dataset}: loss reduction mismatch.\n"
            f"Notebook 10 parameters : {LOSS_REDUCTION}\n"
            f"Metadata                : {metadata_loss_reduction}"
        )


print(
    "✓ Global DP parameters loaded from Notebook 10"
)

print(
    "✓ Section 3 metadata consistency validated"
)


# --------------------------------------------------------------------------------------------------
# 6. Validate Dataset-Specific Noise Multipliers
# --------------------------------------------------------------------------------------------------

noise_values = pd.to_numeric(
    PARAMETER_DF["noise_multiplier"],
    errors="coerce",
)

if noise_values.isna().any():
    raise RuntimeError(
        "Dataset-specific noise_multiplier contains "
        "missing or non-numeric values."
    )

if not np.isfinite(
    noise_values.to_numpy()
).all():
    raise RuntimeError(
        "Dataset-specific noise_multiplier contains "
        "non-finite values."
    )

if (noise_values <= 0).any():
    raise RuntimeError(
        "Dataset-specific noise_multiplier must be positive."
    )

PARAMETER_DF[
    "noise_multiplier"
] = noise_values

print(
    "✓ Dataset-specific noise multipliers validated"
)


# --------------------------------------------------------------------------------------------------
# 7. Validate Dataset-Specific Sampling Rates
# --------------------------------------------------------------------------------------------------

sample_rate_values = pd.to_numeric(
    PARAMETER_DF["sample_rate"],
    errors="coerce",
)

if sample_rate_values.isna().any():
    raise RuntimeError(
        "Dataset-specific sample_rate contains "
        "missing or non-numeric values."
    )

if (
    (sample_rate_values <= 0).any()
    or
    (sample_rate_values > 1).any()
):
    raise RuntimeError(
        "Dataset-specific sample_rate must satisfy "
        "0 < sample_rate <= 1."
    )

PARAMETER_DF[
    "sample_rate"
] = sample_rate_values

print(
    "✓ Dataset-specific sampling rates validated"
)


# --------------------------------------------------------------------------------------------------
# 8. Validate Sampling Rate Against Training Population
# --------------------------------------------------------------------------------------------------

for _, row in PARAMETER_DF.iterrows():

    dataset = str(
        row["dataset"]
    )

    expected_sample_rate = (
        float(DP_BATCH_SIZE)
        /
        float(row["n_train"])
    )

    recorded_sample_rate = float(
        row["sample_rate"]
    )

    if not np.isclose(
        recorded_sample_rate,
        expected_sample_rate,
        rtol=0.0,
        atol=1e-8,
    ):
        raise RuntimeError(
            f"{dataset}: sample_rate mismatch.\n"
            f"Expected batch_size / n_train: "
            f"{expected_sample_rate:.12f}\n"
            f"Recorded sample_rate          : "
            f"{recorded_sample_rate:.12f}"
        )

print(
    "✓ Dataset-specific sampling rates "
    "are consistent with training populations"
)


# --------------------------------------------------------------------------------------------------
# 9. Final DP Parameter Summary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("LOADED DP PARAMETERS")
print("-" * 100)

print(
    f"Target epsilon       : {TARGET_EPSILON}"
)

print(
    f"Maximum gradient norm: {MAX_GRAD_NORM}"
)

print(
    f"DP batch size        : {DP_BATCH_SIZE}"
)

print(
    f"DP epochs            : {DP_EPOCHS}"
)

print(
    f"Accountant           : {ACCOUNTANT}"
)

print(
    f"Sampling             : {SAMPLING_MECHANISM}"
)

print(
    f"Clipping             : {CLIPPING_MECHANISM}"
)

print(
    f"Loss reduction       : {LOSS_REDUCTION}"
)

print(
    f"Grad-sample mode     : {GRAD_SAMPLE_MODE}"
)


print("\n" + "-" * 100)
print("DATASET-SPECIFIC DP PARAMETERS")
print("-" * 100)

for _, row in PARAMETER_DF.iterrows():

    print(
        f"{row['dataset']:20s} | "
        f"N={int(row['n_train']):6d} | "
        f"q={float(row['sample_rate']):.8f} | "
        f"sigma={float(row['noise_multiplier']):.8f} | "
        f"steps={int(row['nominal_total_steps']):7d}"
    )


# --------------------------------------------------------------------------------------------------
# 10. Final Section Validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 4 VALIDATION")
print("-" * 100)

print(
    "✓ Notebook 10 parameters loaded"
)

print(
    "✓ Required global DP parameters present"
)

print(
    "✓ Global DP parameter values validated"
)

print(
    "✓ Section 3 metadata consistency validated"
)

print(
    "✓ Dataset-specific noise multipliers validated"
)

print(
    "✓ Dataset-specific sampling rates validated"
)

print(
    "✓ Sampling rates consistent with training populations"
)

print(
    "✓ RDP / Poisson / flat / mean contract validated"
)

print(
    "✓ No training data loaded"
)

print(
    "✓ No privacy parameters reconstructed"
)

print(
    "✓ Notebook 10 remains authoritative"
)

print("\nSECTION 4 STATUS: PASS")
print("=" * 100)


4. LOAD DP PARAMETERS
✓ Global DP parameters loaded from Notebook 10
✓ Section 3 metadata consistency validated
✓ Dataset-specific noise multipliers validated
✓ Dataset-specific sampling rates validated
✓ Dataset-specific sampling rates are consistent with training populations

----------------------------------------------------------------------------------------------------
LOADED DP PARAMETERS
----------------------------------------------------------------------------------------------------
Target epsilon       : 5.0
Maximum gradient norm: 1.0
DP batch size        : 128
DP epochs            : 300
Accountant           : rdp
Sampling             : poisson
Clipping             : flat
Loss reduction       : mean
Grad-sample mode     : hooks

----------------------------------------------------------------------------------------------------
DATASET-SPECIFIC DP PARAMETERS
----------------------------------------------------------------------------------------------------
adult_income

In [6]:
# ==================================================================================================
# 5. VALIDATE SAMPLING PARAMETERS
# ==================================================================================================

print("\n" + "=" * 100)
print("5. VALIDATE SAMPLING PARAMETERS")
print("=" * 100)

if SAMPLING_MECHANISM != "poisson":
    raise RuntimeError(
        "Sampling mechanism must be Poisson."
    )

SAMPLING_ROWS = []

for row in PARAMETER_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    n_train = int(
        row.n_train
    )

    sample_rate = float(
        row.sample_rate
    )

    expected_sample_rate = (
        DP_BATCH_SIZE /
        n_train
    )

    sample_rate_error = abs(
        sample_rate -
        expected_sample_rate
    )

    expected_batch_size = (
        n_train *
        sample_rate
    )

    valid_probability = (
        0 <
        sample_rate
        <= 1
    )

    valid_batch = (
        0 <
        DP_BATCH_SIZE
        <= n_train
    )

    rate_pass = (
        sample_rate_error
        <=
        1e-12
    )

    expectation_pass = (
        abs(
            expected_batch_size -
            DP_BATCH_SIZE
        )
        <=
        1e-9
    )

    status = (
        "PASS"
        if all([
            valid_probability,
            valid_batch,
            rate_pass,
            expectation_pass,
        ])
        else "FAIL"
    )

    SAMPLING_ROWS.append({

        "dataset":
            dataset_id,

        "n_train":
            n_train,

        "batch_size":
            DP_BATCH_SIZE,

        "configured_sample_rate":
            sample_rate,

        "expected_sample_rate":
            expected_sample_rate,

        "sample_rate_error":
            sample_rate_error,

        "expected_poisson_batch_size":
            expected_batch_size,

        "sampling":
            SAMPLING_MECHANISM,

        "status":
            status,
    })

SAMPLING_VALIDATION_DF = pd.DataFrame(
    SAMPLING_ROWS
)

if not SAMPLING_VALIDATION_DF[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Sampling parameter validation failed."
    )

SAMPLING_VALIDATION_PATH = (
    DIRS["validation"] /
    "sampling_parameter_validation.csv"
)

SAMPLING_VALIDATION_DF.to_csv(
    SAMPLING_VALIDATION_PATH,
    index=False,
)

display(
    SAMPLING_VALIDATION_DF
)

print(
    f"✓ Sampling validation saved:\n"
    f"  {SAMPLING_VALIDATION_PATH}"
)

print(
    "SECTION 5 STATUS: PASS"
)


5. VALIDATE SAMPLING PARAMETERS


,dataset,n_train,batch_size,configured_sample_rate,expected_sample_rate,sample_rate_error,expected_poisson_batch_size,sampling,status
0,adult_income,34189,128,0.003744,0.003744,6.158268e-17,128.0,poisson,PASS
1,bank_marketing,31647,128,0.004045,0.004045,1.474515e-17,128.0,poisson,PASS
2,diabetes_130us,71236,128,0.001797,0.001797,3.035766e-18,128.0,poisson,PASS


✓ Sampling validation saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/validation/sampling_parameter_validation.csv
SECTION 5 STATUS: PASS


In [7]:
# ==================================================================================================
# 6. DETERMINE TRAINING STEPS
# ==================================================================================================

print("\n" + "=" * 100)
print("6. DETERMINE TRAINING STEPS")
print("=" * 100)

import math

# --------------------------------------------------------------------------------------------------
# 6.1 Validate the sampling configuration required for this schedule
# --------------------------------------------------------------------------------------------------

if SAMPLING_MECHANISM != "poisson":
    raise RuntimeError(
        "Training-step determination requires Poisson sampling."
    )

if DP_BATCH_SIZE <= 0:
    raise RuntimeError(
        "DP batch size must be positive."
    )

if DP_EPOCHS <= 0:
    raise RuntimeError(
        "DP epochs must be positive."
    )

# --------------------------------------------------------------------------------------------------
# 6.2 Determine nominal training schedule
#
# IMPORTANT:
#   q = B / N is the Poisson sampling probability.
#
#   q does NOT imply:
#       steps_per_epoch = 1 / q
#
#   The nominal epoch schedule is defined using the conventional dataset/batch relationship:
#
#       steps_per_epoch = ceil(N_train / batch_size)
#
#   The total nominal DP-SGD schedule is:
#
#       total_steps = epochs * steps_per_epoch
#
#   This is the schedule that must subsequently be reconciled with the actual private
#   discriminator optimizer updates performed in Notebook 12.
# --------------------------------------------------------------------------------------------------

STEP_ROWS = []

for row in PARAMETER_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    n_train = int(
        row.n_train
    )

    sample_rate = float(
        row.sample_rate
    )

    metadata_steps_per_epoch = int(
        round(
            float(
                row.nominal_steps_per_epoch
            )
        )
    )

    metadata_total_steps = int(
        round(
            float(
                row.nominal_total_steps
            )
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Poisson sampling probability
    # ----------------------------------------------------------------------------------------------

    expected_sample_rate = (
        DP_BATCH_SIZE /
        n_train
    )

    sample_rate_error = abs(
        sample_rate -
        expected_sample_rate
    )

    # ----------------------------------------------------------------------------------------------
    # Nominal epoch schedule
    # ----------------------------------------------------------------------------------------------

    expected_steps_per_epoch = int(
        math.ceil(
            n_train /
            DP_BATCH_SIZE
        )
    )

    expected_total_steps = (
        DP_EPOCHS *
        expected_steps_per_epoch
    )

    # ----------------------------------------------------------------------------------------------
    # Metadata consistency
    # ----------------------------------------------------------------------------------------------

    steps_per_epoch_error = abs(
        expected_steps_per_epoch -
        metadata_steps_per_epoch
    )

    total_steps_error = abs(
        expected_total_steps -
        metadata_total_steps
    )

    # ----------------------------------------------------------------------------------------------
    # Basic validity checks
    # ----------------------------------------------------------------------------------------------

    valid_n_train = (
        n_train > 0
    )

    valid_batch_size = (
        0 <
        DP_BATCH_SIZE
        <=
        n_train
    )

    valid_epochs = (
        DP_EPOCHS > 0
    )

    valid_sample_rate = (
        0 <
        sample_rate
        <= 1
    )

    sample_rate_pass = (
        sample_rate_error
        <=
        1e-12
    )

    steps_per_epoch_pass = (
        steps_per_epoch_error
        ==
        0
    )

    total_steps_pass = (
        total_steps_error
        ==
        0
    )

    status = (
        "PASS"
        if all([
            valid_n_train,
            valid_batch_size,
            valid_epochs,
            valid_sample_rate,
            sample_rate_pass,
            steps_per_epoch_pass,
            total_steps_pass,
        ])
        else "FAIL"
    )

    STEP_ROWS.append({

        "dataset":
            dataset_id,

        "n_train":
            n_train,

        "batch_size":
            DP_BATCH_SIZE,

        "epochs":
            DP_EPOCHS,

        "sample_rate":
            sample_rate,

        "expected_sample_rate":
            expected_sample_rate,

        "sample_rate_error":
            sample_rate_error,

        "expected_steps_per_epoch":
            expected_steps_per_epoch,

        "metadata_steps_per_epoch":
            metadata_steps_per_epoch,

        "steps_per_epoch_error":
            steps_per_epoch_error,

        "expected_total_steps":
            expected_total_steps,

        "metadata_total_steps":
            metadata_total_steps,

        "total_steps_error":
            total_steps_error,

        "schedule_basis":
            "ceil(n_train / batch_size)",

        "sampling":
            SAMPLING_MECHANISM,

        "status":
            status,
    })


STEP_DF = pd.DataFrame(
    STEP_ROWS
)

# --------------------------------------------------------------------------------------------------
# 6.3 Require complete validation success
# --------------------------------------------------------------------------------------------------

if STEP_DF.empty:
    raise RuntimeError(
        "Training-step validation produced no rows."
    )

if not STEP_DF[
    "status"
].eq("PASS").all():

    display(
        STEP_DF
    )

    raise RuntimeError(
        "Training-step validation failed."
    )

# --------------------------------------------------------------------------------------------------
# 6.4 Verify canonical total-step schedule explicitly
# --------------------------------------------------------------------------------------------------

EXPECTED_CANONICAL_STEPS = {
    "adult_income": 80400,
    "bank_marketing": 74400,
    "diabetes_130us": 167100,
}

for dataset_id, expected_total in EXPECTED_CANONICAL_STEPS.items():

    matching_rows = STEP_DF[
        STEP_DF["dataset"]
        ==
        dataset_id
    ]

    if len(matching_rows) != 1:
        raise RuntimeError(
            f"Expected exactly one training-step row for dataset: "
            f"{dataset_id}"
        )

    observed_total = int(
        matching_rows.iloc[0][
            "expected_total_steps"
        ]
    )

    if observed_total != expected_total:
        raise RuntimeError(
            f"Canonical training-step mismatch for "
            f"{dataset_id}: "
            f"expected {expected_total}, "
            f"observed {observed_total}."
        )

# --------------------------------------------------------------------------------------------------
# 6.5 Persist validation artifact
# --------------------------------------------------------------------------------------------------

STEP_VALIDATION_PATH = (
    DIRS["validation"] /
    "training_step_validation.csv"
)

STEP_DF.to_csv(
    STEP_VALIDATION_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 6.6 Display validation results
# --------------------------------------------------------------------------------------------------

display(
    STEP_DF[
        [
            "dataset",
            "n_train",
            "batch_size",
            "epochs",
            "sample_rate",
            "expected_steps_per_epoch",
            "metadata_steps_per_epoch",
            "expected_total_steps",
            "metadata_total_steps",
            "schedule_basis",
            "sampling",
            "status",
        ]
    ]
)

# --------------------------------------------------------------------------------------------------
# 6.7 Final section summary
# --------------------------------------------------------------------------------------------------

print(
    "✓ Poisson sampling mechanism validated."
)

print(
    "✓ Sampling rate q = batch_size / n_train validated."
)

print(
    "✓ Nominal steps per epoch = ceil(n_train / batch_size) validated."
)

print(
    "✓ Nominal total steps = epochs × steps_per_epoch validated."
)

print(
    "✓ Canonical total-step schedule validated:"
)

for row in STEP_DF.itertuples(
    index=False
):

    print(
        f"  {row.dataset:<20} "
        f"steps/epoch={int(row.expected_steps_per_epoch):>4} "
        f"total_steps={int(row.expected_total_steps):>7}"
    )

print(
    f"✓ Training-step validation saved:\n"
    f"  {STEP_VALIDATION_PATH}"
)

print(
    "SECTION 6 STATUS: PASS"
)


6. DETERMINE TRAINING STEPS


,dataset,n_train,batch_size,epochs,sample_rate,expected_steps_per_epoch,metadata_steps_per_epoch,expected_total_steps,metadata_total_steps,schedule_basis,sampling,status
0,adult_income,34189,128,300,0.003744,268,268,80400,80400,ceil(n_train / batch_size),poisson,PASS
1,bank_marketing,31647,128,300,0.004045,248,248,74400,74400,ceil(n_train / batch_size),poisson,PASS
2,diabetes_130us,71236,128,300,0.001797,557,557,167100,167100,ceil(n_train / batch_size),poisson,PASS


✓ Poisson sampling mechanism validated.
✓ Sampling rate q = batch_size / n_train validated.
✓ Nominal steps per epoch = ceil(n_train / batch_size) validated.
✓ Nominal total steps = epochs × steps_per_epoch validated.
✓ Canonical total-step schedule validated:
  adult_income         steps/epoch= 268 total_steps=  80400
  bank_marketing       steps/epoch= 248 total_steps=  74400
  diabetes_130us       steps/epoch= 557 total_steps= 167100
✓ Training-step validation saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/validation/training_step_validation.csv
SECTION 6 STATUS: PASS


In [9]:
# ==================================================================================================
# 7. DETERMINE NOISE PARAMETERS
# ==================================================================================================

print("\n" + "=" * 100)
print("7. DETERMINE NOISE PARAMETERS")
print("=" * 100)

import sys
import math
import subprocess
import importlib
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 7.1 Validate frozen privacy-accounting contract
# --------------------------------------------------------------------------------------------------

if ACCOUNTANT != "rdp":
    raise RuntimeError(
        "Notebook 11 noise calibration requires the RDP accountant."
    )

if SAMPLING_MECHANISM != "poisson":
    raise RuntimeError(
        "Notebook 11 noise calibration requires Poisson sampling."
    )

if CLIPPING_MECHANISM != "flat":
    raise RuntimeError(
        "Notebook 11 noise calibration requires flat clipping."
    )

if LOSS_REDUCTION != "mean":
    raise RuntimeError(
        "Notebook 11 noise calibration requires mean loss reduction."
    )

if TARGET_EPSILON <= 0:
    raise RuntimeError(
        "Target epsilon must be positive."
    )


# --------------------------------------------------------------------------------------------------
# 7.2 Ensure the authoritative Notebook 10 Opacus version is available
# --------------------------------------------------------------------------------------------------

EXPECTED_OPACUS_VERSION = "1.6.0"

try:

    import opacus

except ImportError:

    print(
        "⚠ Opacus is not available in the current runtime."
    )

    print(
        f"Installing authoritative Notebook 10 version: "
        f"opacus=={EXPECTED_OPACUS_VERSION}"
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"opacus=={EXPECTED_OPACUS_VERSION}",
        ]
    )

    import opacus


OPACUS_VERSION = str(
    opacus.__version__
)

if OPACUS_VERSION != EXPECTED_OPACUS_VERSION:

    raise RuntimeError(
        "Opacus version mismatch.\n"
        f"Expected: {EXPECTED_OPACUS_VERSION}\n"
        f"Found   : {OPACUS_VERSION}"
    )

print(
    f"✓ Opacus version validated: {OPACUS_VERSION}"
)


# --------------------------------------------------------------------------------------------------
# 7.3 Load canonical Opacus RDP implementation
# --------------------------------------------------------------------------------------------------

try:

    from opacus.accountants.analysis.rdp import (
        compute_rdp,
        get_privacy_spent,
    )

except Exception as exc:

    raise RuntimeError(
        "Unable to load the Opacus RDP accounting implementation."
    ) from exc

print(
    "✓ Opacus RDP accounting implementation loaded"
)


# --------------------------------------------------------------------------------------------------
# 7.4 Reproducible RDP order grid
# --------------------------------------------------------------------------------------------------

RDP_ALPHAS = np.concatenate(
    [
        np.arange(
            1.01,
            10.00,
            0.01,
            dtype=np.float64,
        ),
        np.arange(
            10.0,
            1001.0,
            1.0,
            dtype=np.float64,
        ),
    ]
)

RDP_ALPHAS = np.unique(
    RDP_ALPHAS
)

RDP_ALPHAS = RDP_ALPHAS[
    np.isfinite(RDP_ALPHAS)
    &
    (RDP_ALPHAS > 1.0)
]

RDP_ALPHAS = np.sort(
    RDP_ALPHAS
)

if RDP_ALPHAS.size == 0:

    raise RuntimeError(
        "RDP order grid is empty."
    )

print(
    f"✓ RDP order grid validated: "
    f"{len(RDP_ALPHAS)} orders"
)


# --------------------------------------------------------------------------------------------------
# 7.5 RDP epsilon evaluator
# --------------------------------------------------------------------------------------------------

def configured_rdp_epsilon(
    sample_rate,
    noise_multiplier,
    steps,
    delta,
):
    """
    Calculate formal epsilon for a fixed Poisson-sampled Gaussian
    DP-SGD schedule using RDP composition.
    """

    sample_rate = float(
        sample_rate
    )

    noise_multiplier = float(
        noise_multiplier
    )

    steps = int(
        steps
    )

    delta = float(
        delta
    )

    if not (
        0 < sample_rate <= 1
    ):
        raise ValueError(
            "sample_rate must satisfy 0 < q <= 1."
        )

    if noise_multiplier <= 0:
        raise ValueError(
            "noise_multiplier must be positive."
        )

    if steps <= 0:
        raise ValueError(
            "steps must be positive."
        )

    if not (
        0 < delta < 1
    ):
        raise ValueError(
            "delta must satisfy 0 < delta < 1."
        )

    rdp = compute_rdp(
        q=sample_rate,
        noise_multiplier=noise_multiplier,
        steps=steps,
        orders=RDP_ALPHAS,
    )

    epsilon, optimal_order = get_privacy_spent(
        orders=RDP_ALPHAS,
        rdp=rdp,
        delta=delta,
    )

    epsilon = float(
        epsilon
    )

    optimal_order = float(
        optimal_order
    )

    if not np.isfinite(epsilon):

        raise RuntimeError(
            "RDP accountant returned non-finite epsilon."
        )

    if epsilon < 0:

        raise RuntimeError(
            "RDP accountant returned negative epsilon."
        )

    return (
        epsilon,
        optimal_order,
    )


# --------------------------------------------------------------------------------------------------
# 7.6 Calibration configuration
# --------------------------------------------------------------------------------------------------

CALIBRATION_TOLERANCE = 1e-7
CALIBRATION_MAX_ITERATIONS = 100

INITIAL_SIGMA_LOWER = 0.1
INITIAL_SIGMA_UPPER = 10.0


# --------------------------------------------------------------------------------------------------
# 7.7 Dataset-specific noise calibration
# --------------------------------------------------------------------------------------------------

def calibrate_noise_multiplier(
    sample_rate,
    steps,
    delta,
    target_epsilon,
):
    """
    Determine the minimum noise multiplier found by bisection
    such that formal RDP epsilon <= target epsilon.
    """

    lower = float(
        INITIAL_SIGMA_LOWER
    )

    upper = float(
        INITIAL_SIGMA_UPPER
    )

    lower_epsilon, lower_order = (
        configured_rdp_epsilon(
            sample_rate=sample_rate,
            noise_multiplier=lower,
            steps=steps,
            delta=delta,
        )
    )

    upper_epsilon, upper_order = (
        configured_rdp_epsilon(
            sample_rate=sample_rate,
            noise_multiplier=upper,
            steps=steps,
            delta=delta,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Expand upper bound if necessary
    # ----------------------------------------------------------------------------------------------

    expansion_count = 0

    while (
        upper_epsilon > target_epsilon
        and
        expansion_count < 30
    ):

        upper *= 2.0

        upper_epsilon, upper_order = (
            configured_rdp_epsilon(
                sample_rate=sample_rate,
                noise_multiplier=upper,
                steps=steps,
                delta=delta,
            )
        )

        expansion_count += 1

    if upper_epsilon > target_epsilon:

        raise RuntimeError(
            "Unable to bracket the target epsilon.\n"
            f"Upper sigma    : {upper}\n"
            f"Upper epsilon  : {upper_epsilon}\n"
            f"Target epsilon : {target_epsilon}"
        )

    # ----------------------------------------------------------------------------------------------
    # Lower bound already satisfies target
    # ----------------------------------------------------------------------------------------------

    if lower_epsilon <= target_epsilon:

        return (
            lower,
            lower_epsilon,
            lower_order,
        )

    # ----------------------------------------------------------------------------------------------
    # Bisection
    # ----------------------------------------------------------------------------------------------

    best_sigma = upper
    best_epsilon = upper_epsilon
    best_order = upper_order

    for _ in range(
        CALIBRATION_MAX_ITERATIONS
    ):

        midpoint = (
            lower +
            upper
        ) / 2.0

        midpoint_epsilon, midpoint_order = (
            configured_rdp_epsilon(
                sample_rate=sample_rate,
                noise_multiplier=midpoint,
                steps=steps,
                delta=delta,
            )
        )

        if midpoint_epsilon <= target_epsilon:

            upper = midpoint

            best_sigma = midpoint
            best_epsilon = midpoint_epsilon
            best_order = midpoint_order

        else:

            lower = midpoint

        if (
            abs(
                best_epsilon -
                target_epsilon
            )
            <=
            CALIBRATION_TOLERANCE
        ):

            break

        if (
            abs(
                upper -
                lower
            )
            <=
            1e-12
        ):

            break

    # ----------------------------------------------------------------------------------------------
    # Final privacy-budget verification
    # ----------------------------------------------------------------------------------------------

    final_epsilon, final_order = (
        configured_rdp_epsilon(
            sample_rate=sample_rate,
            noise_multiplier=best_sigma,
            steps=steps,
            delta=delta,
        )
    )

    if final_epsilon > (
        target_epsilon +
        CALIBRATION_TOLERANCE
    ):

        raise RuntimeError(
            "Final calibrated epsilon exceeds target.\n"
            f"Sigma           : {best_sigma}\n"
            f"Final epsilon   : {final_epsilon}\n"
            f"Target epsilon  : {target_epsilon}"
        )

    return (
        float(best_sigma),
        float(final_epsilon),
        float(final_order),
    )


# --------------------------------------------------------------------------------------------------
# 7.8 Build dataset-specific calibration records
# --------------------------------------------------------------------------------------------------

NOISE_ROWS = []

CALIBRATED_NOISE_MULTIPLIERS = {}
CALIBRATED_EPSILON = {}
CALIBRATED_ALPHA = {}

for row in PARAMETER_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    n_train = int(
        row.n_train
    )

    sample_rate = float(
        row.sample_rate
    )

    delta = float(
        row.delta
    )

    configured_sigma = float(
        row.noise_multiplier
    )

    max_grad_norm = float(
        row.max_grad_norm
    )

    metadata_total_steps = int(
        row.nominal_total_steps
    )

    # ----------------------------------------------------------------------------------------------
    # Verify Section 6 schedule
    # ----------------------------------------------------------------------------------------------

    matching_step_rows = STEP_DF[
        STEP_DF["dataset"]
        ==
        dataset_id
    ]

    if len(
        matching_step_rows
    ) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one "
            "Section 6 training-step record."
        )

    section6_steps = int(
        matching_step_rows.iloc[0][
            "expected_total_steps"
        ]
    )

    if section6_steps != metadata_total_steps:

        raise RuntimeError(
            f"{dataset_id}: training schedule mismatch.\n"
            f"Section 6 : {section6_steps}\n"
            f"Metadata  : {metadata_total_steps}"
        )

    # ----------------------------------------------------------------------------------------------
    # Calculate epsilon using original configured sigma
    # ----------------------------------------------------------------------------------------------

    configured_epsilon, configured_order = (
        configured_rdp_epsilon(
            sample_rate=sample_rate,
            noise_multiplier=configured_sigma,
            steps=section6_steps,
            delta=delta,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Calibrate sigma
    # ----------------------------------------------------------------------------------------------

    (
        calibrated_sigma,
        calibrated_epsilon,
        calibrated_order,
    ) = calibrate_noise_multiplier(
        sample_rate=sample_rate,
        steps=section6_steps,
        delta=delta,
        target_epsilon=TARGET_EPSILON,
    )

    # ----------------------------------------------------------------------------------------------
    # Noise standard deviations
    # ----------------------------------------------------------------------------------------------

    configured_noise_std = (
        configured_sigma *
        max_grad_norm
    )

    calibrated_noise_std = (
        calibrated_sigma *
        max_grad_norm
    )

    # ----------------------------------------------------------------------------------------------
    # Final checks
    # ----------------------------------------------------------------------------------------------

    if not np.isfinite(
        calibrated_sigma
    ):
        raise RuntimeError(
            f"{dataset_id}: calibrated sigma is non-finite."
        )

    if calibrated_sigma <= 0:
        raise RuntimeError(
            f"{dataset_id}: calibrated sigma must be positive."
        )

    if calibrated_epsilon > (
        TARGET_EPSILON +
        CALIBRATION_TOLERANCE
    ):
        raise RuntimeError(
            f"{dataset_id}: calibrated epsilon exceeds "
            "target epsilon."
        )

    if not np.isfinite(
        calibrated_noise_std
    ):
        raise RuntimeError(
            f"{dataset_id}: calibrated noise standard "
            "deviation is non-finite."
        )

    # ----------------------------------------------------------------------------------------------
    # Register authoritative values
    # ----------------------------------------------------------------------------------------------

    CALIBRATED_NOISE_MULTIPLIERS[
        dataset_id
    ] = calibrated_sigma

    CALIBRATED_EPSILON[
        dataset_id
    ] = calibrated_epsilon

    CALIBRATED_ALPHA[
        dataset_id
    ] = calibrated_order

    NOISE_ROWS.append({

        "dataset":
            dataset_id,

        "n_train":
            n_train,

        "batch_size":
            DP_BATCH_SIZE,

        "sample_rate":
            sample_rate,

        "delta":
            delta,

        "epochs":
            DP_EPOCHS,

        "nominal_total_steps":
            section6_steps,

        "configured_noise_multiplier":
            configured_sigma,

        "configured_epsilon":
            configured_epsilon,

        "configured_optimal_order":
            configured_order,

        "calibrated_noise_multiplier":
            calibrated_sigma,

        "calibrated_epsilon":
            calibrated_epsilon,

        "calibrated_optimal_order":
            calibrated_order,

        "max_grad_norm":
            max_grad_norm,

        "configured_noise_std":
            configured_noise_std,

        "calibrated_noise_std":
            calibrated_noise_std,

        "target_epsilon":
            TARGET_EPSILON,

        "accountant":
            ACCOUNTANT,

        "sampling":
            SAMPLING_MECHANISM,

        "clipping":
            CLIPPING_MECHANISM,

        "loss_reduction":
            LOSS_REDUCTION,

        "calibration_method":
            "RDP binary search",

        "calibration_status":
            "PASS",

        "status":
            "PASS",
    })


# --------------------------------------------------------------------------------------------------
# 7.9 Build authoritative noise DataFrame
# --------------------------------------------------------------------------------------------------

NOISE_PARAMETER_DF = pd.DataFrame(
    NOISE_ROWS
)

if NOISE_PARAMETER_DF.empty:

    raise RuntimeError(
        "Noise calibration produced no records."
    )

if not NOISE_PARAMETER_DF[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Noise calibration validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 7.10 Dataset coverage validation
# --------------------------------------------------------------------------------------------------

if set(
    NOISE_PARAMETER_DF["dataset"]
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Noise calibration dataset coverage mismatch."
    )

if len(
    CALIBRATED_NOISE_MULTIPLIERS
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Calibrated noise-multiplier registry is incomplete."
    )


# --------------------------------------------------------------------------------------------------
# 7.11 Persist Section 7 validation artifact
# --------------------------------------------------------------------------------------------------

NOISE_PARAMETER_PATH = (
    DIRS["validation"] /
    "noise_parameter_validation.csv"
)

NOISE_PARAMETER_DF.to_csv(
    NOISE_PARAMETER_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 7.12 Display calibration results
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("DATASET-SPECIFIC NOISE CALIBRATION")
print("-" * 100)

display(
    NOISE_PARAMETER_DF[
        [
            "dataset",
            "n_train",
            "sample_rate",
            "nominal_total_steps",
            "configured_noise_multiplier",
            "configured_epsilon",
            "calibrated_noise_multiplier",
            "calibrated_epsilon",
            "calibrated_optimal_order",
            "calibrated_noise_std",
            "target_epsilon",
            "calibration_status",
            "status",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 7.13 Final validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 7 VALIDATION")
print("-" * 100)

print(
    "✓ RDP accountant validated"
)

print(
    "✓ Poisson sampling validated"
)

print(
    "✓ Flat clipping validated"
)

print(
    "✓ Mean loss reduction validated"
)

print(
    "✓ Opacus version matches Notebook 10"
)

print(
    "✓ Dataset-specific schedules validated"
)

print(
    "✓ Original configured sigma evaluated"
)

print(
    "✓ Dataset-specific sigma calibrated"
)

print(
    "✓ Calibrated epsilon verified against target"
)

print(
    "✓ Calibrated noise standard deviation validated"
)

print(
    "✓ CALIBRATED_NOISE_MULTIPLIERS registry created"
)

print(
    "✓ CALIBRATED_EPSILON registry created"
)

print(
    "✓ CALIBRATED_ALPHA registry created"
)

print(
    "✓ Notebook 11 is the authoritative "
    "privacy-calibration source"
)

print(
    f"✓ Noise parameter validation saved:\n"
    f"  {NOISE_PARAMETER_PATH}"
)

print(
    "\nSECTION 7 STATUS: PASS"
)

print("=" * 100)


7. DETERMINE NOISE PARAMETERS
⚠ Opacus is not available in the current runtime.
Installing authoritative Notebook 10 version: opacus==1.6.0
✓ Opacus version validated: 1.6.0
✓ Opacus RDP accounting implementation loaded
✓ RDP order grid validated: 1890 orders


/usr/local/lib/python3.13/dist-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the smallest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(



----------------------------------------------------------------------------------------------------
DATASET-SPECIFIC NOISE CALIBRATION
----------------------------------------------------------------------------------------------------


,dataset,n_train,sample_rate,nominal_total_steps,configured_noise_multiplier,configured_epsilon,calibrated_noise_multiplier,calibrated_epsilon,calibrated_optimal_order,calibrated_noise_std,target_epsilon,calibration_status,status
0,adult_income,34189,0.003744,80400,1.0,7.034634,1.217683,5.0,5.18,1.217683,5.0,PASS,PASS
1,bank_marketing,31647,0.004045,74400,1.0,7.363785,1.252411,5.0,5.18,1.252411,5.0,PASS,PASS
2,diabetes_130us,71236,0.001797,167100,1.0,4.572695,0.953647,5.0,5.17,0.953647,5.0,PASS,PASS



----------------------------------------------------------------------------------------------------
SECTION 7 VALIDATION
----------------------------------------------------------------------------------------------------
✓ RDP accountant validated
✓ Poisson sampling validated
✓ Flat clipping validated
✓ Mean loss reduction validated
✓ Opacus version matches Notebook 10
✓ Dataset-specific schedules validated
✓ Original configured sigma evaluated
✓ Dataset-specific sigma calibrated
✓ Calibrated epsilon verified against target
✓ Calibrated noise standard deviation validated
✓ CALIBRATED_NOISE_MULTIPLIERS registry created
✓ CALIBRATED_EPSILON registry created
✓ CALIBRATED_ALPHA registry created
✓ Notebook 11 is the authoritative privacy-calibration source
✓ Noise parameter validation saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/validation/noise_parameter_validation.csv

SECTION 7 STATUS: PASS


In [10]:
# ==================================================================================================
# 8. CONFIGURE PRIVACY ACCOUNTANT
# ==================================================================================================

print("\n" + "=" * 100)
print("8. CONFIGURE PRIVACY ACCOUNTANT")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Runtime dependency validation
# --------------------------------------------------------------------------------------------------

try:
    import opacus

    OPACUS_VERSION = str(
        getattr(
            opacus,
            "__version__",
            "unknown",
        )
    )

except Exception as exc:

    raise ImportError(
        "Opacus is required for the authoritative RDP privacy accountant "
        "in Notebook 11.\n"
        "Install Opacus in the current Colab runtime before continuing."
    ) from exc


try:

    from opacus.accountants.analysis.rdp import (
        compute_rdp,
        get_privacy_spent,
    )

except Exception as exc:

    raise ImportError(
        "The required Opacus RDP accounting utilities "
        "(compute_rdp, get_privacy_spent) are unavailable.\n"
        f"Detected Opacus version: {OPACUS_VERSION}"
    ) from exc


print(
    f"✓ Opacus version       : {OPACUS_VERSION}"
)

# --------------------------------------------------------------------------------------------------
# 2. Validate authoritative accountant configuration
# --------------------------------------------------------------------------------------------------

if ACCOUNTANT != "rdp":

    raise RuntimeError(
        "Notebook 11 requires the RDP accountant. "
        f"Configured accountant: {ACCOUNTANT!r}"
    )


if SAMPLING_MECHANISM != "poisson":

    raise RuntimeError(
        "Notebook 11 requires Poisson sampling for the configured "
        "privacy-accounting protocol."
    )


if CLIPPING_MECHANISM != "flat":

    raise RuntimeError(
        "Notebook 11 requires flat L2 gradient clipping."
    )


if LOSS_REDUCTION != "mean":

    raise RuntimeError(
        "Notebook 11 requires mean loss reduction."
    )

# --------------------------------------------------------------------------------------------------
# 3. Validate Section 6 training-step artifact
# --------------------------------------------------------------------------------------------------

if "STEP_DF" not in globals():

    raise RuntimeError(
        "STEP_DF from Section 6 is not available. "
        "Run Section 6 before configuring the privacy accountant."
    )


REQUIRED_STEP_COLUMNS = {
    "dataset",
    "sample_rate",
    "expected_steps_per_epoch",
    "expected_total_steps",
    "metadata_steps_per_epoch",
    "metadata_total_steps",
    "status",
}

missing_step_columns = (
    REQUIRED_STEP_COLUMNS
    -
    set(STEP_DF.columns)
)

if missing_step_columns:

    raise RuntimeError(
        "Section 6 training-step artifact is missing required columns:\n"
        f"{sorted(missing_step_columns)}"
    )


if STEP_DF.empty:

    raise RuntimeError(
        "Section 6 training-step artifact is empty."
    )


if not STEP_DF[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Section 6 contains failed training-step validation rows."
    )

# --------------------------------------------------------------------------------------------------
# 4. Explicit RDP order grid
#
# Fractional orders:
#     1.01 ... 9.99
#
# Integer orders:
#     10 ... 1000
#
# The grid is persisted conceptually through the downstream accounting
# artifacts and is deliberately broad to reduce the possibility that
# epsilon minimization is constrained by an order boundary.
# --------------------------------------------------------------------------------------------------

RDP_ALPHAS = np.concatenate([

    np.arange(
        1.01,
        10.00,
        0.01,
        dtype=np.float64,
    ),

    np.arange(
        10,
        1001,
        dtype=np.float64,
    ),
])


RDP_ALPHAS = np.unique(
    RDP_ALPHAS
)


RDP_ALPHAS = RDP_ALPHAS[
    np.isfinite(
        RDP_ALPHAS
    )
    &
    (
        RDP_ALPHAS > 1.0
    )
]


RDP_ALPHAS = np.sort(
    RDP_ALPHAS
)


if RDP_ALPHAS.size == 0:

    raise RuntimeError(
        "RDP order grid is empty."
    )


if np.any(
    RDP_ALPHAS <= 1.0
):

    raise RuntimeError(
        "Invalid RDP order detected. "
        "All orders must be strictly greater than 1."
    )


if not np.all(
    np.diff(
        RDP_ALPHAS
    ) > 0
):

    raise RuntimeError(
        "RDP orders must be strictly increasing and unique."
    )


RDP_ALPHA_MIN = float(
    RDP_ALPHAS.min()
)


RDP_ALPHA_MAX = float(
    RDP_ALPHAS.max()
)


RDP_ALPHA_COUNT = int(
    RDP_ALPHAS.size
)


if RDP_ALPHA_MIN <= 1.0:

    raise RuntimeError(
        "RDP minimum order must be greater than 1."
    )


if RDP_ALPHA_MAX < 100:

    raise RuntimeError(
        "RDP order range is insufficient for the configured "
        "research accounting protocol."
    )


if RDP_ALPHA_COUNT < 1000:

    raise RuntimeError(
        "RDP order grid contains fewer than 1000 orders."
    )


if not np.all(
    np.isfinite(
        RDP_ALPHAS
    )
):

    raise RuntimeError(
        "RDP order grid contains non-finite values."
    )

# --------------------------------------------------------------------------------------------------
# 5. Define authoritative RDP evaluation helper
#
# IMPORTANT:
#
# The number of composition steps is supplied explicitly.
#
# It is NOT reconstructed as:
#
#     epochs / sample_rate
#
# because q = batch_size / n_train is the Poisson sampling probability,
# whereas Section 6 establishes the nominal optimizer schedule as:
#
#     steps_per_epoch = ceil(n_train / batch_size)
#
#     total_steps = epochs * steps_per_epoch
#
# The explicit `steps` argument therefore keeps the privacy accountant
# synchronized with the validated Section 6 schedule.
# --------------------------------------------------------------------------------------------------

def evaluate_rdp_privacy(
    *,
    sample_rate,
    noise_multiplier,
    steps,
    delta,
    alphas,
):
    """
    Evaluate the configured Poisson-sampled Gaussian mechanism
    using the authoritative Opacus RDP implementation.

    Parameters
    ----------
    sample_rate : float
        Poisson sampling probability q.

    noise_multiplier : float
        Gaussian noise multiplier sigma.

    steps : int or float
        Explicit number of DP-SGD composition steps validated
        by Notebook 11 Section 6.

    delta : float
        Target delta used for RDP-to-(epsilon, delta)-DP conversion.

    alphas : array-like
        RDP orders.

    Returns
    -------
    epsilon : float
        Epsilon obtained from the Opacus RDP accountant.

    optimal_alpha : float
        RDP order selected by the accountant.

    rdp_values : np.ndarray
        Accumulated RDP values for all supplied orders.
    """

    sample_rate = float(
        sample_rate
    )

    noise_multiplier = float(
        noise_multiplier
    )

    steps = float(
        steps
    )

    delta = float(
        delta
    )

    alphas = np.asarray(
        alphas,
        dtype=np.float64,
    )

    # ----------------------------------------------------------------------------------------------
    # Input validation
    # ----------------------------------------------------------------------------------------------

    if not (
        np.isfinite(sample_rate)
        and
        0.0 < sample_rate < 1.0
    ):

        raise ValueError(
            f"Invalid Poisson sample rate: {sample_rate}"
        )


    if not (
        np.isfinite(noise_multiplier)
        and
        noise_multiplier > 0.0
    ):

        raise ValueError(
            f"Invalid noise multiplier: {noise_multiplier}"
        )


    if not (
        np.isfinite(steps)
        and
        steps > 0.0
        and
        steps == round(steps)
    ):

        raise ValueError(
            f"DP-SGD composition steps must be a positive integer: "
            f"{steps}"
        )


    if not (
        np.isfinite(delta)
        and
        0.0 < delta < 1.0
    ):

        raise ValueError(
            f"Invalid delta: {delta}"
        )


    if alphas.ndim != 1:

        raise ValueError(
            "RDP orders must be one-dimensional."
        )


    if alphas.size == 0:

        raise ValueError(
            "RDP order set cannot be empty."
        )


    if not np.all(
        np.isfinite(
            alphas
        )
    ):

        raise ValueError(
            "RDP orders contain non-finite values."
        )


    if np.any(
        alphas <= 1.0
    ):

        raise ValueError(
            "RDP orders must be strictly greater than 1."
        )

    # ----------------------------------------------------------------------------------------------
    # Opacus RDP computation
    # ----------------------------------------------------------------------------------------------

    steps = int(
        round(
            steps
        )
    )


    rdp_values = compute_rdp(
        q=sample_rate,
        noise_multiplier=noise_multiplier,
        steps=steps,
        orders=alphas,
    )


    rdp_values = np.asarray(
        rdp_values,
        dtype=np.float64,
    )


    if rdp_values.shape != alphas.shape:

        raise RuntimeError(
            "RDP value/order shape mismatch:\n"
            f"RDP shape={rdp_values.shape}, "
            f"alpha shape={alphas.shape}"
        )


    if not np.all(
        np.isfinite(
            rdp_values
        )
    ):

        raise RuntimeError(
            "Opacus RDP computation produced non-finite values."
        )

    # ----------------------------------------------------------------------------------------------
    # Authoritative RDP -> epsilon conversion
    # ----------------------------------------------------------------------------------------------

    privacy_spent = get_privacy_spent(
        orders=alphas,
        rdp=rdp_values,
        delta=delta,
    )


    epsilon = float(
        privacy_spent[0]
    )


    optimal_alpha = float(
        privacy_spent[1]
    )


    if not (
        np.isfinite(epsilon)
        and
        epsilon >= 0.0
    ):

        raise RuntimeError(
            "Opacus returned an invalid epsilon."
        )


    if not (
        np.isfinite(optimal_alpha)
        and
        optimal_alpha > 1.0
    ):

        raise RuntimeError(
            "Opacus returned an invalid optimal RDP order."
        )


    return (
        epsilon,
        optimal_alpha,
        rdp_values,
    )

# --------------------------------------------------------------------------------------------------
# 6. Accountant smoke test
#
# This is not an experimental result.
#
# It verifies that:
#
#   1. Opacus is available.
#   2. The exact RDP interface is available.
#   3. Section 6's explicit training-step schedule can be passed
#      successfully to the authoritative accountant.
# --------------------------------------------------------------------------------------------------

SMOKE_ROW = STEP_DF.iloc[0]


SMOKE_DATASET = str(
    SMOKE_ROW["dataset"]
)


SMOKE_SAMPLE_RATE = float(
    SMOKE_ROW["sample_rate"]
)


SMOKE_NOISE_MULTIPLIER = float(
    PARAMETER_DF[
        PARAMETER_DF["dataset"]
        ==
        SMOKE_DATASET
    ].iloc[0]["noise_multiplier"]
)


SMOKE_DELTA = float(
    PARAMETER_DF[
        PARAMETER_DF["dataset"]
        ==
        SMOKE_DATASET
    ].iloc[0]["delta"]
)


SMOKE_TOTAL_STEPS = int(
    SMOKE_ROW["expected_total_steps"]
)


SMOKE_METADATA_TOTAL_STEPS = int(
    SMOKE_ROW["metadata_total_steps"]
)


if (
    SMOKE_TOTAL_STEPS
    !=
    SMOKE_METADATA_TOTAL_STEPS
):

    raise RuntimeError(
        "Smoke-test total steps do not match Section 6 metadata."
    )


(
    SMOKE_EPSILON,
    SMOKE_OPTIMAL_ALPHA,
    SMOKE_RDP_VALUES,
) = evaluate_rdp_privacy(
    sample_rate=SMOKE_SAMPLE_RATE,
    noise_multiplier=SMOKE_NOISE_MULTIPLIER,
    steps=SMOKE_TOTAL_STEPS,
    delta=SMOKE_DELTA,
    alphas=RDP_ALPHAS,
)


if not np.isfinite(
    SMOKE_EPSILON
):

    raise RuntimeError(
        "RDP accountant smoke test returned non-finite epsilon."
    )


if not np.all(
    np.isfinite(
        SMOKE_RDP_VALUES
    )
):

    raise RuntimeError(
        "RDP accountant smoke test returned non-finite RDP values."
    )

# --------------------------------------------------------------------------------------------------
# 7. Verify the smoke-test schedule explicitly
# --------------------------------------------------------------------------------------------------

EXPECTED_SMOKE_STEPS = int(
    STEP_DF[
        STEP_DF["dataset"]
        ==
        SMOKE_DATASET
    ].iloc[0]["expected_total_steps"]
)


if SMOKE_TOTAL_STEPS != EXPECTED_SMOKE_STEPS:

    raise RuntimeError(
        "Smoke-test steps are inconsistent with Section 6."
    )

# --------------------------------------------------------------------------------------------------
# 8. Final configuration summary
# --------------------------------------------------------------------------------------------------

print(
    f"✓ Opacus version       : {OPACUS_VERSION}"
)

print(
    f"✓ RDP order minimum    : {RDP_ALPHA_MIN:.2f}"
)

print(
    f"✓ RDP order maximum    : {RDP_ALPHA_MAX:.0f}"
)

print(
    f"✓ RDP order count      : {RDP_ALPHA_COUNT}"
)

print(
    f"✓ Sampling mechanism    : {SAMPLING_MECHANISM}"
)

print(
    f"✓ Accountant            : {ACCOUNTANT}"
)

print(
    f"✓ Clipping mechanism    : {CLIPPING_MECHANISM}"
)

print(
    f"✓ Loss reduction        : {LOSS_REDUCTION}"
)

print(
    f"✓ Smoke-test dataset    : {SMOKE_DATASET}"
)

print(
    f"✓ Smoke-test q          : {SMOKE_SAMPLE_RATE:.12f}"
)

print(
    f"✓ Smoke-test sigma      : {SMOKE_NOISE_MULTIPLIER:.6f}"
)

print(
    f"✓ Smoke-test steps      : {SMOKE_TOTAL_STEPS}"
)

print(
    f"✓ Smoke-test delta      : {SMOKE_DELTA:.12g}"
)

print(
    f"✓ Opacus RDP smoke ε    : {SMOKE_EPSILON:.6f}"
)

print(
    f"✓ Smoke-test α*         : {SMOKE_OPTIMAL_ALPHA:.2f}"
)

print(
    "✓ Explicit Section 6 training-step schedule accepted."
)

print(
    "✓ Opacus RDP accountant configured and validated."
)

print(
    "✓ Actual dataset-level accounting remains deferred to the next section."
)

print(
    "✓ No end-to-end privacy claim is established by this section."
)

print(
    "SECTION 8 STATUS: PASS"
)


8. CONFIGURE PRIVACY ACCOUNTANT
✓ Opacus version       : 1.6.0
✓ Opacus version       : 1.6.0
✓ RDP order minimum    : 1.01
✓ RDP order maximum    : 1000
✓ RDP order count      : 1890
✓ Sampling mechanism    : poisson
✓ Accountant            : rdp
✓ Clipping mechanism    : flat
✓ Loss reduction        : mean
✓ Smoke-test dataset    : adult_income
✓ Smoke-test q          : 0.003743894235
✓ Smoke-test sigma      : 1.000000
✓ Smoke-test steps      : 80400
✓ Smoke-test delta      : 1e-05
✓ Opacus RDP smoke ε    : 7.034634
✓ Smoke-test α*         : 4.13
✓ Explicit Section 6 training-step schedule accepted.
✓ Opacus RDP accountant configured and validated.
✓ Actual dataset-level accounting remains deferred to the next section.
✓ No end-to-end privacy claim is established by this section.
SECTION 8 STATUS: PASS


In [11]:
# ==================================================================================================
# 9. CALCULATE CONFIGURED-SCHEDULE EPSILON
# ==================================================================================================

print("\n" + "=" * 100)
print("9. CALCULATE CONFIGURED-SCHEDULE EPSILON")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Accounting scope
#
# This section calculates the formal (epsilon, delta)-privacy expenditure for the
# configured DP-SGD training schedule established and validated in Section 6.
#
# IMPORTANT:
#
#   This is NOT the observed/achieved epsilon from an executed training run.
#
#   Configured-schedule epsilon is determined from:
#
#       q       = Poisson sampling probability
#       sigma   = Gaussian noise multiplier
#       T       = validated nominal DP-SGD composition steps
#       delta   = dataset-specific delta
#
#   The actual achieved training epsilon must be obtained from the realized
#   Notebook 12 privacy accountant state.
#
#   The target epsilon is treated as a privacy-budget requirement.
#   It is NOT assumed that the current noise multiplier achieves that target.
# --------------------------------------------------------------------------------------------------

ACCOUNTING_TYPE = (
    "configured_schedule"
)

ACHIEVED_EPSILON_STATUS = (
    "DEFERRED_TO_NOTEBOOK_12"
)

# Target comparison tolerance.
#
# This is only used to determine whether the calculated epsilon is numerically
# at or below the configured target budget. It is NOT used to force the
# accountant result to equal the target.
TARGET_EPSILON_TOLERANCE = 1e-10

# --------------------------------------------------------------------------------------------------
# 2. Validate required Section 8 objects
# --------------------------------------------------------------------------------------------------

required_objects = {
    "compute_rdp": compute_rdp,
    "get_privacy_spent": get_privacy_spent,
    "RDP_ALPHAS": RDP_ALPHAS,
}

for object_name, object_value in required_objects.items():

    if object_value is None:

        raise RuntimeError(
            f"Required Section 8 object is unavailable: {object_name}"
        )


if len(RDP_ALPHAS) == 0:

    raise RuntimeError(
        "RDP order set is empty."
    )

# --------------------------------------------------------------------------------------------------
# 3. Validate required Section 6 artifact
# --------------------------------------------------------------------------------------------------

if "STEP_DF" not in globals():

    raise RuntimeError(
        "STEP_DF from Section 6 is unavailable. "
        "Run Section 6 before Section 9."
    )


REQUIRED_STEP_COLUMNS = {
    "dataset",
    "n_train",
    "batch_size",
    "epochs",
    "sample_rate",
    "expected_steps_per_epoch",
    "metadata_steps_per_epoch",
    "expected_total_steps",
    "metadata_total_steps",
    "schedule_basis",
    "sampling",
    "status",
}

missing_step_columns = (
    REQUIRED_STEP_COLUMNS
    -
    set(STEP_DF.columns)
)

if missing_step_columns:

    raise RuntimeError(
        "Section 6 training-step artifact is missing required columns:\n"
        f"{sorted(missing_step_columns)}"
    )


if STEP_DF.empty:

    raise RuntimeError(
        "Section 6 training-step artifact is empty."
    )


if not STEP_DF[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Section 6 contains failed training-step validation rows."
    )

# --------------------------------------------------------------------------------------------------
# 4. RDP epsilon calculation helper
#
# The number of composition steps is supplied explicitly.
#
# IMPORTANT:
#
#     steps is NEVER reconstructed as:
#
#         epochs / sample_rate
#
#     Instead, Section 6's validated:
#
#         expected_total_steps
#
#     is passed directly to the Opacus accountant.
# --------------------------------------------------------------------------------------------------

def calculate_configured_rdp_epsilon(
    *,
    sample_rate,
    noise_multiplier,
    steps,
    delta,
    orders,
):
    """
    Calculate configured-schedule epsilon for the
    Poisson-sampled Gaussian mechanism using Opacus RDP.

    Parameters
    ----------
    sample_rate : float
        Poisson sampling probability q.

    noise_multiplier : float
        Gaussian noise multiplier sigma.

    steps : int
        Explicit DP-SGD composition-step count validated by Section 6.

    delta : float
        Delta used for RDP-to-(epsilon, delta)-DP conversion.

    orders : array-like
        RDP alpha orders.

    Returns
    -------
    epsilon : float
        Configured-schedule epsilon.

    optimal_order : float
        RDP order selected by Opacus.

    rdp_values : np.ndarray
        Accumulated RDP values.
    """

    sample_rate = float(
        sample_rate
    )

    noise_multiplier = float(
        noise_multiplier
    )

    steps = int(
        steps
    )

    delta = float(
        delta
    )

    orders = np.asarray(
        orders,
        dtype=np.float64,
    )

    # ----------------------------------------------------------------------------------------------
    # Parameter validation
    # ----------------------------------------------------------------------------------------------

    if not (
        np.isfinite(sample_rate)
        and
        0.0 < sample_rate < 1.0
    ):

        raise ValueError(
            f"Invalid Poisson sample rate: {sample_rate}"
        )


    if not (
        np.isfinite(noise_multiplier)
        and
        noise_multiplier > 0.0
    ):

        raise ValueError(
            f"Invalid noise multiplier: {noise_multiplier}"
        )


    if (
        not isinstance(
            steps,
            (int, np.integer)
        )
        or
        steps <= 0
    ):

        raise ValueError(
            f"Composition-step count must be a positive integer: {steps}"
        )


    if not (
        np.isfinite(delta)
        and
        0.0 < delta < 1.0
    ):

        raise ValueError(
            f"Invalid delta: {delta}"
        )


    if orders.ndim != 1:

        raise ValueError(
            "RDP orders must be one-dimensional."
        )


    if orders.size == 0:

        raise ValueError(
            "RDP order set cannot be empty."
        )


    if not np.all(
        np.isfinite(
            orders
        )
    ):

        raise ValueError(
            "RDP orders contain non-finite values."
        )


    if np.any(
        orders <= 1.0
    ):

        raise ValueError(
            "All RDP orders must be strictly greater than 1."
        )

    # ----------------------------------------------------------------------------------------------
    # Compute accumulated RDP
    # ----------------------------------------------------------------------------------------------

    rdp_values = compute_rdp(
        q=sample_rate,
        noise_multiplier=noise_multiplier,
        steps=steps,
        orders=orders,
    )


    rdp_values = np.asarray(
        rdp_values,
        dtype=np.float64,
    )


    if rdp_values.shape != orders.shape:

        raise RuntimeError(
            "RDP value/order shape mismatch:\n"
            f"RDP shape={rdp_values.shape}\n"
            f"Order shape={orders.shape}"
        )


    if not np.all(
        np.isfinite(
            rdp_values
        )
    ):

        raise RuntimeError(
            "RDP computation produced non-finite values."
        )

    # ----------------------------------------------------------------------------------------------
    # Convert RDP to (epsilon, delta)-DP using Opacus
    # ----------------------------------------------------------------------------------------------

    epsilon, optimal_order = (
        get_privacy_spent(
            orders=orders,
            rdp=rdp_values,
            delta=delta,
        )
    )


    epsilon = float(
        epsilon
    )

    optimal_order = float(
        optimal_order
    )


    if not (
        np.isfinite(epsilon)
        and
        epsilon >= 0.0
    ):

        raise RuntimeError(
            "Opacus returned an invalid epsilon."
        )


    if not (
        np.isfinite(optimal_order)
        and
        optimal_order > 1.0
    ):

        raise RuntimeError(
            "Opacus returned an invalid optimal RDP order."
        )


    return (
        epsilon,
        optimal_order,
        rdp_values,
    )

# --------------------------------------------------------------------------------------------------
# 5. Validate dataset coverage
# --------------------------------------------------------------------------------------------------

if set(
    PARAMETER_DF[
        "dataset"
    ].astype(str)
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Parameter metadata dataset coverage does not match DATASET_IDS."
    )


if set(
    STEP_DF[
        "dataset"
    ].astype(str)
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Training-step validation dataset coverage does not match DATASET_IDS."
    )

# --------------------------------------------------------------------------------------------------
# 6. Calculate configured-schedule epsilon for each dataset
# --------------------------------------------------------------------------------------------------

ACCOUNTING_ROWS = []

for row in PARAMETER_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    # ----------------------------------------------------------------------------------------------
    # Retrieve exactly one validated Section 6 record
    # ----------------------------------------------------------------------------------------------

    matching_steps = STEP_DF.loc[
        STEP_DF[
            "dataset"
        ].astype(str)
        ==
        dataset_id
    ]


    if len(matching_steps) != 1:

        raise RuntimeError(
            f"Expected exactly one training-step record for {dataset_id}, "
            f"found {len(matching_steps)}."
        )


    step_row = matching_steps.iloc[0]

    # ----------------------------------------------------------------------------------------------
    # Use Section 6 authoritative schedule
    # ----------------------------------------------------------------------------------------------

    steps_per_epoch = int(
        step_row[
            "expected_steps_per_epoch"
        ]
    )


    metadata_steps_per_epoch = int(
        step_row[
            "metadata_steps_per_epoch"
        ]
    )


    total_steps = int(
        step_row[
            "expected_total_steps"
        ]
    )


    metadata_total_steps = int(
        step_row[
            "metadata_total_steps"
        ]
    )


    schedule_basis = str(
        step_row[
            "schedule_basis"
        ]
    )


    if (
        steps_per_epoch
        !=
        metadata_steps_per_epoch
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 6 steps-per-epoch "
            "does not match Notebook 10 metadata."
        )


    if (
        total_steps
        !=
        metadata_total_steps
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 6 total steps "
            "does not match Notebook 10 metadata."
        )


    if schedule_basis != "ceil(n_train / batch_size)":

        raise RuntimeError(
            f"{dataset_id}: unexpected training schedule basis: "
            f"{schedule_basis!r}"
        )

    # ----------------------------------------------------------------------------------------------
    # Retrieve persisted privacy parameters
    # ----------------------------------------------------------------------------------------------

    sample_rate = float(
        row.sample_rate
    )


    noise_multiplier = float(
        row.noise_multiplier
    )


    max_grad_norm = float(
        row.max_grad_norm
    )


    delta = float(
        row.delta
    )


    target_epsilon = float(
        row.target_epsilon
    )


    n_train = int(
        row.n_train
    )

    # ----------------------------------------------------------------------------------------------
    # Revalidate Poisson sampling rate
    # ----------------------------------------------------------------------------------------------

    expected_sample_rate = (
        DP_BATCH_SIZE /
        n_train
    )


    sample_rate_error = abs(
        sample_rate -
        expected_sample_rate
    )


    if not np.isclose(
        sample_rate,
        expected_sample_rate,
        rtol=1e-10,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{dataset_id}: sample rate is inconsistent with "
            "the validated Poisson sampling configuration."
        )

    # ----------------------------------------------------------------------------------------------
    # Validate total-step schedule independently
    # ----------------------------------------------------------------------------------------------

    independently_expected_steps = int(
        math.ceil(
            n_train /
            DP_BATCH_SIZE
        )
    )


    independently_expected_total_steps = (
        DP_EPOCHS *
        independently_expected_steps
    )


    if (
        steps_per_epoch
        !=
        independently_expected_steps
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 6 steps-per-epoch value is inconsistent "
            "with ceil(n_train / batch_size)."
        )


    if (
        total_steps
        !=
        independently_expected_total_steps
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 6 total-step value is inconsistent "
            "with epochs × ceil(n_train / batch_size)."
        )

    # ----------------------------------------------------------------------------------------------
    # RDP accounting
    # ----------------------------------------------------------------------------------------------

    (
        configured_epsilon,
        optimal_rdp_order,
        rdp_values,
    ) = calculate_configured_rdp_epsilon(
        sample_rate=sample_rate,
        noise_multiplier=noise_multiplier,
        steps=total_steps,
        delta=delta,
        orders=RDP_ALPHAS,
    )

    # ----------------------------------------------------------------------------------------------
    # Target-budget comparison
    #
    # IMPORTANT:
    #
    # A configured epsilon different from the target is not itself a computational
    # failure. The target is a budget requirement against which the configuration
    # is evaluated.
    # ----------------------------------------------------------------------------------------------

    epsilon_difference = (
        configured_epsilon -
        target_epsilon
    )


    absolute_epsilon_difference = abs(
        epsilon_difference
    )


    within_target_epsilon = (
        configured_epsilon
        <=
        (
            target_epsilon +
            TARGET_EPSILON_TOLERANCE
        )
    )


    epsilon_budget_status = (
        "WITHIN_TARGET"
        if within_target_epsilon
        else "EXCEEDS_TARGET"
    )

    # ----------------------------------------------------------------------------------------------
    # RDP boundary validation
    # ----------------------------------------------------------------------------------------------

    optimal_at_lower_boundary = np.isclose(
        optimal_rdp_order,
        float(
            RDP_ALPHAS.min()
        ),
        rtol=0.0,
        atol=1e-12,
    )


    optimal_at_upper_boundary = np.isclose(
        optimal_rdp_order,
        float(
            RDP_ALPHAS.max()
        ),
        rtol=0.0,
        atol=1e-12,
    )


    optimal_at_boundary = (
        optimal_at_lower_boundary
        or
        optimal_at_upper_boundary
    )


    if optimal_at_boundary:

        raise RuntimeError(
            f"{dataset_id}: optimal RDP order "
            f"{optimal_rdp_order:.6f} is at the accounting-order boundary."
        )

    # ----------------------------------------------------------------------------------------------
    # RDP numerical validation
    # ----------------------------------------------------------------------------------------------

    if not np.all(
        np.isfinite(
            rdp_values
        )
    ):

        raise RuntimeError(
            f"{dataset_id}: non-finite RDP values detected."
        )


    if not np.isfinite(
        configured_epsilon
    ):

        raise RuntimeError(
            f"{dataset_id}: non-finite configured-schedule epsilon."
        )

    # ----------------------------------------------------------------------------------------------
    # Record accounting result
    # ----------------------------------------------------------------------------------------------

    ACCOUNTING_ROWS.append({

        "dataset":
            dataset_id,

        "n_train":
            n_train,

        "batch_size":
            int(DP_BATCH_SIZE),

        "sample_rate":
            sample_rate,

        "sample_rate_error":
            sample_rate_error,

        "noise_multiplier":
            noise_multiplier,

        "max_grad_norm":
            max_grad_norm,

        "epochs":
            int(DP_EPOCHS),

        "steps_per_epoch":
            steps_per_epoch,

        "metadata_steps_per_epoch":
            metadata_steps_per_epoch,

        "total_steps":
            total_steps,

        "metadata_total_steps":
            metadata_total_steps,

        "schedule_basis":
            schedule_basis,

        "delta":
            delta,

        "target_epsilon":
            target_epsilon,

        "configured_schedule_epsilon":
            configured_epsilon,

        "epsilon_difference_from_target":
            epsilon_difference,

        "absolute_epsilon_difference":
            absolute_epsilon_difference,

        "target_epsilon_tolerance":
            TARGET_EPSILON_TOLERANCE,

        "within_target_epsilon":
            within_target_epsilon,

        "epsilon_budget_status":
            epsilon_budget_status,

        "optimal_rdp_order":
            optimal_rdp_order,

        "rdp_order_min":
            float(
                RDP_ALPHAS.min()
            ),

        "rdp_order_max":
            float(
                RDP_ALPHAS.max()
            ),

        "rdp_order_count":
            int(
                len(
                    RDP_ALPHAS
                )
            ),

        "optimal_rdp_order_at_boundary":
            optimal_at_boundary,

        "accounting_type":
            ACCOUNTING_TYPE,

        "achieved_epsilon_status":
            ACHIEVED_EPSILON_STATUS,

        "status":
            "PASS",
    })

# --------------------------------------------------------------------------------------------------
# 7. Construct accounting DataFrame
# --------------------------------------------------------------------------------------------------

CONFIGURED_ACCOUNTING_DF = pd.DataFrame(
    ACCOUNTING_ROWS
)


if CONFIGURED_ACCOUNTING_DF.empty:

    raise RuntimeError(
        "No configured-schedule privacy accounting records were generated."
    )


if len(
    CONFIGURED_ACCOUNTING_DF
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Configured-schedule accounting does not contain exactly one "
        "record per canonical dataset."
    )


if set(
    CONFIGURED_ACCOUNTING_DF[
        "dataset"
    ].astype(str)
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Configured-schedule accounting dataset coverage is incomplete."
    )

# --------------------------------------------------------------------------------------------------
# 8. Final numerical validation
# --------------------------------------------------------------------------------------------------

numeric_columns = [

    "n_train",

    "batch_size",

    "sample_rate",

    "sample_rate_error",

    "noise_multiplier",

    "max_grad_norm",

    "epochs",

    "steps_per_epoch",

    "metadata_steps_per_epoch",

    "total_steps",

    "metadata_total_steps",

    "delta",

    "target_epsilon",

    "configured_schedule_epsilon",

    "epsilon_difference_from_target",

    "absolute_epsilon_difference",

    "target_epsilon_tolerance",

    "optimal_rdp_order",

    "rdp_order_min",

    "rdp_order_max",

    "rdp_order_count",
]


for column in numeric_columns:

    values = pd.to_numeric(
        CONFIGURED_ACCOUNTING_DF[
            column
        ],
        errors="coerce",
    )


    if not np.all(
        np.isfinite(
            values.to_numpy()
        )
    ):

        raise RuntimeError(
            f"Non-finite values detected in accounting column: {column}"
        )

# --------------------------------------------------------------------------------------------------
# 9. Validate accounting status
#
# Every dataset must have a valid numerical accounting result.
#
# EXCEEDS_TARGET is NOT treated as a computational failure.
# It is a scientifically meaningful budget-status result.
# --------------------------------------------------------------------------------------------------

if not CONFIGURED_ACCOUNTING_DF[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Configured-schedule privacy accounting computation failed."
    )


VALID_BUDGET_STATUSES = {
    "WITHIN_TARGET",
    "EXCEEDS_TARGET",
}


if not CONFIGURED_ACCOUNTING_DF[
    "epsilon_budget_status"
].isin(
    VALID_BUDGET_STATUSES
).all():

    raise RuntimeError(
        "Unexpected epsilon budget status detected."
    )

# --------------------------------------------------------------------------------------------------
# 10. Persist configured-schedule accounting
# --------------------------------------------------------------------------------------------------

ACCOUNTING_PATH = (
    DIRS["accounting"] /
    "sppgan_configured_schedule_rdp_accounting.csv"
)


CONFIGURED_ACCOUNTING_DF.to_csv(
    ACCOUNTING_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 11. Display results
# --------------------------------------------------------------------------------------------------

display(
    CONFIGURED_ACCOUNTING_DF[
        [
            "dataset",
            "n_train",
            "sample_rate",
            "noise_multiplier",
            "epochs",
            "steps_per_epoch",
            "total_steps",
            "delta",
            "target_epsilon",
            "configured_schedule_epsilon",
            "epsilon_difference_from_target",
            "within_target_epsilon",
            "epsilon_budget_status",
            "optimal_rdp_order",
            "status",
        ]
    ]
)


print(
    "\nConfigured-schedule privacy accounting:"
)


for row in CONFIGURED_ACCOUNTING_DF.itertuples(
    index=False
):

    print(
        f"  {row.dataset:18s} "
        f"ε_config={row.configured_schedule_epsilon:.6f} "
        f"ε_target={row.target_epsilon:.6f} "
        f"δ={row.delta:.8g} "
        f"T={int(row.total_steps):>7} "
        f"α*={row.optimal_rdp_order:.2f} "
        f"|Δε|={row.absolute_epsilon_difference:.6f} "
        f"| {row.epsilon_budget_status}"
    )

# --------------------------------------------------------------------------------------------------
# 12. Privacy provenance statement
# --------------------------------------------------------------------------------------------------

print(
    "\nPrivacy accounting provenance:"
)


print(
    "  ✓ RDP accountant                : Opacus"
)


print(
    "  ✓ Sampling mechanism            : Poisson"
)


print(
    "  ✓ Privacy component             : discriminator / critic"
)


print(
    "  ✓ Accounting schedule           : validated Section 6 schedule"
)


print(
    "  ✓ Schedule basis                : "
    "ceil(n_train / batch_size)"
)


print(
    "  ✓ Epsilon type                  : configured-schedule epsilon"
)


print(
    "  ✓ Target epsilon                : budget comparison only"
)


print(
    "  ✓ Achieved training epsilon     : deferred to Notebook 12"
)


print(
    "  ✓ End-to-end privacy claim      : not established"
)


print(
    f"\n✓ Configured-schedule RDP accounting saved:\n"
    f"  {ACCOUNTING_PATH}"
)


print(
    "SECTION 9 STATUS: PASS"
)


9. CALCULATE CONFIGURED-SCHEDULE EPSILON


,dataset,n_train,sample_rate,noise_multiplier,epochs,steps_per_epoch,total_steps,delta,target_epsilon,configured_schedule_epsilon,epsilon_difference_from_target,within_target_epsilon,epsilon_budget_status,optimal_rdp_order,status
0,adult_income,34189,0.003744,1.0,300,268,80400,0.00001,5.0,7.034634,2.034634,False,EXCEEDS_TARGET,4.13,PASS
1,bank_marketing,31647,0.004045,1.0,300,248,74400,0.00001,5.0,7.363785,2.363785,False,EXCEEDS_TARGET,4.01,PASS
2,diabetes_130us,71236,0.001797,1.0,300,557,167100,0.00001,5.0,4.572695,-0.427305,True,WITHIN_TARGET,5.50,PASS



Configured-schedule privacy accounting:
  adult_income       ε_config=7.034634 ε_target=5.000000 δ=1e-05 T=  80400 α*=4.13 |Δε|=2.034634 | EXCEEDS_TARGET
  bank_marketing     ε_config=7.363785 ε_target=5.000000 δ=1e-05 T=  74400 α*=4.01 |Δε|=2.363785 | EXCEEDS_TARGET
  diabetes_130us     ε_config=4.572695 ε_target=5.000000 δ=1e-05 T= 167100 α*=5.50 |Δε|=0.427305 | WITHIN_TARGET

Privacy accounting provenance:
  ✓ RDP accountant                : Opacus
  ✓ Sampling mechanism            : Poisson
  ✓ Privacy component             : discriminator / critic
  ✓ Accounting schedule           : validated Section 6 schedule
  ✓ Schedule basis                : ceil(n_train / batch_size)
  ✓ Epsilon type                  : configured-schedule epsilon
  ✓ Target epsilon                : budget comparison only
  ✓ Achieved training epsilon     : deferred to Notebook 12
  ✓ End-to-end privacy claim      : not established

✓ Configured-schedule RDP accounting saved:
  /content/drive/MyDrive/SPP_GAN

In [12]:
# ==================================================================================================
# 10. VALIDATE DELTA
# ==================================================================================================

print("\n" + "=" * 100)
print("10. VALIDATE DELTA")
print("=" * 100)

DELTA_RULE = (
    "min(1e-5, 1/N_train)"
)

DELTA_ROWS = []

for row in PARAMETER_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    n_train = int(
        row.n_train
    )

    configured_delta = float(
        row.delta
    )

    expected_delta = min(
        1e-5,
        1.0 / n_train,
    )

    absolute_error = abs(
        configured_delta -
        expected_delta
    )

    status = (
        "PASS"
        if (
            0 <
            configured_delta
            <
            1
            and
            absolute_error <= 1e-15
        )
        else "FAIL"
    )

    DELTA_ROWS.append({

        "dataset":
            dataset_id,

        "n_train":
            n_train,

        "delta_rule":
            DELTA_RULE,

        "configured_delta":
            configured_delta,

        "expected_delta":
            expected_delta,

        "absolute_error":
            absolute_error,

        "status":
            status,
    })

DELTA_DF = pd.DataFrame(
    DELTA_ROWS
)

if not DELTA_DF[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Delta validation failed."
    )

DELTA_PATH = (
    DIRS["validation"] /
    "delta_validation.csv"
)

DELTA_DF.to_csv(
    DELTA_PATH,
    index=False,
)

display(
    DELTA_DF
)

print(
    f"✓ Delta validation saved:\n"
    f"  {DELTA_PATH}"
)

print(
    "SECTION 10 STATUS: PASS"
)


10. VALIDATE DELTA


,dataset,n_train,delta_rule,configured_delta,expected_delta,absolute_error,status
0,adult_income,34189,"min(1e-5, 1/N_train)",0.00001,0.00001,0.0,PASS
1,bank_marketing,31647,"min(1e-5, 1/N_train)",0.00001,0.00001,0.0,PASS
2,diabetes_130us,71236,"min(1e-5, 1/N_train)",0.00001,0.00001,0.0,PASS


✓ Delta validation saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/validation/delta_validation.csv
SECTION 10 STATUS: PASS


In [13]:
# ==================================================================================================
# 11. COMPARE TARGET VS CONFIGURED-SCHEDULE PRIVACY
# ==================================================================================================

print("\n" + "=" * 100)
print("11. COMPARE TARGET VS CONFIGURED-SCHEDULE PRIVACY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate Section 9 dependency
# --------------------------------------------------------------------------------------------------

if "CONFIGURED_ACCOUNTING_DF" not in globals():

    raise RuntimeError(
        "CONFIGURED_ACCOUNTING_DF from Section 9 is unavailable."
    )

# --------------------------------------------------------------------------------------------------
# 2. Validate Section 9 schema
# --------------------------------------------------------------------------------------------------

REQUIRED_COLUMNS = {
    "dataset",
    "target_epsilon",
    "configured_schedule_epsilon",
    "epsilon_difference_from_target",
    "within_target_epsilon",
    "epsilon_budget_status",
    "optimal_rdp_order",
}

missing_columns = (
    REQUIRED_COLUMNS
    -
    set(CONFIGURED_ACCOUNTING_DF.columns)
)

if missing_columns:

    raise RuntimeError(
        "Configured accounting is missing required columns:\n"
        f"{sorted(missing_columns)}"
    )

# --------------------------------------------------------------------------------------------------
# 3. Validate Section 9 dataset coverage
# --------------------------------------------------------------------------------------------------

ACCOUNTING_DATASETS = set(
    CONFIGURED_ACCOUNTING_DF["dataset"]
    .astype(str)
)

EXPECTED_DATASETS = set(
    str(dataset_id)
    for dataset_id in DATASET_IDS
)

if ACCOUNTING_DATASETS != EXPECTED_DATASETS:

    raise RuntimeError(
        "Configured accounting dataset coverage is inconsistent.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found:    {sorted(ACCOUNTING_DATASETS)}"
    )

if len(CONFIGURED_ACCOUNTING_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Configured accounting must contain exactly one row per dataset."
    )

if (
    CONFIGURED_ACCOUNTING_DF["dataset"]
    .astype(str)
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Configured accounting contains duplicate dataset records."
    )

# --------------------------------------------------------------------------------------------------
# 4. Validate accounting provenance
# --------------------------------------------------------------------------------------------------

if "accounting_type" in CONFIGURED_ACCOUNTING_DF.columns:

    if not CONFIGURED_ACCOUNTING_DF[
        "accounting_type"
    ].eq(
        "configured_schedule"
    ).all():

        raise RuntimeError(
            "Accounting provenance is inconsistent. "
            "Section 11 requires configured-schedule accounting."
        )

else:

    print(
        "ℹ 'accounting_type' column not present in Section 9 output; "
        "provenance will be recorded explicitly in Section 11."
    )

# --------------------------------------------------------------------------------------------------
# 5. Validate achieved-epsilon provenance
# --------------------------------------------------------------------------------------------------

if "achieved_epsilon_status" in CONFIGURED_ACCOUNTING_DF.columns:

    if not CONFIGURED_ACCOUNTING_DF[
        "achieved_epsilon_status"
    ].eq(
        "DEFERRED_TO_NOTEBOOK_12"
    ).all():

        raise RuntimeError(
            "Achieved-epsilon provenance is inconsistent."
        )

else:

    print(
        "ℹ 'achieved_epsilon_status' column not present in Section 9 output; "
        "Section 11 will record achieved epsilon as deferred to Notebook 12."
    )

# --------------------------------------------------------------------------------------------------
# 6. Compare target epsilon with configured-schedule epsilon
#
# IMPORTANT:
#
# This section is an AUDIT / COMPARISON section.
#
# It does NOT perform noise-multiplier calibration.
# It does NOT require configured epsilon to equal the target epsilon.
# It does NOT fail when configured epsilon exceeds the target.
#
# The target epsilon is treated as a privacy-budget constraint:
#
#       configured epsilon <= target epsilon
#
# If the configured epsilon exceeds the target, the result is recorded as
# EXCEEDS_TARGET and the section continues successfully.
#
# Final achieved training epsilon remains deferred to Notebook 12.
# --------------------------------------------------------------------------------------------------

COMPARISON_ROWS = []

for row in CONFIGURED_ACCOUNTING_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    target_epsilon = float(
        row.target_epsilon
    )

    configured_epsilon = float(
        row.configured_schedule_epsilon
    )

    reported_difference = float(
        row.epsilon_difference_from_target
    )

    calculated_difference = (
        configured_epsilon -
        target_epsilon
    )

    absolute_difference = abs(
        calculated_difference
    )

    within_target_epsilon = (
        configured_epsilon <=
        target_epsilon
    )

    epsilon_ratio = (
        configured_epsilon /
        target_epsilon
    )

    # ----------------------------------------------------------------------------------------------
    # Numerical validation
    # ----------------------------------------------------------------------------------------------

    if not (
        np.isfinite(target_epsilon)
        and
        target_epsilon > 0
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid target epsilon."
        )

    if not (
        np.isfinite(configured_epsilon)
        and
        configured_epsilon >= 0
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid configured-schedule epsilon."
        )

    if not np.isfinite(
        reported_difference
    ):

        raise RuntimeError(
            f"{dataset_id}: non-finite reported epsilon difference."
        )

    if not np.isfinite(
        calculated_difference
    ):

        raise RuntimeError(
            f"{dataset_id}: non-finite calculated epsilon difference."
        )

    if not np.isfinite(
        absolute_difference
    ):

        raise RuntimeError(
            f"{dataset_id}: non-finite absolute epsilon difference."
        )

    if not np.isfinite(
        epsilon_ratio
    ):

        raise RuntimeError(
            f"{dataset_id}: non-finite epsilon ratio."
        )

    # ----------------------------------------------------------------------------------------------
    # Verify Section 9 epsilon difference
    # ----------------------------------------------------------------------------------------------

    difference_error = abs(
        reported_difference -
        calculated_difference
    )

    if difference_error > 1e-10:

        raise RuntimeError(
            f"{dataset_id}: Section 9 epsilon difference is inconsistent.\n"
            f"Reported difference : {reported_difference:.12f}\n"
            f"Calculated difference: {calculated_difference:.12f}\n"
            f"Absolute error       : {difference_error:.12e}"
        )

    # ----------------------------------------------------------------------------------------------
    # Verify Section 9 budget status
    # ----------------------------------------------------------------------------------------------

    expected_budget_status = (
        "WITHIN_TARGET"
        if within_target_epsilon
        else
        "EXCEEDS_TARGET"
    )

    reported_budget_status = str(
        row.epsilon_budget_status
    )

    if reported_budget_status != expected_budget_status:

        raise RuntimeError(
            f"{dataset_id}: Section 9 epsilon budget status is inconsistent.\n"
            f"Reported : {reported_budget_status}\n"
            f"Expected : {expected_budget_status}"
        )

    # ----------------------------------------------------------------------------------------------
    # Verify Section 9 within-target flag
    # ----------------------------------------------------------------------------------------------

    reported_within_target = bool(
        row.within_target_epsilon
    )

    if reported_within_target != within_target_epsilon:

        raise RuntimeError(
            f"{dataset_id}: Section 9 within-target flag is inconsistent.\n"
            f"Reported : {reported_within_target}\n"
            f"Expected : {within_target_epsilon}"
        )

    # ----------------------------------------------------------------------------------------------
    # Configure audit status
    # ----------------------------------------------------------------------------------------------

    if within_target_epsilon:

        configured_schedule_status = (
            "WITHIN_TARGET"
        )

    else:

        configured_schedule_status = (
            "EXCEEDS_TARGET"
        )

    # ----------------------------------------------------------------------------------------------
    # Optimal RDP order validation
    # ----------------------------------------------------------------------------------------------

    optimal_rdp_order = float(
        row.optimal_rdp_order
    )

    if not (
        np.isfinite(
            optimal_rdp_order
        )
        and
        optimal_rdp_order >= 1.01
        and
        optimal_rdp_order <= 1000
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid optimal RDP order "
            f"{optimal_rdp_order}."
        )

    # ----------------------------------------------------------------------------------------------
    # Append audit record
    # ----------------------------------------------------------------------------------------------

    COMPARISON_ROWS.append({

        "dataset":
            dataset_id,

        "target_epsilon":
            target_epsilon,

        "configured_schedule_epsilon":
            configured_epsilon,

        "epsilon_difference_from_target":
            calculated_difference,

        "absolute_epsilon_difference":
            absolute_difference,

        "epsilon_ratio":
            epsilon_ratio,

        "within_target_epsilon":
            within_target_epsilon,

        "epsilon_budget_status":
            expected_budget_status,

        "configured_schedule_status":
            configured_schedule_status,

        "optimal_rdp_order":
            optimal_rdp_order,

        "accounting_type":
            "configured_schedule",

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "end_to_end_privacy_claim":
            False,

        "status":
            "PASS",
    })


# --------------------------------------------------------------------------------------------------
# 7. Construct comparison DataFrame
# --------------------------------------------------------------------------------------------------

TARGET_COMPARISON_DF = pd.DataFrame(
    COMPARISON_ROWS
)

if TARGET_COMPARISON_DF.empty:

    raise RuntimeError(
        "No target-versus-configured-schedule comparison records generated."
    )

# --------------------------------------------------------------------------------------------------
# 8. Validate dataset coverage
# --------------------------------------------------------------------------------------------------

if len(
    TARGET_COMPARISON_DF
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Target comparison does not contain exactly one record per dataset."
    )

if set(
    TARGET_COMPARISON_DF[
        "dataset"
    ].astype(str)
) != set(
    str(dataset_id)
    for dataset_id in DATASET_IDS
):

    raise RuntimeError(
        "Target comparison dataset coverage is incomplete."
    )

# --------------------------------------------------------------------------------------------------
# 9. Validate comparison results
#
# IMPORTANT:
#     EXCEEDS_TARGET is a valid scientific result.
#     It is NOT treated as a computational failure.
# --------------------------------------------------------------------------------------------------

EXPECTED_STATUS_MAP = {}

for row in TARGET_COMPARISON_DF.itertuples(
    index=False
):

    expected_status = (
        "WITHIN_TARGET"
        if row.configured_schedule_epsilon
        <= row.target_epsilon
        else
        "EXCEEDS_TARGET"
    )

    EXPECTED_STATUS_MAP[
        str(row.dataset)
    ] = expected_status

for row in TARGET_COMPARISON_DF.itertuples(
    index=False
):

    if (
        str(row.epsilon_budget_status)
        !=
        EXPECTED_STATUS_MAP[
            str(row.dataset)
        ]
    ):

        raise RuntimeError(
            f"{row.dataset}: invalid epsilon budget classification."
        )

# --------------------------------------------------------------------------------------------------
# 10. Validate provenance fields
# --------------------------------------------------------------------------------------------------

if not TARGET_COMPARISON_DF[
    "accounting_type"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Invalid accounting provenance detected."
    )

if not TARGET_COMPARISON_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Invalid achieved-epsilon provenance detected."
    )

if not TARGET_COMPARISON_DF[
    "end_to_end_privacy_claim"
].eq(
    False
).all():

    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )

if not TARGET_COMPARISON_DF[
    "status"
].eq(
    "PASS"
).all():

    raise RuntimeError(
        "Target-versus-configured-schedule comparison failed."
    )

# --------------------------------------------------------------------------------------------------
# 11. Summarize budget status
# --------------------------------------------------------------------------------------------------

WITHIN_TARGET_COUNT = int(
    TARGET_COMPARISON_DF[
        "epsilon_budget_status"
    ].eq(
        "WITHIN_TARGET"
    ).sum()
)

EXCEEDS_TARGET_COUNT = int(
    TARGET_COMPARISON_DF[
        "epsilon_budget_status"
    ].eq(
        "EXCEEDS_TARGET"
    ).sum()
)

TOTAL_DATASET_COUNT = int(
    len(TARGET_COMPARISON_DF)
)

# --------------------------------------------------------------------------------------------------
# 12. Persist audit artifact
# --------------------------------------------------------------------------------------------------

TARGET_COMPARISON_PATH = (
    DIRS["audit"] /
    "target_vs_configured_schedule_epsilon.csv"
)

TARGET_COMPARISON_DF.to_csv(
    TARGET_COMPARISON_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 13. Display results
# --------------------------------------------------------------------------------------------------

display(
    TARGET_COMPARISON_DF
)

print(
    "\nTarget vs configured-schedule epsilon:"
)

for row in TARGET_COMPARISON_DF.itertuples(
    index=False
):

    print(
        f"  {row.dataset:18s} "
        f"target={row.target_epsilon:.6f} "
        f"ε_config={row.configured_schedule_epsilon:.6f} "
        f"|Δε|={row.absolute_epsilon_difference:.6f} "
        f"ratio={row.epsilon_ratio:.6f} "
        f"status={row.epsilon_budget_status}"
    )

# --------------------------------------------------------------------------------------------------
# 14. Budget summary
# --------------------------------------------------------------------------------------------------

print(
    "\nConfigured privacy-budget summary:"
)

print(
    f"  Total datasets             : "
    f"{TOTAL_DATASET_COUNT}"
)

print(
    f"  Within target              : "
    f"{WITHIN_TARGET_COUNT}"
)

print(
    f"  Exceeds target             : "
    f"{EXCEEDS_TARGET_COUNT}"
)

# --------------------------------------------------------------------------------------------------
# 15. Privacy provenance statement
# --------------------------------------------------------------------------------------------------

print(
    "\nPrivacy provenance:"
)

print(
    "  ✓ Comparison basis          : configured DP-SGD schedule"
)

print(
    "  ✓ RDP accountant            : Opacus"
)

print(
    "  ✓ Sampling mechanism        : Poisson"
)

print(
    "  ✓ Accounting type           : configured-schedule epsilon"
)

print(
    "  ✓ Target epsilon            : privacy-budget comparison"
)

print(
    "  ✓ Noise calibration        : "
    "not performed in Section 11"
)

print(
    "  ✓ Achieved training epsilon : "
    "deferred to Notebook 12"
)

print(
    "  ✓ End-to-end privacy claim  : "
    "not established"
)

# --------------------------------------------------------------------------------------------------
# 16. Final status
# --------------------------------------------------------------------------------------------------

print(
    f"\n✓ Target comparison audit saved:\n"
    f"  {TARGET_COMPARISON_PATH}"
)

print(
    "\nSECTION 11 STATUS: PASS"
)

print(
    "NOTE: EXCEEDS_TARGET results are retained as valid "
    "configured-schedule privacy findings and are not treated "
    "as computational failures."
)


11. COMPARE TARGET VS CONFIGURED-SCHEDULE PRIVACY


,dataset,target_epsilon,configured_schedule_epsilon,epsilon_difference_from_target,absolute_epsilon_difference,epsilon_ratio,within_target_epsilon,epsilon_budget_status,configured_schedule_status,optimal_rdp_order,accounting_type,achieved_epsilon_status,end_to_end_privacy_claim,status
0,adult_income,5.0,7.034634,2.034634,2.034634,1.406927,False,EXCEEDS_TARGET,EXCEEDS_TARGET,4.13,configured_schedule,DEFERRED_TO_NOTEBOOK_12,False,PASS
1,bank_marketing,5.0,7.363785,2.363785,2.363785,1.472757,False,EXCEEDS_TARGET,EXCEEDS_TARGET,4.01,configured_schedule,DEFERRED_TO_NOTEBOOK_12,False,PASS
2,diabetes_130us,5.0,4.572695,-0.427305,0.427305,0.914539,True,WITHIN_TARGET,WITHIN_TARGET,5.50,configured_schedule,DEFERRED_TO_NOTEBOOK_12,False,PASS



Target vs configured-schedule epsilon:
  adult_income       target=5.000000 ε_config=7.034634 |Δε|=2.034634 ratio=1.406927 status=EXCEEDS_TARGET
  bank_marketing     target=5.000000 ε_config=7.363785 |Δε|=2.363785 ratio=1.472757 status=EXCEEDS_TARGET
  diabetes_130us     target=5.000000 ε_config=4.572695 |Δε|=0.427305 ratio=0.914539 status=WITHIN_TARGET

Configured privacy-budget summary:
  Total datasets             : 3
  Within target              : 1
  Exceeds target             : 2

Privacy provenance:
  ✓ Comparison basis          : configured DP-SGD schedule
  ✓ RDP accountant            : Opacus
  ✓ Sampling mechanism        : Poisson
  ✓ Accounting type           : configured-schedule epsilon
  ✓ Target epsilon            : privacy-budget comparison
  ✓ Noise calibration        : not performed in Section 11
  ✓ Achieved training epsilon : deferred to Notebook 12
  ✓ End-to-end privacy claim  : not established

✓ Target comparison audit saved:
  /content/drive/MyDrive/SPP_GAN_R

In [14]:
# ==================================================================================================
# 12. GENERATE PER-EXPERIMENT PRIVACY RECORDS
# ==================================================================================================

print("\n" + "=" * 100)
print("12. GENERATE PER-EXPERIMENT PRIVACY RECORDS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate Section 9 dependency
# --------------------------------------------------------------------------------------------------

if "CONFIGURED_ACCOUNTING_DF" not in globals():

    raise RuntimeError(
        "CONFIGURED_ACCOUNTING_DF from Section 9 is unavailable."
    )

REQUIRED_COLUMNS = {
    "dataset",
    "n_train",
    "sample_rate",
    "noise_multiplier",
    "max_grad_norm",
    "epochs",
    "steps_per_epoch",
    "total_steps",
    "delta",
    "target_epsilon",
    "configured_schedule_epsilon",
    "optimal_rdp_order",
    "rdp_order_count",
    "accounting_type",
    "achieved_epsilon_status",
}

missing_columns = (
    REQUIRED_COLUMNS
    -
    set(CONFIGURED_ACCOUNTING_DF.columns)
)

if missing_columns:

    raise RuntimeError(
        "Configured accounting is missing required columns:\n"
        f"{sorted(missing_columns)}"
    )

# --------------------------------------------------------------------------------------------------
# 2. Validate accounting provenance
# --------------------------------------------------------------------------------------------------

if not CONFIGURED_ACCOUNTING_DF[
    "accounting_type"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Section 12 requires configured-schedule accounting records."
    )

if not CONFIGURED_ACCOUNTING_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Achieved-epsilon provenance is inconsistent."
    )

# --------------------------------------------------------------------------------------------------
# 3. Generate one privacy record per canonical dataset
#
# These are accounting records for the configured experimental protocol.
# They are NOT records of completed model-training experiments.
# --------------------------------------------------------------------------------------------------

PER_EXPERIMENT_ROWS = []

for row in CONFIGURED_ACCOUNTING_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    target_epsilon = float(
        row.target_epsilon
    )

    configured_schedule_epsilon = float(
        row.configured_schedule_epsilon
    )

    experiment_id = (
        f"{dataset_id}_"
        f"DP-SGD_"
        f"RDP_"
        f"configured-schedule_"
        f"eps{target_epsilon:g}"
    )

    # ----------------------------------------------------------------------------------------------
    # Numerical validation
    # ----------------------------------------------------------------------------------------------

    numeric_values = {
        "n_train":
            float(row.n_train),

        "sample_rate":
            float(row.sample_rate),

        "noise_multiplier":
            float(row.noise_multiplier),

        "max_grad_norm":
            float(row.max_grad_norm),

        "epochs":
            float(row.epochs),

        "steps_per_epoch":
            float(row.steps_per_epoch),

        "total_steps":
            float(row.total_steps),

        "delta":
            float(row.delta),

        "target_epsilon":
            target_epsilon,

        "configured_schedule_epsilon":
            configured_schedule_epsilon,

        "optimal_rdp_order":
            float(row.optimal_rdp_order),
    }

    for field_name, field_value in numeric_values.items():

        if not np.isfinite(
            field_value
        ):

            raise RuntimeError(
                f"{dataset_id}: non-finite value in '{field_name}'."
            )

    # ----------------------------------------------------------------------------------------------
    # Record
    # ----------------------------------------------------------------------------------------------

    PER_EXPERIMENT_ROWS.append({

        "experiment_id":
            experiment_id,

        "dataset":
            dataset_id,

        "protected_component":
            "SPP-GAN discriminator / critic",

        "mechanism":
            "DP-SGD",

        "accountant":
            "RDP",

        "sampling":
            "Poisson",

        "clipping":
            "flat L2",

        "loss_reduction":
            LOSS_REDUCTION,

        "n_train":
            int(row.n_train),

        "batch_size_nominal":
            int(DP_BATCH_SIZE),

        "sample_rate":
            float(row.sample_rate),

        "epochs":
            int(row.epochs),

        "steps_per_epoch":
            float(row.steps_per_epoch),

        "total_steps":
            float(row.total_steps),

        "max_grad_norm":
            float(row.max_grad_norm),

        "noise_multiplier":
            float(row.noise_multiplier),

        "delta":
            float(row.delta),

        "target_epsilon":
            target_epsilon,

        "configured_schedule_epsilon":
            configured_schedule_epsilon,

        "optimal_rdp_order":
            float(row.optimal_rdp_order),

        "rdp_order_count":
            int(row.rdp_order_count),

        "accounting_type":
            "configured_schedule",

        "training_performed":
            False,

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,

        "status":
            "PASS",
    })

# --------------------------------------------------------------------------------------------------
# 4. Construct DataFrame
# --------------------------------------------------------------------------------------------------

PER_EXPERIMENT_DF = pd.DataFrame(
    PER_EXPERIMENT_ROWS
)

if PER_EXPERIMENT_DF.empty:

    raise RuntimeError(
        "No per-experiment privacy records were generated."
    )

if len(
    PER_EXPERIMENT_DF
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Incorrect number of per-experiment privacy records."
    )

if set(
    PER_EXPERIMENT_DF["dataset"].astype(str)
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Per-experiment privacy dataset coverage is incomplete."
    )

# --------------------------------------------------------------------------------------------------
# 5. Validate privacy provenance
# --------------------------------------------------------------------------------------------------

if not PER_EXPERIMENT_DF[
    "accounting_type"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Invalid accounting type detected."
    )

if not PER_EXPERIMENT_DF[
    "training_performed"
].eq(
    False
).all():

    raise RuntimeError(
        "Configured-schedule records must not claim that training was performed."
    )

if not PER_EXPERIMENT_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Invalid achieved-epsilon status detected."
    )

if not PER_EXPERIMENT_DF[
    "end_to_end_privacy_claim"
].eq(
    False
).all():

    raise RuntimeError(
        "End-to-end privacy must remain unclaimed."
    )

# --------------------------------------------------------------------------------------------------
# 6. Validate mechanism configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_MECHANISM_FIELDS = {
    "protected_component":
        "SPP-GAN discriminator / critic",

    "mechanism":
        "DP-SGD",

    "accountant":
        "RDP",

    "sampling":
        "Poisson",

    "clipping":
        "flat L2",

    "loss_reduction":
        LOSS_REDUCTION,
}

for column_name, expected_value in (
    EXPECTED_MECHANISM_FIELDS.items()
):

    if not PER_EXPERIMENT_DF[
        column_name
    ].eq(
        expected_value
    ).all():

        raise RuntimeError(
            f"Mechanism configuration mismatch in '{column_name}'."
        )

# --------------------------------------------------------------------------------------------------
# 7. Persist per-experiment accounting records
# --------------------------------------------------------------------------------------------------

PER_EXPERIMENT_PATH = (
    DIRS["accounting"] /
    "per_experiment_privacy_records.csv"
)

PER_EXPERIMENT_DF.to_csv(
    PER_EXPERIMENT_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 8. Display records
# --------------------------------------------------------------------------------------------------

display(
    PER_EXPERIMENT_DF
)

print(
    "\nPer-experiment privacy records:"
)

for row in PER_EXPERIMENT_DF.itertuples(
    index=False
):

    print(
        f"  {row.dataset:18s} "
        f"ε_config={row.configured_schedule_epsilon:.6f} "
        f"δ={row.delta:.8g} "
        f"α*={row.optimal_rdp_order:.2f} "
        f"training={row.training_performed}"
    )

# --------------------------------------------------------------------------------------------------
# 9. Privacy provenance statement
# --------------------------------------------------------------------------------------------------

print(
    "\nPer-experiment record provenance:"
)

print(
    "  ✓ Record basis                  : configured privacy schedule"
)

print(
    "  ✓ Protected component           : SPP-GAN discriminator / critic"
)

print(
    "  ✓ Mechanism                     : DP-SGD"
)

print(
    "  ✓ Sampling                      : Poisson"
)

print(
    "  ✓ Accountant                    : RDP / Opacus"
)

print(
    "  ✓ Training performed            : False"
)

print(
    "  ✓ Achieved epsilon              : deferred to Notebook 12"
)

print(
    "  ✓ Generator independently private: False"
)

print(
    "  ✓ Statistical guidance private  : False"
)

print(
    "  ✓ Preprocessing private         : False"
)

print(
    "  ✓ End-to-end privacy claim      : False"
)

print(
    f"\n✓ Per-experiment records saved:\n"
    f"  {PER_EXPERIMENT_PATH}"
)

print(
    "SECTION 12 STATUS: PASS"
)


12. GENERATE PER-EXPERIMENT PRIVACY RECORDS


,experiment_id,dataset,protected_component,mechanism,accountant,sampling,clipping,loss_reduction,n_train,batch_size_nominal,...,optimal_rdp_order,rdp_order_count,accounting_type,training_performed,achieved_epsilon_status,generator_private,statistical_guidance_private,preprocessing_private,end_to_end_privacy_claim,status
0,adult_income_DP-SGD_RDP_configured-schedule_eps5,adult_income,SPP-GAN discriminator / critic,DP-SGD,RDP,Poisson,flat L2,mean,34189,128,...,4.13,1890,configured_schedule,False,DEFERRED_TO_NOTEBOOK_12,False,False,False,False,PASS
1,bank_marketing_DP-SGD_RDP_configured-schedule_...,bank_marketing,SPP-GAN discriminator / critic,DP-SGD,RDP,Poisson,flat L2,mean,31647,128,...,4.01,1890,configured_schedule,False,DEFERRED_TO_NOTEBOOK_12,False,False,False,False,PASS
2,diabetes_130us_DP-SGD_RDP_configured-schedule_...,diabetes_130us,SPP-GAN discriminator / critic,DP-SGD,RDP,Poisson,flat L2,mean,71236,128,...,5.50,1890,configured_schedule,False,DEFERRED_TO_NOTEBOOK_12,False,False,False,False,PASS



Per-experiment privacy records:
  adult_income       ε_config=7.034634 δ=1e-05 α*=4.13 training=False
  bank_marketing     ε_config=7.363785 δ=1e-05 α*=4.01 training=False
  diabetes_130us     ε_config=4.572695 δ=1e-05 α*=5.50 training=False

Per-experiment record provenance:
  ✓ Record basis                  : configured privacy schedule
  ✓ Protected component           : SPP-GAN discriminator / critic
  ✓ Mechanism                     : DP-SGD
  ✓ Sampling                      : Poisson
  ✓ Accountant                    : RDP / Opacus
  ✓ Training performed            : False
  ✓ Achieved epsilon              : deferred to Notebook 12
  ✓ Generator independently private: False
  ✓ Statistical guidance private  : False
  ✓ Preprocessing private         : False
  ✓ End-to-end privacy claim      : False

✓ Per-experiment records saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/accounting/per_experiment_privacy_records.csv
SECTION 12 STATUS: PASS


In [15]:
# ==================================================================================================
# 13. GENERATE DATASET-LEVEL PRIVACY SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("13. GENERATE DATASET-LEVEL PRIVACY SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate Section 9 dependency
# --------------------------------------------------------------------------------------------------

if "CONFIGURED_ACCOUNTING_DF" not in globals():

    raise RuntimeError(
        "CONFIGURED_ACCOUNTING_DF from Section 9 is unavailable."
    )

# --------------------------------------------------------------------------------------------------
# 2. Required Section 9 columns
# --------------------------------------------------------------------------------------------------

REQUIRED_COLUMNS = [
    "dataset",
    "n_train",
    "target_epsilon",
    "configured_schedule_epsilon",
    "delta",
    "sample_rate",
    "noise_multiplier",
    "steps_per_epoch",
    "total_steps",
    "optimal_rdp_order",
    "rdp_order_count",
    "accounting_type",
    "achieved_epsilon_status",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in CONFIGURED_ACCOUNTING_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Configured accounting table is missing required columns:\n"
        f"{missing_columns}"
    )

# --------------------------------------------------------------------------------------------------
# 3. Validate accounting provenance
# --------------------------------------------------------------------------------------------------

if not CONFIGURED_ACCOUNTING_DF[
    "accounting_type"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Section 13 requires configured-schedule accounting records."
    )

if not CONFIGURED_ACCOUNTING_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Achieved epsilon status is inconsistent with the "
        "Notebook 11 privacy boundary."
    )

# --------------------------------------------------------------------------------------------------
# 4. Validate canonical dataset coverage
# --------------------------------------------------------------------------------------------------

ACCOUNTING_DATASETS = set(
    CONFIGURED_ACCOUNTING_DF[
        "dataset"
    ].astype(str)
)

EXPECTED_DATASETS = set(
    str(dataset_id)
    for dataset_id in DATASET_IDS
)

if ACCOUNTING_DATASETS != EXPECTED_DATASETS:

    raise RuntimeError(
        "Dataset coverage does not match the canonical dataset registry.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found:    {sorted(ACCOUNTING_DATASETS)}"
    )

if len(
    CONFIGURED_ACCOUNTING_DF
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Dataset-level summary must contain exactly one record "
        "per canonical dataset."
    )

if CONFIGURED_ACCOUNTING_DF[
    "dataset"
].astype(str).duplicated().any():

    raise RuntimeError(
        "Duplicate dataset records detected in configured accounting."
    )

# --------------------------------------------------------------------------------------------------
# 5. Create dataset-level summary
#
# IMPORTANT:
#
# This section summarizes the configured privacy schedule.
#
# It does NOT perform noise calibration.
# It does NOT require configured epsilon to equal target epsilon.
# It does NOT treat EXCEEDS_TARGET as a computational failure.
#
# The target epsilon is a privacy-budget constraint:
#
#       configured epsilon <= target epsilon
#
# Current configured results may therefore legitimately contain
# EXCEEDS_TARGET records.
# --------------------------------------------------------------------------------------------------

DATASET_PRIVACY_SUMMARY_DF = (
    CONFIGURED_ACCOUNTING_DF[
        REQUIRED_COLUMNS
    ]
    .copy()
)

# --------------------------------------------------------------------------------------------------
# 6. Calculate epsilon differences
# --------------------------------------------------------------------------------------------------

DATASET_PRIVACY_SUMMARY_DF[
    "epsilon_difference"
] = (
    DATASET_PRIVACY_SUMMARY_DF[
        "configured_schedule_epsilon"
    ]
    -
    DATASET_PRIVACY_SUMMARY_DF[
        "target_epsilon"
    ]
)

DATASET_PRIVACY_SUMMARY_DF[
    "absolute_epsilon_difference"
] = (
    DATASET_PRIVACY_SUMMARY_DF[
        "epsilon_difference"
    ]
    .abs()
)

DATASET_PRIVACY_SUMMARY_DF[
    "epsilon_ratio"
] = (
    DATASET_PRIVACY_SUMMARY_DF[
        "configured_schedule_epsilon"
    ]
    /
    DATASET_PRIVACY_SUMMARY_DF[
        "target_epsilon"
    ]
)

# --------------------------------------------------------------------------------------------------
# 7. Classify configured privacy-budget status
# --------------------------------------------------------------------------------------------------

DATASET_PRIVACY_SUMMARY_DF[
    "within_target_epsilon"
] = (
    DATASET_PRIVACY_SUMMARY_DF[
        "configured_schedule_epsilon"
    ]
    <=
    DATASET_PRIVACY_SUMMARY_DF[
        "target_epsilon"
    ]
)

DATASET_PRIVACY_SUMMARY_DF[
    "epsilon_budget_status"
] = np.where(
    DATASET_PRIVACY_SUMMARY_DF[
        "within_target_epsilon"
    ],
    "WITHIN_TARGET",
    "EXCEEDS_TARGET",
)

DATASET_PRIVACY_SUMMARY_DF[
    "configured_schedule_status"
] = (
    DATASET_PRIVACY_SUMMARY_DF[
        "epsilon_budget_status"
    ]
)

# --------------------------------------------------------------------------------------------------
# 8. Validate numerical values
# --------------------------------------------------------------------------------------------------

NUMERIC_COLUMNS = [
    "n_train",
    "target_epsilon",
    "configured_schedule_epsilon",
    "delta",
    "sample_rate",
    "noise_multiplier",
    "steps_per_epoch",
    "total_steps",
    "optimal_rdp_order",
    "rdp_order_count",
    "epsilon_difference",
    "absolute_epsilon_difference",
    "epsilon_ratio",
]

for column in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        DATASET_PRIVACY_SUMMARY_DF[
            column
        ],
        errors="coerce",
    )

    if not np.all(
        np.isfinite(
            values.to_numpy()
        )
    ):

        raise RuntimeError(
            f"Non-finite values detected in column: {column}"
        )

# --------------------------------------------------------------------------------------------------
# 9. Validate epsilon-budget classification
# --------------------------------------------------------------------------------------------------

for row in DATASET_PRIVACY_SUMMARY_DF.itertuples(
    index=False
):

    expected_status = (
        "WITHIN_TARGET"
        if row.configured_schedule_epsilon
        <= row.target_epsilon
        else
        "EXCEEDS_TARGET"
    )

    if row.epsilon_budget_status != expected_status:

        raise RuntimeError(
            f"{row.dataset}: invalid epsilon budget classification.\n"
            f"Reported: {row.epsilon_budget_status}\n"
            f"Expected: {expected_status}"
        )

    expected_difference = (
        row.configured_schedule_epsilon
        -
        row.target_epsilon
    )

    difference_error = abs(
        row.epsilon_difference -
        expected_difference
    )

    if difference_error > 1e-10:

        raise RuntimeError(
            f"{row.dataset}: epsilon difference is inconsistent.\n"
            f"Reported: {row.epsilon_difference:.12f}\n"
            f"Expected: {expected_difference:.12f}"
        )

# --------------------------------------------------------------------------------------------------
# 10. Add explicit accounting scope and privacy boundary
# --------------------------------------------------------------------------------------------------

DATASET_PRIVACY_SUMMARY_DF[
    "privacy_accounting_scope"
] = (
    "configured_schedule"
)

DATASET_PRIVACY_SUMMARY_DF[
    "training_performed"
] = False

DATASET_PRIVACY_SUMMARY_DF[
    "achieved_epsilon_status"
] = (
    "DEFERRED_TO_NOTEBOOK_12"
)

DATASET_PRIVACY_SUMMARY_DF[
    "end_to_end_privacy_claim"
] = False

DATASET_PRIVACY_SUMMARY_DF[
    "status"
] = "PASS"

# --------------------------------------------------------------------------------------------------
# 11. Validate privacy provenance
# --------------------------------------------------------------------------------------------------

if not DATASET_PRIVACY_SUMMARY_DF[
    "privacy_accounting_scope"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Invalid privacy accounting scope detected."
    )

if not DATASET_PRIVACY_SUMMARY_DF[
    "training_performed"
].eq(
    False
).all():

    raise RuntimeError(
        "Configured-schedule records must not claim completed training."
    )

if not DATASET_PRIVACY_SUMMARY_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Invalid achieved-epsilon provenance detected."
    )

if not DATASET_PRIVACY_SUMMARY_DF[
    "end_to_end_privacy_claim"
].eq(
    False
).all():

    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )

# --------------------------------------------------------------------------------------------------
# 12. Count privacy-budget classifications
# --------------------------------------------------------------------------------------------------

WITHIN_TARGET_COUNT = int(
    DATASET_PRIVACY_SUMMARY_DF[
        "epsilon_budget_status"
    ].eq(
        "WITHIN_TARGET"
    ).sum()
)

EXCEEDS_TARGET_COUNT = int(
    DATASET_PRIVACY_SUMMARY_DF[
        "epsilon_budget_status"
    ].eq(
        "EXCEEDS_TARGET"
    ).sum()
)

TOTAL_DATASET_COUNT = int(
    len(
        DATASET_PRIVACY_SUMMARY_DF
    )
)

if (
    WITHIN_TARGET_COUNT +
    EXCEEDS_TARGET_COUNT
) != TOTAL_DATASET_COUNT:

    raise RuntimeError(
        "Privacy-budget classification counts are inconsistent."
    )

# --------------------------------------------------------------------------------------------------
# 13. Persist dataset-level summary
# --------------------------------------------------------------------------------------------------

DATASET_SUMMARY_PATH = (
    DIRS["accounting"] /
    "dataset_privacy_summary.csv"
)

DATASET_PRIVACY_SUMMARY_DF.to_csv(
    DATASET_SUMMARY_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 14. Display summary
# --------------------------------------------------------------------------------------------------

display(
    DATASET_PRIVACY_SUMMARY_DF
)

print(
    "\nDataset-level configured-schedule privacy summary:"
)

for row in DATASET_PRIVACY_SUMMARY_DF.itertuples(
    index=False
):

    print(
        f"  {row.dataset:18s} "
        f"target={row.target_epsilon:.6f} "
        f"ε_config={row.configured_schedule_epsilon:.6f} "
        f"δ={row.delta:.8g} "
        f"α*={row.optimal_rdp_order:.2f} "
        f"|Δε|={row.absolute_epsilon_difference:.6f} "
        f"ratio={row.epsilon_ratio:.6f} "
        f"status={row.epsilon_budget_status}"
    )

# --------------------------------------------------------------------------------------------------
# 15. Privacy-budget summary
# --------------------------------------------------------------------------------------------------

print(
    "\nConfigured privacy-budget summary:"
)

print(
    f"  Total datasets             : "
    f"{TOTAL_DATASET_COUNT}"
)

print(
    f"  Within target              : "
    f"{WITHIN_TARGET_COUNT}"
)

print(
    f"  Exceeds target             : "
    f"{EXCEEDS_TARGET_COUNT}"
)

# --------------------------------------------------------------------------------------------------
# 16. Provenance statement
# --------------------------------------------------------------------------------------------------

print(
    "\nDataset-level privacy provenance:"
)

print(
    "  ✓ Accounting scope             : configured schedule"
)

print(
    "  ✓ Accountant                   : RDP / Opacus"
)

print(
    "  ✓ Sampling mechanism           : Poisson"
)

print(
    f"  ✓ DP epochs                    : {DP_EPOCHS}"
)

print(
    "  ✓ Noise calibration            : "
    "not performed in Section 13"
)

print(
    "  ✓ Target comparison            : "
    "privacy-budget classification"
)

print(
    "  ✓ Training performed           : False"
)

print(
    "  ✓ Achieved epsilon             : "
    "deferred to Notebook 12"
)

print(
    "  ✓ End-to-end privacy claim     : False"
)

# --------------------------------------------------------------------------------------------------
# 17. Final status
# --------------------------------------------------------------------------------------------------

print(
    f"\n✓ Dataset-level privacy summary saved:\n"
    f"  {DATASET_SUMMARY_PATH}"
)

print(
    "\nSECTION 13 STATUS: PASS"
)

print(
    "NOTE: EXCEEDS_TARGET results are retained as valid "
    "configured-schedule privacy findings and are not treated "
    "as computational failures."
)


13. GENERATE DATASET-LEVEL PRIVACY SUMMARY


,dataset,n_train,target_epsilon,configured_schedule_epsilon,delta,sample_rate,noise_multiplier,steps_per_epoch,total_steps,optimal_rdp_order,...,epsilon_difference,absolute_epsilon_difference,epsilon_ratio,within_target_epsilon,epsilon_budget_status,configured_schedule_status,privacy_accounting_scope,training_performed,end_to_end_privacy_claim,status
0,adult_income,34189,5.0,7.034634,0.00001,0.003744,1.0,268,80400,4.13,...,2.034634,2.034634,1.406927,False,EXCEEDS_TARGET,EXCEEDS_TARGET,configured_schedule,False,False,PASS
1,bank_marketing,31647,5.0,7.363785,0.00001,0.004045,1.0,248,74400,4.01,...,2.363785,2.363785,1.472757,False,EXCEEDS_TARGET,EXCEEDS_TARGET,configured_schedule,False,False,PASS
2,diabetes_130us,71236,5.0,4.572695,0.00001,0.001797,1.0,557,167100,5.50,...,-0.427305,0.427305,0.914539,True,WITHIN_TARGET,WITHIN_TARGET,configured_schedule,False,False,PASS



Dataset-level configured-schedule privacy summary:
  adult_income       target=5.000000 ε_config=7.034634 δ=1e-05 α*=4.13 |Δε|=2.034634 ratio=1.406927 status=EXCEEDS_TARGET
  bank_marketing     target=5.000000 ε_config=7.363785 δ=1e-05 α*=4.01 |Δε|=2.363785 ratio=1.472757 status=EXCEEDS_TARGET
  diabetes_130us     target=5.000000 ε_config=4.572695 δ=1e-05 α*=5.50 |Δε|=0.427305 ratio=0.914539 status=WITHIN_TARGET

Configured privacy-budget summary:
  Total datasets             : 3
  Within target              : 1
  Exceeds target             : 2

Dataset-level privacy provenance:
  ✓ Accounting scope             : configured schedule
  ✓ Accountant                   : RDP / Opacus
  ✓ Sampling mechanism           : Poisson
  ✓ DP epochs                    : 300
  ✓ Noise calibration            : not performed in Section 13
  ✓ Target comparison            : privacy-budget classification
  ✓ Training performed           : False
  ✓ Achieved epsilon             : deferred to Notebook 12


In [16]:
# ==================================================================================================
# 14. GENERATE MODEL-LEVEL PRIVACY SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("14. GENERATE MODEL-LEVEL PRIVACY SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate Section 9 accounting dependency
# --------------------------------------------------------------------------------------------------

if "CONFIGURED_ACCOUNTING_DF" not in globals():

    raise RuntimeError(
        "CONFIGURED_ACCOUNTING_DF from Section 9 is unavailable."
    )

REQUIRED_COLUMNS = [
    "dataset",
    "n_train",
    "target_epsilon",
    "configured_schedule_epsilon",
    "delta",
    "sample_rate",
    "noise_multiplier",
    "total_steps",
    "optimal_rdp_order",
    "accounting_type",
    "achieved_epsilon_status",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in CONFIGURED_ACCOUNTING_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Configured accounting table is missing required columns:\n"
        f"{missing_columns}"
    )

# --------------------------------------------------------------------------------------------------
# 2. Validate configured-schedule provenance
# --------------------------------------------------------------------------------------------------

if not CONFIGURED_ACCOUNTING_DF[
    "accounting_type"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Model-level summary requires configured-schedule accounting."
    )

if not CONFIGURED_ACCOUNTING_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Achieved epsilon status is inconsistent with Notebook 11."
    )

# --------------------------------------------------------------------------------------------------
# 3. Validate canonical dataset coverage
# --------------------------------------------------------------------------------------------------

ACCOUNTING_DATASETS = set(
    CONFIGURED_ACCOUNTING_DF[
        "dataset"
    ].astype(str)
)

EXPECTED_DATASETS = set(
    str(dataset_id)
    for dataset_id in DATASET_IDS
)

if ACCOUNTING_DATASETS != EXPECTED_DATASETS:

    raise RuntimeError(
        "Configured accounting does not cover exactly the canonical datasets.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found:    {sorted(ACCOUNTING_DATASETS)}"
    )

if len(
    CONFIGURED_ACCOUNTING_DF
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Expected exactly one configured accounting record per dataset."
    )

if CONFIGURED_ACCOUNTING_DF[
    "dataset"
].astype(str).duplicated().any():

    raise RuntimeError(
        "Duplicate configured accounting records detected."
    )

# --------------------------------------------------------------------------------------------------
# 4. Extract configured-schedule epsilon values
# --------------------------------------------------------------------------------------------------

CONFIGURED_EPSILON = pd.to_numeric(
    CONFIGURED_ACCOUNTING_DF[
        "configured_schedule_epsilon"
    ],
    errors="coerce",
)

TARGET_EPSILON_SERIES = pd.to_numeric(
    CONFIGURED_ACCOUNTING_DF[
        "target_epsilon"
    ],
    errors="coerce",
)

EPSILON_DIFFERENCE = (
    CONFIGURED_EPSILON
    -
    TARGET_EPSILON_SERIES
)

ABSOLUTE_EPSILON_DIFFERENCE = (
    EPSILON_DIFFERENCE
    .abs()
)

EPSILON_RATIO = (
    CONFIGURED_EPSILON
    /
    TARGET_EPSILON_SERIES
)

# --------------------------------------------------------------------------------------------------
# 5. Validate numerical accounting values
# --------------------------------------------------------------------------------------------------

NUMERIC_COLUMNS = [
    "n_train",
    "target_epsilon",
    "configured_schedule_epsilon",
    "delta",
    "sample_rate",
    "noise_multiplier",
    "total_steps",
    "optimal_rdp_order",
]

for column in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        CONFIGURED_ACCOUNTING_DF[
            column
        ],
        errors="coerce",
    )

    if not np.all(
        np.isfinite(
            values.to_numpy()
        )
    ):

        raise RuntimeError(
            f"Non-finite values detected in column: {column}"
        )

if not np.all(
    np.isfinite(
        CONFIGURED_EPSILON.to_numpy()
    )
):

    raise RuntimeError(
        "Non-finite configured-schedule epsilon values detected."
    )

if not np.all(
    np.isfinite(
        EPSILON_DIFFERENCE.to_numpy()
    )
):

    raise RuntimeError(
        "Non-finite epsilon differences detected."
    )

if not np.all(
    np.isfinite(
        EPSILON_RATIO.to_numpy()
    )
):

    raise RuntimeError(
        "Non-finite epsilon ratios detected."
    )

# --------------------------------------------------------------------------------------------------
# 6. Classify dataset-level privacy-budget status
#
# IMPORTANT:
#
# This is NOT a calibration test.
#
# Configured epsilon is compared against the target privacy budget:
#
#     configured epsilon <= target epsilon
#
# EXCEEDS_TARGET is a valid scientific result and does not cause
# the model-level summary to fail.
# --------------------------------------------------------------------------------------------------

WITHIN_TARGET = (
    CONFIGURED_EPSILON
    <=
    TARGET_EPSILON_SERIES
)

EPSILON_BUDGET_STATUS = np.where(
    WITHIN_TARGET,
    "WITHIN_TARGET",
    "EXCEEDS_TARGET",
)

WITHIN_TARGET_COUNT = int(
    WITHIN_TARGET.sum()
)

EXCEEDS_TARGET_COUNT = int(
    (~WITHIN_TARGET).sum()
)

TOTAL_DATASET_COUNT = int(
    len(DATASET_IDS)
)

# --------------------------------------------------------------------------------------------------
# 7. Validate budget classification
# --------------------------------------------------------------------------------------------------

if (
    WITHIN_TARGET_COUNT
    +
    EXCEEDS_TARGET_COUNT
) != TOTAL_DATASET_COUNT:

    raise RuntimeError(
        "Privacy-budget classification counts are inconsistent."
    )

for index, row in CONFIGURED_ACCOUNTING_DF.iterrows():

    dataset_id = str(
        row["dataset"]
    )

    configured_epsilon = float(
        row["configured_schedule_epsilon"]
    )

    target_epsilon = float(
        row["target_epsilon"]
    )

    expected_status = (
        "WITHIN_TARGET"
        if configured_epsilon <= target_epsilon
        else
        "EXCEEDS_TARGET"
    )

    if EPSILON_BUDGET_STATUS[index] != expected_status:

        raise RuntimeError(
            f"{dataset_id}: invalid epsilon budget classification."
        )

# --------------------------------------------------------------------------------------------------
# 8. Generate model-level privacy summary
# --------------------------------------------------------------------------------------------------

MODEL_LEVEL_PRIVACY_SUMMARY = {

    "model":
        FRAMEWORK_NAME,

    "protected_component":
        "SPP-GAN discriminator / critic",

    "mechanism":
        "DP-SGD",

    "accountant":
        "RDP / Opacus",

    "sampling":
        "Poisson",

    "clipping":
        "flat L2",

    "loss_reduction":
        "mean",

    "target_epsilon":
        float(
            TARGET_EPSILON
        ),

    "datasets_accounted":
        TOTAL_DATASET_COUNT,

    "datasets_within_target":
        WITHIN_TARGET_COUNT,

    "datasets_exceeding_target":
        EXCEEDS_TARGET_COUNT,

    "configured_schedule_epsilon_min":
        float(
            CONFIGURED_EPSILON.min()
        ),

    "configured_schedule_epsilon_max":
        float(
            CONFIGURED_EPSILON.max()
        ),

    "configured_schedule_epsilon_mean":
        float(
            CONFIGURED_EPSILON.mean()
        ),

    "absolute_epsilon_difference_max":
        float(
            ABSOLUTE_EPSILON_DIFFERENCE.max()
        ),

    "epsilon_ratio_min":
        float(
            EPSILON_RATIO.min()
        ),

    "epsilon_ratio_max":
        float(
            EPSILON_RATIO.max()
        ),

    "training_performed":
        False,

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,

    "achieved_epsilon_status":
        "DEFERRED_TO_NOTEBOOK_12",

    "accounting_type":
        "configured_schedule",

    "noise_calibration_status":
        "NOT_PERFORMED",

    "status":
        "PASS",
}

MODEL_SUMMARY_DF = pd.DataFrame(
    [
        MODEL_LEVEL_PRIVACY_SUMMARY
    ]
)

# --------------------------------------------------------------------------------------------------
# 9. Validate model-level summary
# --------------------------------------------------------------------------------------------------

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "datasets_accounted"
    ]
    !=
    TOTAL_DATASET_COUNT
):

    raise RuntimeError(
        "Model-level dataset count does not match "
        "the canonical dataset registry."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "datasets_within_target"
    ]
    +
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "datasets_exceeding_target"
    ]
    !=
    TOTAL_DATASET_COUNT
):

    raise RuntimeError(
        "Model-level privacy-budget counts are inconsistent."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "configured_schedule_epsilon_min"
    ]
    >
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "configured_schedule_epsilon_max"
    ]
):

    raise RuntimeError(
        "Invalid configured-schedule epsilon range."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "configured_schedule_epsilon_min"
    ]
    !=
    float(
        CONFIGURED_EPSILON.min()
    )
):

    raise RuntimeError(
        "Configured epsilon minimum is inconsistent."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "configured_schedule_epsilon_max"
    ]
    !=
    float(
        CONFIGURED_EPSILON.max()
    )
):

    raise RuntimeError(
        "Configured epsilon maximum is inconsistent."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "training_performed"
    ]
    is not False
):

    raise RuntimeError(
        "Training must remain marked as not performed in Notebook 11."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "end_to_end_privacy_claim"
    ]
    is not False
):

    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "achieved_epsilon_status"
    ]
    !=
    "DEFERRED_TO_NOTEBOOK_12"
):

    raise RuntimeError(
        "Achieved epsilon must remain deferred to Notebook 12."
    )

if (
    MODEL_LEVEL_PRIVACY_SUMMARY[
        "noise_calibration_status"
    ]
    !=
    "NOT_PERFORMED"
):

    raise RuntimeError(
        "Noise calibration status is inconsistent."
    )

# --------------------------------------------------------------------------------------------------
# 10. Persist model-level summary
# --------------------------------------------------------------------------------------------------

MODEL_SUMMARY_PATH = (
    DIRS["accounting"] /
    "model_level_privacy_summary.csv"
)

MODEL_SUMMARY_DF.to_csv(
    MODEL_SUMMARY_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 11. Display model-level summary
# --------------------------------------------------------------------------------------------------

display(
    MODEL_SUMMARY_DF
)

print(
    "\nModel-level configured-schedule privacy summary:"
)

print(
    f"  Model                         : "
    f"{FRAMEWORK_NAME}"
)

print(
    "  Protected component           : "
    "SPP-GAN discriminator / critic"
)

print(
    "  Mechanism                     : DP-SGD"
)

print(
    "  Accountant                    : RDP / Opacus"
)

print(
    "  Sampling                      : Poisson"
)

print(
    "  Clipping                      : flat L2"
)

print(
    "  Loss reduction                : mean"
)

print(
    f"  Target epsilon                : "
    f"{TARGET_EPSILON:.6f}"
)

print(
    f"  Datasets accounted            : "
    f"{TOTAL_DATASET_COUNT}"
)

print(
    f"  Datasets within target        : "
    f"{WITHIN_TARGET_COUNT}"
)

print(
    f"  Datasets exceeding target     : "
    f"{EXCEEDS_TARGET_COUNT}"
)

print(
    f"  Configured epsilon range      : "
    f"{MODEL_LEVEL_PRIVACY_SUMMARY['configured_schedule_epsilon_min']:.6f}"
    f" to "
    f"{MODEL_LEVEL_PRIVACY_SUMMARY['configured_schedule_epsilon_max']:.6f}"
)

print(
    f"  Configured epsilon mean       : "
    f"{MODEL_LEVEL_PRIVACY_SUMMARY['configured_schedule_epsilon_mean']:.6f}"
)

print(
    f"  Maximum |Δε|                  : "
    f"{MODEL_LEVEL_PRIVACY_SUMMARY['absolute_epsilon_difference_max']:.6f}"
)

print(
    f"  Epsilon ratio range           : "
    f"{MODEL_LEVEL_PRIVACY_SUMMARY['epsilon_ratio_min']:.6f}"
    f" to "
    f"{MODEL_LEVEL_PRIVACY_SUMMARY['epsilon_ratio_max']:.6f}"
)

print(
    "  Noise calibration             : "
    "NOT PERFORMED"
)

# --------------------------------------------------------------------------------------------------
# 12. Privacy boundary and provenance
# --------------------------------------------------------------------------------------------------

print(
    "\nModel-level privacy provenance:"
)

print(
    "  ✓ Accounting scope             : configured schedule"
)

print(
    "  ✓ Accountant                   : RDP / Opacus"
)

print(
    "  ✓ Sampling mechanism           : Poisson"
)

print(
    f"  ✓ DP epochs                    : {DP_EPOCHS}"
)

print(
    "  ✓ Target comparison            : privacy-budget classification"
)

print(
    "  ✓ Noise calibration            : not performed in Section 14"
)

print(
    "  ✓ Training performed           : False"
)

print(
    "  ✓ Achieved epsilon             : deferred to Notebook 12"
)

print(
    "  ✓ Generator private            : False"
)

print(
    "  ✓ Statistical guidance private : False"
)

print(
    "  ✓ Preprocessing private        : False"
)

print(
    "  ✓ End-to-end privacy claim     : False"
)

# --------------------------------------------------------------------------------------------------
# 13. Final persistence confirmation
# --------------------------------------------------------------------------------------------------

if not MODEL_SUMMARY_PATH.exists():

    raise RuntimeError(
        "Model-level privacy summary was not persisted successfully."
    )

print(
    f"\n✓ Model-level privacy summary saved:\n"
    f"  {MODEL_SUMMARY_PATH}"
)

print(
    "\nSECTION 14 STATUS: PASS"
)

print(
    "NOTE: EXCEEDS_TARGET results are retained as valid "
    "configured-schedule privacy findings and are not treated "
    "as computational failures."
)


14. GENERATE MODEL-LEVEL PRIVACY SUMMARY


,model,protected_component,mechanism,accountant,sampling,clipping,loss_reduction,target_epsilon,datasets_accounted,datasets_within_target,...,epsilon_ratio_max,training_performed,generator_private,statistical_guidance_private,preprocessing_private,end_to_end_privacy_claim,achieved_epsilon_status,accounting_type,noise_calibration_status,status
0,SPP-GAN,SPP-GAN discriminator / critic,DP-SGD,RDP / Opacus,Poisson,flat L2,mean,5.0,3,1,...,1.472757,False,False,False,False,False,DEFERRED_TO_NOTEBOOK_12,configured_schedule,NOT_PERFORMED,PASS



Model-level configured-schedule privacy summary:
  Model                         : SPP-GAN
  Protected component           : SPP-GAN discriminator / critic
  Mechanism                     : DP-SGD
  Accountant                    : RDP / Opacus
  Sampling                      : Poisson
  Clipping                      : flat L2
  Loss reduction                : mean
  Target epsilon                : 5.000000
  Datasets accounted            : 3
  Datasets within target        : 1
  Datasets exceeding target     : 2
  Configured epsilon range      : 4.572695 to 7.363785
  Configured epsilon mean       : 6.323705
  Maximum |Δε|                  : 2.363785
  Epsilon ratio range           : 0.914539 to 1.472757
  Noise calibration             : NOT PERFORMED

Model-level privacy provenance:
  ✓ Accounting scope             : configured schedule
  ✓ Accountant                   : RDP / Opacus
  ✓ Sampling mechanism           : Poisson
  ✓ DP epochs                    : 300
  ✓ Target comparis

In [19]:
# ==================================================================================================
# 15. SAVE PRIVACY ACCOUNTING
# ==================================================================================================

print("\n" + "=" * 100)
print("15. SAVE PRIVACY ACCOUNTING")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Required imports
# --------------------------------------------------------------------------------------------------

import json
import hashlib
import numpy as np
import pandas as pd

from datetime import datetime, timezone


# --------------------------------------------------------------------------------------------------
# 2. Validate required upstream objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "NOTEBOOK_ID",
    "NOTEBOOK_NAME",
    "FRAMEWORK_NAME",
    "TARGET_EPSILON",
    "DELTA_RULE",
    "MAX_GRAD_NORM",
    "DP_BATCH_SIZE",
    "DP_EPOCHS",
    "RDP_ALPHAS",
    "DATASET_IDS",
    "NB10_ROOT",
    "DIRS",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required upstream objects are unavailable:\n"
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate RDP order grid
# --------------------------------------------------------------------------------------------------

if not isinstance(RDP_ALPHAS, np.ndarray):
    RDP_ALPHAS = np.asarray(
        RDP_ALPHAS,
        dtype=float,
    )

if len(RDP_ALPHAS) == 0:
    raise RuntimeError(
        "RDP order grid is empty."
    )

if not np.all(
    np.isfinite(RDP_ALPHAS)
):
    raise RuntimeError(
        "RDP order grid contains non-finite values."
    )

if not np.all(
    RDP_ALPHAS > 1
):
    raise RuntimeError(
        "All RDP orders must be greater than 1."
    )

if not np.all(
    np.diff(RDP_ALPHAS) > 0
):
    raise RuntimeError(
        "RDP order grid must be strictly increasing."
    )


# --------------------------------------------------------------------------------------------------
# 4. Construct configured-schedule privacy-accounting configuration
# --------------------------------------------------------------------------------------------------

PRIVACY_ACCOUNTING_CONFIGURATION = {

    "configuration_version":
        "1.0",

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "accounting_type":
        "configured_schedule",

    "mechanism":
        "DP-SGD",

    "protected_component":
        "SPP-GAN discriminator",

    "accountant":
        "RDP",

    "sampling":
        "Poisson",

    "clipping":
        "flat L2",

    "noise":
        "Gaussian",

    "target_epsilon":
        float(TARGET_EPSILON),

    "delta_rule":
        DELTA_RULE,

    "max_grad_norm":
        float(MAX_GRAD_NORM),

    "batch_size":
        int(DP_BATCH_SIZE),

    "epochs":
        int(DP_EPOCHS),

    "rdp_order_min":
        float(RDP_ALPHAS.min()),

    "rdp_order_max":
        float(RDP_ALPHAS.max()),

    "rdp_order_count":
        int(len(RDP_ALPHAS)),

    "datasets":
        list(DATASET_IDS),

    "training_performed":
        False,

    "achieved_epsilon_status":
        "DEFERRED_TO_NOTEBOOK_12",

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,

    "source_notebook_10":
        str(NB10_ROOT),

    "downstream_notebook_12":
        "SPP-GAN DP Training",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 5. Persist configured-schedule accounting configuration
# --------------------------------------------------------------------------------------------------

PRIVACY_ACCOUNTING_CONFIGURATION_PATH = (
    DIRS["configuration"]
    /
    "sppgan_privacy_accounting_configuration.json"
)

with open(
    PRIVACY_ACCOUNTING_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        PRIVACY_ACCOUNTING_CONFIGURATION,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 6. Save RDP order grid
# --------------------------------------------------------------------------------------------------

RDP_ORDER_PATH = (
    DIRS["configuration"]
    /
    "rdp_accounting_orders.csv"
)

pd.DataFrame(
    {
        "rdp_order": RDP_ALPHAS
    }
).to_csv(
    RDP_ORDER_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 7. Reload configured accounting configuration
# --------------------------------------------------------------------------------------------------

with open(
    PRIVACY_ACCOUNTING_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:

    reloaded_configuration = json.load(f)


# --------------------------------------------------------------------------------------------------
# 8. Validate persisted accounting configuration schema
# --------------------------------------------------------------------------------------------------

REQUIRED_KEYS = {
    "configuration_version",
    "notebook",
    "name",
    "framework",
    "accounting_type",
    "mechanism",
    "protected_component",
    "accountant",
    "sampling",
    "clipping",
    "noise",
    "target_epsilon",
    "delta_rule",
    "max_grad_norm",
    "batch_size",
    "epochs",
    "rdp_order_min",
    "rdp_order_max",
    "rdp_order_count",
    "datasets",
    "training_performed",
    "achieved_epsilon_status",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
    "source_notebook_10",
    "downstream_notebook_12",
    "created_utc",
}

missing_keys = (
    REQUIRED_KEYS
    -
    set(reloaded_configuration.keys())
)

if missing_keys:
    raise RuntimeError(
        "Persisted accounting configuration is incomplete:\n"
        f"{sorted(missing_keys)}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Validate persisted accounting values
# --------------------------------------------------------------------------------------------------

if (
    reloaded_configuration["accounting_type"]
    !=
    "configured_schedule"
):
    raise RuntimeError(
        "Persisted accounting type is incorrect."
    )

if (
    reloaded_configuration["accountant"]
    !=
    "RDP"
):
    raise RuntimeError(
        "Persisted accountant is incorrect."
    )

if (
    reloaded_configuration["sampling"]
    !=
    "Poisson"
):
    raise RuntimeError(
        "Persisted sampling mechanism is incorrect."
    )

if (
    reloaded_configuration["clipping"]
    !=
    "flat L2"
):
    raise RuntimeError(
        "Persisted clipping mechanism is incorrect."
    )

if (
    reloaded_configuration["mechanism"]
    !=
    "DP-SGD"
):
    raise RuntimeError(
        "Persisted privacy mechanism is incorrect."
    )

if (
    reloaded_configuration["training_performed"]
    is not False
):
    raise RuntimeError(
        "Notebook 11 must not mark training as performed."
    )

if (
    reloaded_configuration["achieved_epsilon_status"]
    !=
    "DEFERRED_TO_NOTEBOOK_12"
):
    raise RuntimeError(
        "Achieved epsilon must remain deferred to Notebook 12."
    )

if (
    reloaded_configuration["generator_private"]
    is not False
):
    raise RuntimeError(
        "Generator privacy boundary is incorrect."
    )

if (
    reloaded_configuration["statistical_guidance_private"]
    is not False
):
    raise RuntimeError(
        "Statistical guidance privacy boundary is incorrect."
    )

if (
    reloaded_configuration["preprocessing_private"]
    is not False
):
    raise RuntimeError(
        "Preprocessing privacy boundary is incorrect."
    )

if (
    reloaded_configuration["end_to_end_privacy_claim"]
    is not False
):
    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )


# --------------------------------------------------------------------------------------------------
# 10. Validate persisted RDP order artifact
# --------------------------------------------------------------------------------------------------

if not RDP_ORDER_PATH.exists():
    raise RuntimeError(
        "RDP order grid was not persisted."
    )

RELOADED_RDP_ORDERS = pd.read_csv(
    RDP_ORDER_PATH
)

if "rdp_order" not in RELOADED_RDP_ORDERS.columns:
    raise RuntimeError(
        "Persisted RDP order artifact has an invalid schema."
    )

if len(RELOADED_RDP_ORDERS) != len(RDP_ALPHAS):
    raise RuntimeError(
        "Persisted RDP order count does not match the configured grid."
    )

if not np.allclose(
    RELOADED_RDP_ORDERS["rdp_order"].to_numpy(dtype=float),
    RDP_ALPHAS,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Persisted RDP order grid does not match the configured grid."
    )


# ==================================================================================================
# 11. PERSIST AUTHORITATIVE CALIBRATED PRIVACY ARTIFACTS
# ==================================================================================================

print("\n" + "-" * 100)
print("AUTHORITATIVE CALIBRATED PRIVACY ARTIFACTS")
print("-" * 100)


# --------------------------------------------------------------------------------------------------
# 11.1 Validate Section 7 calibration objects
# --------------------------------------------------------------------------------------------------

CALIBRATION_REQUIRED_OBJECTS = [
    "CALIBRATED_NOISE_MULTIPLIERS",
    "CALIBRATED_EPSILON",
    "CALIBRATED_ALPHA",
]

missing_calibration_objects = [
    name
    for name in CALIBRATION_REQUIRED_OBJECTS
    if name not in globals()
]

if missing_calibration_objects:
    raise RuntimeError(
        "Section 7 calibration objects are unavailable:\n"
        f"{missing_calibration_objects}\n\n"
        "Section 7 must be executed successfully before Section 15."
    )


# --------------------------------------------------------------------------------------------------
# 11.2 Validate dataset coverage
# --------------------------------------------------------------------------------------------------

canonical_datasets = list(DATASET_IDS)

for dataset in canonical_datasets:

    if dataset not in CALIBRATED_NOISE_MULTIPLIERS:
        raise RuntimeError(
            f"Missing calibrated noise multiplier for dataset: {dataset}"
        )

    if dataset not in CALIBRATED_EPSILON:
        raise RuntimeError(
            f"Missing calibrated epsilon for dataset: {dataset}"
        )

    if dataset not in CALIBRATED_ALPHA:
        raise RuntimeError(
            f"Missing calibrated RDP order for dataset: {dataset}"
        )


# --------------------------------------------------------------------------------------------------
# 11.3 Locate Section 7 calibration records if available
# --------------------------------------------------------------------------------------------------

CALIBRATION_RECORDS_OBJECT = (
    globals().get("PRIVACY_CALIBRATION_RECORDS", None)
)


# --------------------------------------------------------------------------------------------------
# 11.4 Build canonical calibration validation records
# --------------------------------------------------------------------------------------------------

CALIBRATION_RECORDS = []

for dataset in canonical_datasets:

    calibrated_sigma = float(
        CALIBRATED_NOISE_MULTIPLIERS[dataset]
    )

    calibrated_epsilon = float(
        CALIBRATED_EPSILON[dataset]
    )

    calibrated_alpha = float(
        CALIBRATED_ALPHA[dataset]
    )

    # ------------------------------------------------------------------
    # Find matching Section 7 record when available.
    # ------------------------------------------------------------------

    source_record = None

    if CALIBRATION_RECORDS_OBJECT is not None:

        if isinstance(
            CALIBRATION_RECORDS_OBJECT,
            list,
        ):

            for record in CALIBRATION_RECORDS_OBJECT:

                if not isinstance(record, dict):
                    continue

                if str(
                    record.get("dataset", "")
                ) == dataset:

                    source_record = record
                    break

    # ------------------------------------------------------------------
    # Recover authoritative accounting values from Section 9 when
    # available. These remain configured-schedule values.
    # ------------------------------------------------------------------

    configured_row = None

    if (
        "CONFIGURED_ACCOUNTING_DF" in globals()
        and
        isinstance(
            CONFIGURED_ACCOUNTING_DF,
            pd.DataFrame,
        )
    ):

        matching_rows = CONFIGURED_ACCOUNTING_DF.loc[
            CONFIGURED_ACCOUNTING_DF["dataset"].astype(str)
            ==
            dataset
        ]

        if len(matching_rows) == 1:
            configured_row = matching_rows.iloc[0]

    # ------------------------------------------------------------------
    # Dataset-level parameters.
    # ------------------------------------------------------------------

    if configured_row is not None:

        n_train = int(
            configured_row["n_train"]
        )

        sample_rate = float(
            configured_row["sample_rate"]
        )

        epochs = int(
            configured_row["epochs"]
        )

        steps_per_epoch = int(
            configured_row["steps_per_epoch"]
        )

        total_steps = int(
            configured_row["total_steps"]
        )

        delta = float(
            configured_row["delta"]
        )

    else:

        # These values must already exist in the Section 7 record.
        if source_record is None:
            raise RuntimeError(
                f"No authoritative accounting record found for {dataset}."
            )

        required_source_fields = [
            "n_train",
            "sample_rate",
            "epochs",
            "steps_per_epoch",
            "total_steps",
            "delta",
        ]

        missing_source_fields = [
            field
            for field in required_source_fields
            if field not in source_record
        ]

        if missing_source_fields:
            raise RuntimeError(
                f"Calibration record for {dataset} is missing fields:\n"
                f"{missing_source_fields}"
            )

        n_train = int(
            source_record["n_train"]
        )

        sample_rate = float(
            source_record["sample_rate"]
        )

        epochs = int(
            source_record["epochs"]
        )

        steps_per_epoch = int(
            source_record["steps_per_epoch"]
        )

        total_steps = int(
            source_record["total_steps"]
        )

        delta = float(
            source_record["delta"]
        )

    record = {

        "dataset":
            dataset,

        "n_train":
            n_train,

        "batch_size":
            int(DP_BATCH_SIZE),

        "sample_rate":
            sample_rate,

        "epochs":
            epochs,

        "steps_per_epoch":
            steps_per_epoch,

        "total_steps":
            total_steps,

        "target_epsilon":
            float(TARGET_EPSILON),

        "calibrated_epsilon":
            calibrated_epsilon,

        "optimal_rdp_order":
            calibrated_alpha,

        "delta":
            delta,

        "calibrated_noise_multiplier":
            calibrated_sigma,

        "max_grad_norm":
            float(MAX_GRAD_NORM),

        "accountant":
            "RDP",

        "sampling":
            "Poisson",

        "clipping":
            "flat L2",

        "loss_reduction":
            "mean",

        "mechanism":
            "DP-SGD",

        "protected_component":
            "SPP-GAN discriminator / critic",

        "training_performed":
            False,

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,

        "calibration_source":
            "Notebook 11 Section 7",

        "calibration_authority":
            "Notebook 11",

        "status":
            "PASS",
    }

    CALIBRATION_RECORDS.append(record)


CALIBRATED_NOISE_VALIDATION_DF = pd.DataFrame(
    CALIBRATION_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 11.5 Validate calibration artifact schema
# --------------------------------------------------------------------------------------------------

REQUIRED_CALIBRATION_COLUMNS = [
    "dataset",
    "n_train",
    "batch_size",
    "sample_rate",
    "epochs",
    "steps_per_epoch",
    "total_steps",
    "target_epsilon",
    "calibrated_epsilon",
    "optimal_rdp_order",
    "delta",
    "calibrated_noise_multiplier",
    "max_grad_norm",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "mechanism",
    "protected_component",
    "training_performed",
    "achieved_epsilon_status",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
    "calibration_source",
    "calibration_authority",
    "status",
]

missing_calibration_columns = [
    column
    for column in REQUIRED_CALIBRATION_COLUMNS
    if column not in CALIBRATED_NOISE_VALIDATION_DF.columns
]

if missing_calibration_columns:
    raise RuntimeError(
        "Calibrated privacy artifact is missing required columns:\n"
        f"{missing_calibration_columns}"
    )


# --------------------------------------------------------------------------------------------------
# 11.6 Validate calibration dataset coverage
# --------------------------------------------------------------------------------------------------

if len(CALIBRATED_NOISE_VALIDATION_DF) != len(canonical_datasets):
    raise RuntimeError(
        "Calibrated privacy artifact must contain exactly one row per dataset."
    )

if set(
    CALIBRATED_NOISE_VALIDATION_DF["dataset"].astype(str)
) != set(canonical_datasets):
    raise RuntimeError(
        "Calibrated privacy artifact dataset coverage does not match DATASET_IDS."
    )

if (
    CALIBRATED_NOISE_VALIDATION_DF["dataset"]
    .duplicated()
    .any()
):
    raise RuntimeError(
        "Duplicate dataset rows found in calibrated privacy artifact."
    )


# --------------------------------------------------------------------------------------------------
# 11.7 Validate calibrated numerical values
# --------------------------------------------------------------------------------------------------

NUMERIC_CALIBRATION_COLUMNS = [
    "n_train",
    "batch_size",
    "sample_rate",
    "epochs",
    "steps_per_epoch",
    "total_steps",
    "target_epsilon",
    "calibrated_epsilon",
    "optimal_rdp_order",
    "delta",
    "calibrated_noise_multiplier",
    "max_grad_norm",
]

for column in NUMERIC_CALIBRATION_COLUMNS:

    values = pd.to_numeric(
        CALIBRATED_NOISE_VALIDATION_DF[column],
        errors="coerce",
    )

    if values.isna().any():
        raise RuntimeError(
            f"Non-numeric or missing values found in calibration column: {column}"
        )

    if not np.all(
        np.isfinite(values.to_numpy(dtype=float))
    ):
        raise RuntimeError(
            f"Non-finite values found in calibration column: {column}"
        )


# --------------------------------------------------------------------------------------------------
# 11.8 Validate calibrated privacy budget
# --------------------------------------------------------------------------------------------------

epsilon_tolerance = 1e-6

calibrated_epsilons = pd.to_numeric(
    CALIBRATED_NOISE_VALIDATION_DF[
        "calibrated_epsilon"
    ],
    errors="coerce",
).to_numpy(dtype=float)

target_epsilons = pd.to_numeric(
    CALIBRATED_NOISE_VALIDATION_DF[
        "target_epsilon"
    ],
    errors="coerce",
).to_numpy(dtype=float)


# -----------------------------------------------------------------------------------------------
# 11.8.1 Numerical validity
# -----------------------------------------------------------------------------------------------

if not np.all(
    np.isfinite(calibrated_epsilons)
):
    raise RuntimeError(
        "Calibrated epsilon contains non-finite values."
    )

if not np.all(
    np.isfinite(target_epsilons)
):
    raise RuntimeError(
        "Target epsilon contains non-finite values."
    )


# -----------------------------------------------------------------------------------------------
# 11.8.2 Privacy-budget compliance
#
# Calibration is a budget-constrained procedure.
# The calibrated epsilon must not exceed the target budget beyond numerical tolerance.
# Exact equality is NOT required.
# -----------------------------------------------------------------------------------------------

epsilon_excess = (
    calibrated_epsilons
    -
    target_epsilons
)

if np.any(
    epsilon_excess > epsilon_tolerance
):

    offending_rows = CALIBRATED_NOISE_VALIDATION_DF.loc[
        epsilon_excess > epsilon_tolerance,
        [
            "dataset",
            "target_epsilon",
            "calibrated_epsilon",
            "calibrated_noise_multiplier",
            "optimal_rdp_order",
        ],
    ].copy()

    raise RuntimeError(
        "Calibrated privacy budget is exceeded beyond numerical tolerance.\n"
        f"Tolerance: {epsilon_tolerance}\n"
        f"Offending rows:\n{offending_rows.to_string(index=False)}"
    )


# -----------------------------------------------------------------------------------------------
# 11.8.3 Record budget compliance status
# -----------------------------------------------------------------------------------------------

CALIBRATED_NOISE_VALIDATION_DF[
    "privacy_budget_status"
] = np.where(
    epsilon_excess <= epsilon_tolerance,
    "WITHIN_TARGET",
    "EXCEEDS_TARGET",
)


# -----------------------------------------------------------------------------------------------
# 11.8.4 Validation summary
# -----------------------------------------------------------------------------------------------

print(
    "✓ Calibrated epsilon budget validation: PASS"
)

for _, row in CALIBRATED_NOISE_VALIDATION_DF.iterrows():

    print(
        f"  {row['dataset']:<20} | "
        f"target ε={float(row['target_epsilon']):.10f} | "
        f"calibrated ε={float(row['calibrated_epsilon']):.10f} | "
        f"σ={float(row['calibrated_noise_multiplier']):.6f} | "
        f"{row['privacy_budget_status']}"
    )

# --------------------------------------------------------------------------------------------------
# 11.9 Validate DP accounting contract
# --------------------------------------------------------------------------------------------------

if not np.all(
    CALIBRATED_NOISE_VALIDATION_DF["accountant"].astype(str)
    ==
    "RDP"
):
    raise RuntimeError(
        "Calibrated artifact accountant must be RDP."
    )

if not np.all(
    CALIBRATED_NOISE_VALIDATION_DF["sampling"].astype(str)
    ==
    "Poisson"
):
    raise RuntimeError(
        "Calibrated artifact sampling must be Poisson."
    )

if not np.all(
    CALIBRATED_NOISE_VALIDATION_DF["clipping"].astype(str)
    ==
    "flat L2"
):
    raise RuntimeError(
        "Calibrated artifact clipping must be flat L2."
    )

if not np.all(
    CALIBRATED_NOISE_VALIDATION_DF["loss_reduction"].astype(str)
    ==
    "mean"
):
    raise RuntimeError(
        "Calibrated artifact loss reduction must be mean."
    )

if not np.all(
    CALIBRATED_NOISE_VALIDATION_DF["mechanism"].astype(str)
    ==
    "DP-SGD"
):
    raise RuntimeError(
        "Calibrated artifact mechanism must be DP-SGD."
    )


# --------------------------------------------------------------------------------------------------
# 11.10 Validate privacy boundary
# --------------------------------------------------------------------------------------------------

for column in [
    "training_performed",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
]:

    if not np.all(
        CALIBRATED_NOISE_VALIDATION_DF[column].astype(bool)
        ==
        False
    ):
        raise RuntimeError(
            f"Privacy boundary violation in calibrated artifact: {column}"
        )

if not np.all(
    CALIBRATED_NOISE_VALIDATION_DF[
        "achieved_epsilon_status"
    ].astype(str)
    ==
    "DEFERRED_TO_NOTEBOOK_12"
):
    raise RuntimeError(
        "Achieved epsilon must remain deferred to Notebook 12."
    )


# --------------------------------------------------------------------------------------------------
# 11.11 Persist calibrated noise validation CSV
# --------------------------------------------------------------------------------------------------

CALIBRATED_NOISE_VALIDATION_PATH = (
    DIRS["validation"]
    /
    "calibrated_noise_validation.csv"
)

CALIBRATED_NOISE_VALIDATION_DF.to_csv(
    CALIBRATED_NOISE_VALIDATION_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 11.12 Build calibrated privacy configuration
# --------------------------------------------------------------------------------------------------

CALIBRATED_PRIVACY_CONFIGURATION = {

    "configuration_version":
        "1.0",

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "calibration_authority":
        "Notebook 11",

    "calibration_source":
        "Notebook 11 Section 7",

    "accounting_type":
        "calibrated_schedule",

    "mechanism":
        "DP-SGD",

    "protected_component":
        "SPP-GAN discriminator / critic",

    "accountant":
        "RDP",

    "sampling":
        "Poisson",

    "clipping":
        "flat L2",

    "noise":
        "Gaussian",

    "loss_reduction":
        "mean",

    "target_epsilon":
        float(TARGET_EPSILON),

    "delta_rule":
        DELTA_RULE,

    "max_grad_norm":
        float(MAX_GRAD_NORM),

    "batch_size":
        int(DP_BATCH_SIZE),

    "epochs":
        int(DP_EPOCHS),

    "datasets":
        list(canonical_datasets),

    "calibrated_noise_multipliers": {
        str(row["dataset"]):
        float(row["calibrated_noise_multiplier"])
        for _, row
        in CALIBRATED_NOISE_VALIDATION_DF.iterrows()
    },

    "calibrated_epsilon": {
        str(row["dataset"]):
        float(row["calibrated_epsilon"])
        for _, row
        in CALIBRATED_NOISE_VALIDATION_DF.iterrows()
    },

    "optimal_rdp_order": {
        str(row["dataset"]):
        float(row["optimal_rdp_order"])
        for _, row
        in CALIBRATED_NOISE_VALIDATION_DF.iterrows()
    },

    "training_performed":
        False,

    "achieved_epsilon_status":
        "DEFERRED_TO_NOTEBOOK_12",

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,

    "downstream_notebook":
        "Notebook 12 — SPP-GAN DP Training",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 11.13 Persist calibrated privacy configuration
# --------------------------------------------------------------------------------------------------

CALIBRATED_PRIVACY_CONFIGURATION_PATH = (
    DIRS["configuration"]
    /
    "sppgan_calibrated_privacy_configuration.json"
)

with open(
    CALIBRATED_PRIVACY_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        CALIBRATED_PRIVACY_CONFIGURATION,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 11.14 Calculate configuration hash
# --------------------------------------------------------------------------------------------------

with open(
    CALIBRATED_PRIVACY_CONFIGURATION_PATH,
    "rb",
) as f:

    CALIBRATED_PRIVACY_CONFIGURATION_SHA256 = (
        hashlib.sha256(
            f.read()
        ).hexdigest()
    )


# --------------------------------------------------------------------------------------------------
# 11.15 Reload calibrated artifacts
# --------------------------------------------------------------------------------------------------

if not CALIBRATED_NOISE_VALIDATION_PATH.exists():
    raise RuntimeError(
        "Calibrated noise validation artifact was not persisted."
    )

if not CALIBRATED_PRIVACY_CONFIGURATION_PATH.exists():
    raise RuntimeError(
        "Calibrated privacy configuration was not persisted."
    )

RELOADED_CALIBRATED_NOISE_DF = pd.read_csv(
    CALIBRATED_NOISE_VALIDATION_PATH
)

with open(
    CALIBRATED_PRIVACY_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:

    RELOADED_CALIBRATED_CONFIGURATION = json.load(f)


# --------------------------------------------------------------------------------------------------
# 11.16 Validate reloaded calibrated artifacts
# --------------------------------------------------------------------------------------------------

if len(
    RELOADED_CALIBRATED_NOISE_DF
) != len(canonical_datasets):
    raise RuntimeError(
        "Reloaded calibrated noise artifact has an invalid row count."
    )

if set(
    RELOADED_CALIBRATED_NOISE_DF["dataset"].astype(str)
) != set(canonical_datasets):
    raise RuntimeError(
        "Reloaded calibrated noise artifact has incorrect dataset coverage."
    )

if (
    RELOADED_CALIBRATED_CONFIGURATION[
        "calibration_authority"
    ]
    !=
    "Notebook 11"
):
    raise RuntimeError(
        "Calibrated privacy authority is not Notebook 11."
    )

if (
    RELOADED_CALIBRATED_CONFIGURATION[
        "calibration_source"
    ]
    !=
    "Notebook 11 Section 7"
):
    raise RuntimeError(
        "Calibrated privacy source is incorrect."
    )

if (
    RELOADED_CALIBRATED_CONFIGURATION[
        "accounting_type"
    ]
    !=
    "calibrated_schedule"
):
    raise RuntimeError(
        "Calibrated privacy accounting type is incorrect."
    )

if (
    RELOADED_CALIBRATED_CONFIGURATION[
        "training_performed"
    ]
    is not False
):
    raise RuntimeError(
        "Calibrated configuration incorrectly marks training as performed."
    )

if (
    RELOADED_CALIBRATED_CONFIGURATION[
        "achieved_epsilon_status"
    ]
    !=
    "DEFERRED_TO_NOTEBOOK_12"
):
    raise RuntimeError(
        "Calibrated configuration incorrectly claims achieved epsilon."
    )

if (
    RELOADED_CALIBRATED_CONFIGURATION[
        "end_to_end_privacy_claim"
    ]
    is not False
):
    raise RuntimeError(
        "Calibrated configuration contains an invalid end-to-end privacy claim."
    )


# --------------------------------------------------------------------------------------------------
# 12. Expose canonical Section 15 objects for Notebook 12
# --------------------------------------------------------------------------------------------------

CALIBRATED_NOISE_MULTIPLIERS = {
    str(row["dataset"]):
    float(row["calibrated_noise_multiplier"])
    for _, row
    in RELOADED_CALIBRATED_NOISE_DF.iterrows()
}

CALIBRATED_EPSILON = {
    str(row["dataset"]):
    float(row["calibrated_epsilon"])
    for _, row
    in RELOADED_CALIBRATED_NOISE_DF.iterrows()
}

CALIBRATED_ALPHA = {
    str(row["dataset"]):
    float(row["optimal_rdp_order"])
    for _, row
    in RELOADED_CALIBRATED_NOISE_DF.iterrows()
}


# --------------------------------------------------------------------------------------------------
# 13. Final persistence checks
# --------------------------------------------------------------------------------------------------

if not PRIVACY_ACCOUNTING_CONFIGURATION_PATH.exists():
    raise RuntimeError(
        "Privacy accounting configuration was not persisted."
    )

if not RDP_ORDER_PATH.exists():
    raise RuntimeError(
        "RDP order artifact was not persisted."
    )

if not CALIBRATED_NOISE_VALIDATION_PATH.exists():
    raise RuntimeError(
        "Calibrated noise validation artifact was not persisted."
    )

if not CALIBRATED_PRIVACY_CONFIGURATION_PATH.exists():
    raise RuntimeError(
        "Calibrated privacy configuration was not persisted."
    )


# --------------------------------------------------------------------------------------------------
# 14. Final output
# --------------------------------------------------------------------------------------------------

print(
    f"\n✓ Accounting configuration saved:\n"
    f"  {PRIVACY_ACCOUNTING_CONFIGURATION_PATH}"
)

print(
    f"✓ RDP order grid saved:\n"
    f"  {RDP_ORDER_PATH}"
)

print(
    f"✓ Calibrated noise validation saved:\n"
    f"  {CALIBRATED_NOISE_VALIDATION_PATH}"
)

print(
    f"✓ Calibrated privacy configuration saved:\n"
    f"  {CALIBRATED_PRIVACY_CONFIGURATION_PATH}"
)

print(
    "\n✓ Configuration reload validation: PASS"
)

print(
    "✓ Persisted RDP order validation: PASS"
)

print(
    "✓ Calibrated privacy artifact validation: PASS"
)

print(
    "✓ Calibration authority: NOTEBOOK 11 SECTION 7"
)

print(
    "✓ Training status: NOT PERFORMED"
)

print(
    "✓ Achieved epsilon: DEFERRED TO NOTEBOOK 12"
)

print(
    "✓ End-to-end privacy claim: FALSE"
)

print(
    "\nCALIBRATED NOISE MULTIPLIERS"
)

for dataset in canonical_datasets:

    print(
        f"  {dataset:<20} : "
        f"{CALIBRATED_NOISE_MULTIPLIERS[dataset]:.6f}"
    )

print(
    "\nSECTION 15 STATUS: PASS"
)


15. SAVE PRIVACY ACCOUNTING

----------------------------------------------------------------------------------------------------
AUTHORITATIVE CALIBRATED PRIVACY ARTIFACTS
----------------------------------------------------------------------------------------------------
✓ Calibrated epsilon budget validation: PASS
  adult_income         | target ε=5.0000000000 | calibrated ε=4.9999999431 | σ=1.217683 | WITHIN_TARGET
  bank_marketing       | target ε=5.0000000000 | calibrated ε=4.9999999279 | σ=1.252411 | WITHIN_TARGET
  diabetes_130us       | target ε=5.0000000000 | calibrated ε=4.9999999301 | σ=0.953647 | WITHIN_TARGET

✓ Accounting configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/configuration/sppgan_privacy_accounting_configuration.json
✓ RDP order grid saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/configuration/rdp_accounting_orders.csv
✓ Calibrated noise validation saved:
  /content/drive/MyDrive/SPP_

In [21]:
# ==================================================================================================
# 16. SAVE PRIVACY AUDIT
# ==================================================================================================

print("\n" + "=" * 100)
print("16. SAVE PRIVACY AUDIT")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate Section 12 dependency
# --------------------------------------------------------------------------------------------------

if "PER_EXPERIMENT_DF" not in globals():
    raise RuntimeError(
        "PER_EXPERIMENT_DF from Section 12 is unavailable."
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate required Section 15 calibration objects
# --------------------------------------------------------------------------------------------------

REQUIRED_CALIBRATION_OBJECTS = [
    "CALIBRATED_NOISE_VALIDATION_PATH",
    "CALIBRATED_PRIVACY_CONFIGURATION_PATH",
    "CALIBRATED_NOISE_VALIDATION_DF",
]

missing_calibration_objects = [
    name
    for name in REQUIRED_CALIBRATION_OBJECTS
    if name not in globals()
]

if missing_calibration_objects:
    raise RuntimeError(
        "Required Section 15 calibration objects are unavailable:\n"
        f"{missing_calibration_objects}\n\n"
        "Section 15 must be executed successfully before Section 16."
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate calibration artifact paths
# --------------------------------------------------------------------------------------------------

if not CALIBRATED_NOISE_VALIDATION_PATH.exists():
    raise RuntimeError(
        "Notebook 11 calibrated noise validation artifact does not exist:\n"
        f"{CALIBRATED_NOISE_VALIDATION_PATH}"
    )

if not CALIBRATED_PRIVACY_CONFIGURATION_PATH.exists():
    raise RuntimeError(
        "Notebook 11 calibrated privacy configuration does not exist:\n"
        f"{CALIBRATED_PRIVACY_CONFIGURATION_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Validate Section 12 audit input schema
# --------------------------------------------------------------------------------------------------

REQUIRED_COLUMNS = [
    "experiment_id",
    "dataset",
    "mechanism",
    "protected_component",
    "accountant",
    "sampling",
    "clipping",
    "target_epsilon",
    "configured_schedule_epsilon",
    "delta",
    "noise_multiplier",
    "sample_rate",
    "total_steps",
    "optimal_rdp_order",
    "training_performed",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in PER_EXPERIMENT_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "Per-experiment privacy records are missing required columns:\n"
        f"{missing_columns}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Validate privacy-accounting provenance
# --------------------------------------------------------------------------------------------------

if not PER_EXPERIMENT_DF[
    "accountant"
].eq(
    "RDP"
).all():

    raise RuntimeError(
        "Privacy audit requires RDP accounting records."
    )


if not PER_EXPERIMENT_DF[
    "sampling"
].eq(
    "Poisson"
).all():

    raise RuntimeError(
        "Privacy audit requires Poisson sampling records."
    )


if not PER_EXPERIMENT_DF[
    "clipping"
].eq(
    "flat L2"
).all():

    raise RuntimeError(
        "Privacy audit requires flat L2 clipping records."
    )


if not PER_EXPERIMENT_DF[
    "training_performed"
].eq(
    False
).all():

    raise RuntimeError(
        "Notebook 11 audit records must indicate that training "
        "has not yet been performed."
    )


if not PER_EXPERIMENT_DF[
    "end_to_end_privacy_claim"
].eq(
    False
).all():

    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )


# --------------------------------------------------------------------------------------------------
# 6. Validate canonical dataset coverage
# --------------------------------------------------------------------------------------------------

if len(PER_EXPERIMENT_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Per-experiment privacy records must contain exactly "
        "one record per canonical dataset."
    )


if set(
    PER_EXPERIMENT_DF["dataset"].astype(str)
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Per-experiment privacy dataset coverage does not match "
        "the canonical dataset registry."
    )


if PER_EXPERIMENT_DF[
    "dataset"
].duplicated().any():

    raise RuntimeError(
        "Duplicate dataset records detected in PER_EXPERIMENT_DF."
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate Section 15 calibrated artifact
# --------------------------------------------------------------------------------------------------

CALIBRATION_REQUIRED_COLUMNS = [
    "dataset",
    "target_epsilon",
    "calibrated_epsilon",
    "optimal_rdp_order",
    "calibrated_noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "mechanism",
    "protected_component",
    "training_performed",
    "achieved_epsilon_status",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
    "calibration_source",
    "calibration_authority",
    "status",
]

missing_calibration_columns = [
    column
    for column in CALIBRATION_REQUIRED_COLUMNS
    if column not in CALIBRATED_NOISE_VALIDATION_DF.columns
]

if missing_calibration_columns:
    raise RuntimeError(
        "Section 15 calibrated privacy artifact is missing required columns:\n"
        f"{missing_calibration_columns}"
    )


if len(
    CALIBRATED_NOISE_VALIDATION_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Calibrated privacy artifact must contain exactly "
        "one record per canonical dataset."
    )


if set(
    CALIBRATED_NOISE_VALIDATION_DF["dataset"].astype(str)
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Calibrated privacy artifact dataset coverage does not "
        "match the canonical dataset registry."
    )


if CALIBRATED_NOISE_VALIDATION_DF[
    "dataset"
].duplicated().any():

    raise RuntimeError(
        "Duplicate dataset records detected in calibrated privacy artifact."
    )


# --------------------------------------------------------------------------------------------------
# 8. Validate calibration authority
# --------------------------------------------------------------------------------------------------

if not CALIBRATED_NOISE_VALIDATION_DF[
    "calibration_source"
].eq(
    "Notebook 11 Section 7"
).all():

    raise RuntimeError(
        "Calibration source is not Notebook 11 Section 7."
    )


if not CALIBRATED_NOISE_VALIDATION_DF[
    "calibration_authority"
].eq(
    "Notebook 11"
).all():

    raise RuntimeError(
        "Calibration authority is not Notebook 11."
    )


# --------------------------------------------------------------------------------------------------
# 9. Validate calibrated privacy boundary
# --------------------------------------------------------------------------------------------------

for column in [
    "training_performed",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
]:

    if not CALIBRATED_NOISE_VALIDATION_DF[
        column
    ].eq(
        False
    ).all():

        raise RuntimeError(
            f"Calibrated privacy artifact violates privacy boundary: {column}"
        )


if not CALIBRATED_NOISE_VALIDATION_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Calibrated privacy artifact must defer achieved epsilon to Notebook 12."
    )


# --------------------------------------------------------------------------------------------------
# 10. Generate CSV privacy-audit records
# --------------------------------------------------------------------------------------------------

PRIVACY_AUDIT_ROWS = []

for row in PER_EXPERIMENT_DF.itertuples(
    index=False
):

    epsilon_difference = (
        float(row.configured_schedule_epsilon)
        -
        float(row.target_epsilon)
    )

    PRIVACY_AUDIT_ROWS.append({

        "experiment_id":
            row.experiment_id,

        "dataset":
            row.dataset,

        "mechanism":
            row.mechanism,

        "protected_component":
            row.protected_component,

        "accountant":
            row.accountant,

        "sampling":
            row.sampling,

        "clipping":
            row.clipping,

        "target_epsilon":
            float(row.target_epsilon),

        "configured_schedule_epsilon":
            float(row.configured_schedule_epsilon),

        "epsilon_difference":
            epsilon_difference,

        "absolute_epsilon_difference":
            abs(
                epsilon_difference
            ),

        "delta":
            float(row.delta),

        "noise_multiplier":
            float(row.noise_multiplier),

        "sample_rate":
            float(row.sample_rate),

        "total_steps":
            int(row.total_steps),

        "optimal_rdp_order":
            float(row.optimal_rdp_order),

        "training_performed":
            bool(row.training_performed),

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "generator_private":
            bool(row.generator_private),

        "statistical_guidance_private":
            bool(row.statistical_guidance_private),

        "preprocessing_private":
            bool(row.preprocessing_private),

        "end_to_end_privacy_claim":
            bool(row.end_to_end_privacy_claim),

        # ------------------------------------------------------------------------------------------
        # Calibration provenance
        # ------------------------------------------------------------------------------------------

        "calibration_authority":
            "Notebook 11",

        "calibration_source":
            "Notebook 11 Section 7",

        "calibrated_noise_artifact":
            str(
                CALIBRATED_NOISE_VALIDATION_PATH
            ),

        "calibrated_privacy_configuration":
            str(
                CALIBRATED_PRIVACY_CONFIGURATION_PATH
            ),

        "audit_scope":
            "configured_schedule",

        "audit_status":
            "PASS",
    })


PRIVACY_AUDIT_DF = pd.DataFrame(
    PRIVACY_AUDIT_ROWS
)


# --------------------------------------------------------------------------------------------------
# 11. Validate generated audit records
# --------------------------------------------------------------------------------------------------

if len(
    PRIVACY_AUDIT_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Privacy audit must contain exactly one record per canonical dataset."
    )


if set(
    PRIVACY_AUDIT_DF["dataset"].astype(str)
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Privacy audit dataset coverage does not match the canonical registry."
    )


if PRIVACY_AUDIT_DF[
    "dataset"
].duplicated().any():

    raise RuntimeError(
        "Privacy audit contains duplicate dataset records."
    )


if not PRIVACY_AUDIT_DF[
    "training_performed"
].eq(
    False
).all():

    raise RuntimeError(
        "Privacy audit incorrectly indicates that training was performed."
    )


if not PRIVACY_AUDIT_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Achieved epsilon status is inconsistent."
    )


if not PRIVACY_AUDIT_DF[
    "end_to_end_privacy_claim"
].eq(
    False
).all():

    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )


if not PRIVACY_AUDIT_DF[
    "calibration_authority"
].eq(
    "Notebook 11"
).all():

    raise RuntimeError(
        "Privacy audit calibration authority is incorrect."
    )


if not PRIVACY_AUDIT_DF[
    "calibration_source"
].eq(
    "Notebook 11 Section 7"
).all():

    raise RuntimeError(
        "Privacy audit calibration source is incorrect."
    )


if not PRIVACY_AUDIT_DF[
    "audit_scope"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Privacy audit scope must remain configured_schedule."
    )


# --------------------------------------------------------------------------------------------------
# 12. Persist CSV privacy audit
# --------------------------------------------------------------------------------------------------

PRIVACY_AUDIT_PATH = (
    DIRS["audit"]
    /
    "sppgan_privacy_audit.csv"
)

PRIVACY_AUDIT_DF.to_csv(
    PRIVACY_AUDIT_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 13. Generate JSON privacy audit
# --------------------------------------------------------------------------------------------------

PRIVACY_AUDIT_JSON = {

    "notebook":
        NOTEBOOK_ID,

    "framework":
        FRAMEWORK_NAME,

    "accounting_type":
        "configured_schedule",

    "audit_scope":
        "SPP-GAN discriminator DP-SGD",

    "target_epsilon":
        float(TARGET_EPSILON),

    "datasets":
        list(DATASET_IDS),

    # ----------------------------------------------------------------------------------------------
    # Calibration provenance
    # ----------------------------------------------------------------------------------------------

    "calibration_authority":
        "Notebook 11",

    "calibration_source":
        "Notebook 11 Section 7",

    "calibrated_noise_artifact":
        str(
            CALIBRATED_NOISE_VALIDATION_PATH
        ),

    "calibrated_privacy_configuration":
        str(
            CALIBRATED_PRIVACY_CONFIGURATION_PATH
        ),

    # ----------------------------------------------------------------------------------------------
    # Training / privacy boundary
    # ----------------------------------------------------------------------------------------------

    "training_performed":
        False,

    "achieved_epsilon_status":
        "DEFERRED_TO_NOTEBOOK_12",

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,

    # ----------------------------------------------------------------------------------------------
    # Audit records
    # ----------------------------------------------------------------------------------------------

    "records":
        PRIVACY_AUDIT_ROWS,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# --------------------------------------------------------------------------------------------------
# 14. Persist JSON privacy audit
# --------------------------------------------------------------------------------------------------

PRIVACY_AUDIT_JSON_PATH = (
    DIRS["audit"]
    /
    "sppgan_privacy_audit.json"
)

with open(
    PRIVACY_AUDIT_JSON_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        PRIVACY_AUDIT_JSON,
        f,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 15. Reload and validate JSON audit
# --------------------------------------------------------------------------------------------------

with open(
    PRIVACY_AUDIT_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:

    RELOADED_AUDIT_JSON = json.load(f)


# --------------------------------------------------------------------------------------------------
# 16. Validate JSON schema
# --------------------------------------------------------------------------------------------------

REQUIRED_JSON_KEYS = {
    "notebook",
    "framework",
    "accounting_type",
    "audit_scope",
    "target_epsilon",
    "datasets",

    # Calibration provenance
    "calibration_authority",
    "calibration_source",
    "calibrated_noise_artifact",
    "calibrated_privacy_configuration",

    # Privacy boundary
    "training_performed",
    "achieved_epsilon_status",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",

    # Audit records
    "records",
    "created_utc",
}

missing_json_keys = (
    REQUIRED_JSON_KEYS
    -
    set(
        RELOADED_AUDIT_JSON.keys()
    )
)

if missing_json_keys:

    raise RuntimeError(
        "Privacy audit JSON is missing required keys:\n"
        f"{sorted(missing_json_keys)}"
    )


# --------------------------------------------------------------------------------------------------
# 17. Validate persisted JSON values
# --------------------------------------------------------------------------------------------------

if (
    RELOADED_AUDIT_JSON[
        "accounting_type"
    ]
    !=
    "configured_schedule"
):

    raise RuntimeError(
        "Persisted audit accounting type is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "audit_scope"
    ]
    !=
    "SPP-GAN discriminator DP-SGD"
):

    raise RuntimeError(
        "Persisted audit scope is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "calibration_authority"
    ]
    !=
    "Notebook 11"
):

    raise RuntimeError(
        "Persisted calibration authority is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "calibration_source"
    ]
    !=
    "Notebook 11 Section 7"
):

    raise RuntimeError(
        "Persisted calibration source is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "training_performed"
    ]
    is not False
):

    raise RuntimeError(
        "Persisted audit incorrectly marks training as performed."
    )


if (
    RELOADED_AUDIT_JSON[
        "achieved_epsilon_status"
    ]
    !=
    "DEFERRED_TO_NOTEBOOK_12"
):

    raise RuntimeError(
        "Persisted achieved-epsilon status is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "generator_private"
    ]
    is not False
):

    raise RuntimeError(
        "Persisted generator privacy boundary is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "statistical_guidance_private"
    ]
    is not False
):

    raise RuntimeError(
        "Persisted statistical-guidance privacy boundary is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "preprocessing_private"
    ]
    is not False
):

    raise RuntimeError(
        "Persisted preprocessing privacy boundary is incorrect."
    )


if (
    RELOADED_AUDIT_JSON[
        "end_to_end_privacy_claim"
    ]
    is not False
):

    raise RuntimeError(
        "Persisted end-to-end privacy claim must remain False."
    )


# --------------------------------------------------------------------------------------------------
# 18. Validate persisted JSON dataset coverage
# --------------------------------------------------------------------------------------------------

if set(
    RELOADED_AUDIT_JSON[
        "datasets"
    ]
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Persisted audit JSON dataset coverage is incorrect."
    )


if len(
    RELOADED_AUDIT_JSON[
        "records"
    ]
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Persisted audit JSON record count is incorrect."
    )


# --------------------------------------------------------------------------------------------------
# 19. Final persistence validation
# --------------------------------------------------------------------------------------------------

if not PRIVACY_AUDIT_PATH.exists():

    raise RuntimeError(
        "Privacy audit CSV was not persisted."
    )


if not PRIVACY_AUDIT_JSON_PATH.exists():

    raise RuntimeError(
        "Privacy audit JSON was not persisted."
    )


# --------------------------------------------------------------------------------------------------
# 20. Display results
# --------------------------------------------------------------------------------------------------

display(
    PRIVACY_AUDIT_DF
)


print(
    f"\n✓ Privacy audit CSV saved:\n"
    f"  {PRIVACY_AUDIT_PATH}"
)

print(
    f"✓ Privacy audit JSON saved:\n"
    f"  {PRIVACY_AUDIT_JSON_PATH}"
)

print(
    f"✓ Calibration authority: NOTEBOOK 11 SECTION 7"
)

print(
    f"✓ Calibrated noise artifact validated:\n"
    f"  {CALIBRATED_NOISE_VALIDATION_PATH}"
)

print(
    f"✓ Calibrated privacy configuration validated:\n"
    f"  {CALIBRATED_PRIVACY_CONFIGURATION_PATH}"
)

print(
    "✓ Audit record validation: PASS"
)

print(
    "✓ JSON reload validation: PASS"
)

print(
    "✓ Calibration provenance validation: PASS"
)

print(
    "✓ Privacy boundary validation: PASS"
)

print(
    "✓ Training status: NOT PERFORMED"
)

print(
    "✓ Achieved epsilon: DEFERRED TO NOTEBOOK 12"
)

print(
    "✓ End-to-end privacy claim: FALSE"
)

print(
    "\nSECTION 16 STATUS: PASS"
)


16. SAVE PRIVACY AUDIT


,experiment_id,dataset,mechanism,protected_component,accountant,sampling,clipping,target_epsilon,configured_schedule_epsilon,epsilon_difference,...,generator_private,statistical_guidance_private,preprocessing_private,end_to_end_privacy_claim,calibration_authority,calibration_source,calibrated_noise_artifact,calibrated_privacy_configuration,audit_scope,audit_status
0,adult_income_DP-SGD_RDP_configured-schedule_eps5,adult_income,DP-SGD,SPP-GAN discriminator / critic,RDP,Poisson,flat L2,5.0,7.034634,2.034634,...,False,False,False,False,Notebook 11,Notebook 11 Section 7,/content/drive/MyDrive/SPP_GAN_Research/result...,/content/drive/MyDrive/SPP_GAN_Research/result...,configured_schedule,PASS
1,bank_marketing_DP-SGD_RDP_configured-schedule_...,bank_marketing,DP-SGD,SPP-GAN discriminator / critic,RDP,Poisson,flat L2,5.0,7.363785,2.363785,...,False,False,False,False,Notebook 11,Notebook 11 Section 7,/content/drive/MyDrive/SPP_GAN_Research/result...,/content/drive/MyDrive/SPP_GAN_Research/result...,configured_schedule,PASS
2,diabetes_130us_DP-SGD_RDP_configured-schedule_...,diabetes_130us,DP-SGD,SPP-GAN discriminator / critic,RDP,Poisson,flat L2,5.0,4.572695,-0.427305,...,False,False,False,False,Notebook 11,Notebook 11 Section 7,/content/drive/MyDrive/SPP_GAN_Research/result...,/content/drive/MyDrive/SPP_GAN_Research/result...,configured_schedule,PASS



✓ Privacy audit CSV saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/audit/sppgan_privacy_audit.csv
✓ Privacy audit JSON saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/audit/sppgan_privacy_audit.json
✓ Calibration authority: NOTEBOOK 11 SECTION 7
✓ Calibrated noise artifact validated:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/validation/calibrated_noise_validation.csv
✓ Calibrated privacy configuration validated:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/configuration/sppgan_calibrated_privacy_configuration.json
✓ Audit record validation: PASS
✓ JSON reload validation: PASS
✓ Calibration provenance validation: PASS
✓ Privacy boundary validation: PASS
✓ Training status: NOT PERFORMED
✓ Achieved epsilon: DEFERRED TO NOTEBOOK 12
✓ End-to-end privacy claim: FALSE

SECTION 16 STATUS: PASS


In [23]:
# ==================================================================================================
# 17. FINAL PRIVACY VERIFICATION
# ==================================================================================================

print("\n" + "=" * 100)
print("17. FINAL PRIVACY VERIFICATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Required upstream objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "NB10_ROOT",
    "NB10_CONFIGURATION_PATH",
    "NB10_METADATA_PATH",
    "ACCOUNTANT",
    "SAMPLING_MECHANISM",
    "CLIPPING_MECHANISM",
    "TARGET_EPSILON",
    "MAX_GRAD_NORM",
    "DATASET_IDS",
    "PER_EXPERIMENT_DF",
    "DELTA_DF",
    "SAMPLING_VALIDATION_DF",
    "STEP_DF",
    "NOISE_PARAMETER_DF",
    "ACCOUNTING_PATH",
    "PER_EXPERIMENT_PATH",
    "DATASET_SUMMARY_PATH",
    "MODEL_SUMMARY_PATH",
    "PRIVACY_ACCOUNTING_CONFIGURATION_PATH",
    "RDP_ORDER_PATH",
    "PRIVACY_AUDIT_PATH",
    "PRIVACY_AUDIT_JSON_PATH",
    "CALIBRATED_NOISE_VALIDATION_PATH",
    "CALIBRATED_PRIVACY_CONFIGURATION_PATH",
    "CALIBRATED_NOISE_VALIDATION_DF",
    "DIRS",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required upstream objects are unavailable:\n"
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Initialize verification registry
# --------------------------------------------------------------------------------------------------

FINAL_PRIVACY_CHECKS = []


def add_privacy_check(
    name,
    condition,
):
    FINAL_PRIVACY_CHECKS.append({
        "check": name,
        "status": (
            "PASS"
            if bool(condition)
            else "FAIL"
        ),
    })


# ==================================================================================================
# 3. SOURCE ARTIFACT VALIDATION
# ==================================================================================================

add_privacy_check(
    "Notebook 10 root exists",
    NB10_ROOT.exists(),
)

add_privacy_check(
    "Notebook 10 privacy configuration exists",
    NB10_CONFIGURATION_PATH.exists(),
)

add_privacy_check(
    "Notebook 10 privacy metadata exists",
    NB10_METADATA_PATH.exists(),
)


# ==================================================================================================
# 4. PRIVACY CONFIGURATION VALIDATION
# ==================================================================================================

add_privacy_check(
    "RDP accountant selected",
    str(ACCOUNTANT).lower() == "rdp",
)

add_privacy_check(
    "Poisson sampling selected",
    str(SAMPLING_MECHANISM).lower() == "poisson",
)

add_privacy_check(
    "Flat clipping selected",
    str(CLIPPING_MECHANISM).lower() == "flat",
)

add_privacy_check(
    "Positive target epsilon",
    float(TARGET_EPSILON) > 0,
)

add_privacy_check(
    "Positive clipping norm",
    float(MAX_GRAD_NORM) > 0,
)


# ==================================================================================================
# 5. PER-EXPERIMENT ACCOUNTING SCHEMA VALIDATION
# ==================================================================================================

REQUIRED_ACCOUNTING_COLUMNS = [
    "experiment_id",
    "dataset",
    "target_epsilon",
    "configured_schedule_epsilon",
    "delta",
    "noise_multiplier",
    "sample_rate",
    "total_steps",
    "optimal_rdp_order",
    "training_performed",
    "achieved_epsilon_status",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
]

missing_accounting_columns = [
    column
    for column in REQUIRED_ACCOUNTING_COLUMNS
    if column not in PER_EXPERIMENT_DF.columns
]

add_privacy_check(
    "Per-experiment accounting schema complete",
    len(missing_accounting_columns) == 0,
)

if missing_accounting_columns:
    display(
        pd.DataFrame({
            "missing_column": missing_accounting_columns
        })
    )

    raise RuntimeError(
        "Per-experiment accounting schema is incomplete."
    )


# ==================================================================================================
# 6. DATASET COVERAGE VALIDATION
# ==================================================================================================

add_privacy_check(
    "All canonical datasets accounted",
    set(
        PER_EXPERIMENT_DF["dataset"].astype(str)
    )
    ==
    set(DATASET_IDS),
)

add_privacy_check(
    "Exactly one accounting record per dataset",
    len(PER_EXPERIMENT_DF) == len(DATASET_IDS),
)

add_privacy_check(
    "No duplicate dataset accounting records",
    not PER_EXPERIMENT_DF["dataset"].duplicated().any(),
)


# ==================================================================================================
# 7. CONFIGURED-SCHEDULE EPSILON VALIDATION
# ==================================================================================================

configured_epsilon = pd.to_numeric(
    PER_EXPERIMENT_DF[
        "configured_schedule_epsilon"
    ],
    errors="coerce",
)

target_epsilon = pd.to_numeric(
    PER_EXPERIMENT_DF[
        "target_epsilon"
    ],
    errors="coerce",
)

epsilon_difference = (
    configured_epsilon
    -
    target_epsilon
)

add_privacy_check(
    "Configured-schedule epsilon finite",
    np.isfinite(
        configured_epsilon.to_numpy(
            dtype=float
        )
    ).all(),
)

add_privacy_check(
    "Configured-schedule epsilon non-negative",
    (
        configured_epsilon
        >=
        0
    ).all(),
)

add_privacy_check(
    "Target epsilon values positive",
    (
        target_epsilon
        >
        0
    ).all(),
)

add_privacy_check(
    "Configured epsilon differences finite",
    np.isfinite(
        epsilon_difference.to_numpy(
            dtype=float
        )
    ).all(),
)


# ==================================================================================================
# 8. RDP OPTIMAL-ORDER VALIDATION
# ==================================================================================================

optimal_orders = pd.to_numeric(
    PER_EXPERIMENT_DF[
        "optimal_rdp_order"
    ],
    errors="coerce",
)

add_privacy_check(
    "Optimal RDP orders finite",
    np.isfinite(
        optimal_orders.to_numpy(
            dtype=float
        )
    ).all(),
)

add_privacy_check(
    "Optimal RDP orders greater than one",
    (
        optimal_orders
        >
        1
    ).all(),
)


# ==================================================================================================
# 9. DELTA VALIDATION
# ==================================================================================================

add_privacy_check(
    "Delta validation passes",
    (
        "status" in DELTA_DF.columns
        and
        DELTA_DF["status"].eq("PASS").all()
    ),
)


# ==================================================================================================
# 10. SAMPLING VALIDATION
# ==================================================================================================

add_privacy_check(
    "Sampling validation passes",
    (
        "status" in SAMPLING_VALIDATION_DF.columns
        and
        SAMPLING_VALIDATION_DF["status"].eq("PASS").all()
    ),
)


# ==================================================================================================
# 11. TRAINING-STEP VALIDATION
# ==================================================================================================

add_privacy_check(
    "Training-step validation passes",
    (
        "status" in STEP_DF.columns
        and
        STEP_DF["status"].eq("PASS").all()
    ),
)


# ==================================================================================================
# 12. NOISE-PARAMETER VALIDATION
# ==================================================================================================

add_privacy_check(
    "Noise validation passes",
    (
        "status" in NOISE_PARAMETER_DF.columns
        and
        NOISE_PARAMETER_DF["status"].eq("PASS").all()
    ),
)


# ==================================================================================================
# 13. PRIVACY-BOUNDARY VALIDATION
# ==================================================================================================

add_privacy_check(
    "Generator privacy claim disabled",
    not PER_EXPERIMENT_DF[
        "generator_private"
    ].fillna(True).astype(bool).any(),
)

add_privacy_check(
    "Statistical guidance privacy claim disabled",
    not PER_EXPERIMENT_DF[
        "statistical_guidance_private"
    ].fillna(True).astype(bool).any(),
)

add_privacy_check(
    "Preprocessing privacy claim disabled",
    not PER_EXPERIMENT_DF[
        "preprocessing_private"
    ].fillna(True).astype(bool).any(),
)

add_privacy_check(
    "End-to-end privacy claim disabled",
    not PER_EXPERIMENT_DF[
        "end_to_end_privacy_claim"
    ].fillna(True).astype(bool).any(),
)

add_privacy_check(
    "Training not performed",
    not PER_EXPERIMENT_DF[
        "training_performed"
    ].fillna(True).astype(bool).any(),
)

add_privacy_check(
    "Achieved epsilon deferred to Notebook 12",
    PER_EXPERIMENT_DF[
        "achieved_epsilon_status"
    ].eq(
        "DEFERRED_TO_NOTEBOOK_12"
    ).all(),
)


# ==================================================================================================
# 14. VALIDATE SECTION 15 CALIBRATION ARTIFACTS
# ==================================================================================================

print("\n" + "-" * 100)
print("CALIBRATION ARTIFACT VALIDATION")
print("-" * 100)


# --------------------------------------------------------------------------------------------------
# 14.1 Artifact existence
# --------------------------------------------------------------------------------------------------

add_privacy_check(
    "Calibrated noise validation exists",
    CALIBRATED_NOISE_VALIDATION_PATH.exists(),
)

add_privacy_check(
    "Calibrated privacy configuration exists",
    CALIBRATED_PRIVACY_CONFIGURATION_PATH.exists(),
)


# --------------------------------------------------------------------------------------------------
# 14.2 Calibration dataframe schema
# --------------------------------------------------------------------------------------------------

REQUIRED_CALIBRATION_COLUMNS = [
    "dataset",
    "target_epsilon",
    "calibrated_epsilon",
    "optimal_rdp_order",
    "calibrated_noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "mechanism",
    "protected_component",
    "training_performed",
    "achieved_epsilon_status",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
    "calibration_source",
    "calibration_authority",
    "status",
]

missing_calibration_columns = [
    column
    for column in REQUIRED_CALIBRATION_COLUMNS
    if column not in CALIBRATED_NOISE_VALIDATION_DF.columns
]

add_privacy_check(
    "Calibrated privacy schema complete",
    len(missing_calibration_columns) == 0,
)

if missing_calibration_columns:
    display(
        pd.DataFrame({
            "missing_calibration_column":
                missing_calibration_columns
        })
    )

    raise RuntimeError(
        "Calibrated privacy artifact schema is incomplete."
    )


# --------------------------------------------------------------------------------------------------
# 14.3 Calibration dataset coverage
# --------------------------------------------------------------------------------------------------

add_privacy_check(
    "All canonical datasets calibrated",
    set(
        CALIBRATED_NOISE_VALIDATION_DF[
            "dataset"
        ].astype(str)
    )
    ==
    set(DATASET_IDS),
)

add_privacy_check(
    "Exactly one calibrated record per dataset",
    len(
        CALIBRATED_NOISE_VALIDATION_DF
    )
    ==
    len(DATASET_IDS),
)

add_privacy_check(
    "No duplicate calibrated dataset records",
    not CALIBRATED_NOISE_VALIDATION_DF[
        "dataset"
    ].duplicated().any(),
)


# --------------------------------------------------------------------------------------------------
# 14.4 Calibrated epsilon validation
# --------------------------------------------------------------------------------------------------

calibrated_epsilon = pd.to_numeric(
    CALIBRATED_NOISE_VALIDATION_DF[
        "calibrated_epsilon"
    ],
    errors="coerce",
)

calibrated_target_epsilon = pd.to_numeric(
    CALIBRATED_NOISE_VALIDATION_DF[
        "target_epsilon"
    ],
    errors="coerce",
)

calibrated_sigma = pd.to_numeric(
    CALIBRATED_NOISE_VALIDATION_DF[
        "calibrated_noise_multiplier"
    ],
    errors="coerce",
)

calibrated_alpha = pd.to_numeric(
    CALIBRATED_NOISE_VALIDATION_DF[
        "optimal_rdp_order"
    ],
    errors="coerce",
)

CALIBRATED_EPSILON_TOLERANCE = 1e-6

calibrated_epsilon_excess = (
    calibrated_epsilon
    -
    calibrated_target_epsilon
)


add_privacy_check(
    "Calibrated epsilon finite",
    np.isfinite(
        calibrated_epsilon.to_numpy(
            dtype=float
        )
    ).all(),
)

add_privacy_check(
    "Calibrated epsilon does not exceed target",
    (
        calibrated_epsilon_excess
        <=
        CALIBRATED_EPSILON_TOLERANCE
    ).all(),
)

add_privacy_check(
    "Calibrated noise multipliers positive",
    (
        calibrated_sigma
        >
        0
    ).all(),
)

add_privacy_check(
    "Calibrated RDP orders finite",
    np.isfinite(
        calibrated_alpha.to_numpy(
            dtype=float
        )
    ).all(),
)

add_privacy_check(
    "Calibrated RDP orders greater than one",
    (
        calibrated_alpha
        >
        1
    ).all(),
)


# --------------------------------------------------------------------------------------------------
# 14.5 Calibration mechanism validation
# --------------------------------------------------------------------------------------------------

add_privacy_check(
    "Calibrated accountant is RDP",
    CALIBRATED_NOISE_VALIDATION_DF[
        "accountant"
    ].eq("RDP").all(),
)

add_privacy_check(
    "Calibrated sampling is Poisson",
    CALIBRATED_NOISE_VALIDATION_DF[
        "sampling"
    ].eq("Poisson").all(),
)

add_privacy_check(
    "Calibrated clipping is flat L2",
    CALIBRATED_NOISE_VALIDATION_DF[
        "clipping"
    ].eq("flat L2").all(),
)

add_privacy_check(
    "Calibrated loss reduction is mean",
    CALIBRATED_NOISE_VALIDATION_DF[
        "loss_reduction"
    ].eq("mean").all(),
)

add_privacy_check(
    "Calibrated mechanism is DP-SGD",
    CALIBRATED_NOISE_VALIDATION_DF[
        "mechanism"
    ].eq("DP-SGD").all(),
)


# --------------------------------------------------------------------------------------------------
# 14.6 Calibration authority validation
# --------------------------------------------------------------------------------------------------

add_privacy_check(
    "Calibration authority is Notebook 11",
    CALIBRATED_NOISE_VALIDATION_DF[
        "calibration_authority"
    ].eq("Notebook 11").all(),
)

add_privacy_check(
    "Calibration source is Notebook 11 Section 7",
    CALIBRATED_NOISE_VALIDATION_DF[
        "calibration_source"
    ].eq("Notebook 11 Section 7").all(),
)


# --------------------------------------------------------------------------------------------------
# 14.7 Calibrated privacy boundary validation
# --------------------------------------------------------------------------------------------------

add_privacy_check(
    "Calibrated training status is false",
    CALIBRATED_NOISE_VALIDATION_DF[
        "training_performed"
    ].eq(False).all(),
)

add_privacy_check(
    "Calibrated achieved epsilon remains deferred",
    CALIBRATED_NOISE_VALIDATION_DF[
        "achieved_epsilon_status"
    ].eq(
        "DEFERRED_TO_NOTEBOOK_12"
    ).all(),
)

add_privacy_check(
    "Calibrated generator privacy claim disabled",
    CALIBRATED_NOISE_VALIDATION_DF[
        "generator_private"
    ].eq(False).all(),
)

add_privacy_check(
    "Calibrated statistical guidance privacy claim disabled",
    CALIBRATED_NOISE_VALIDATION_DF[
        "statistical_guidance_private"
    ].eq(False).all(),
)

add_privacy_check(
    "Calibrated preprocessing privacy claim disabled",
    CALIBRATED_NOISE_VALIDATION_DF[
        "preprocessing_private"
    ].eq(False).all(),
)

add_privacy_check(
    "Calibrated end-to-end privacy claim disabled",
    CALIBRATED_NOISE_VALIDATION_DF[
        "end_to_end_privacy_claim"
    ].eq(False).all(),
)


# ==================================================================================================
# 15. PERSISTED ARTIFACT VALIDATION
# ==================================================================================================

PERSISTED_ARTIFACTS = {

    "RDP accounting":
        ACCOUNTING_PATH,

    "Per-experiment records":
        PER_EXPERIMENT_PATH,

    "Dataset summary":
        DATASET_SUMMARY_PATH,

    "Model summary":
        MODEL_SUMMARY_PATH,

    "Accounting configuration":
        PRIVACY_ACCOUNTING_CONFIGURATION_PATH,

    "RDP order grid":
        RDP_ORDER_PATH,

    "Privacy audit CSV":
        PRIVACY_AUDIT_PATH,

    "Privacy audit JSON":
        PRIVACY_AUDIT_JSON_PATH,

    "Calibrated noise validation":
        CALIBRATED_NOISE_VALIDATION_PATH,

    "Calibrated privacy configuration":
        CALIBRATED_PRIVACY_CONFIGURATION_PATH,
}


for artifact_name, artifact_path in PERSISTED_ARTIFACTS.items():

    add_privacy_check(
        f"{artifact_name} exists",
        artifact_path.exists(),
    )


# ==================================================================================================
# 16. VALIDATE PERSISTED PRIVACY AUDIT
# ==================================================================================================

if PRIVACY_AUDIT_PATH.exists():

    persisted_audit_df = pd.read_csv(
        PRIVACY_AUDIT_PATH
    )

    add_privacy_check(
        "Persisted privacy audit has one record per dataset",
        len(persisted_audit_df)
        ==
        len(DATASET_IDS),
    )

    add_privacy_check(
        "Persisted privacy audit dataset coverage",
        set(
            persisted_audit_df[
                "dataset"
            ].astype(str)
        )
        ==
        set(DATASET_IDS),
    )

    add_privacy_check(
        "Persisted audit calibration authority is Notebook 11",
        (
            "calibration_authority"
            in
            persisted_audit_df.columns
            and
            persisted_audit_df[
                "calibration_authority"
            ].eq("Notebook 11").all()
        ),
    )

    add_privacy_check(
        "Persisted audit calibration source is Section 7",
        (
            "calibration_source"
            in
            persisted_audit_df.columns
            and
            persisted_audit_df[
                "calibration_source"
            ].eq(
                "Notebook 11 Section 7"
            ).all()
        ),
    )

    add_privacy_check(
        "Persisted audit scope is configured schedule",
        (
            "audit_scope"
            in
            persisted_audit_df.columns
            and
            persisted_audit_df[
                "audit_scope"
            ].eq(
                "configured_schedule"
            ).all()
        ),
    )

else:

    persisted_audit_df = None


# ==================================================================================================
# 17. VALIDATE PERSISTED ACCOUNTING CONFIGURATION
# ==================================================================================================

if PRIVACY_ACCOUNTING_CONFIGURATION_PATH.exists():

    with open(
        PRIVACY_ACCOUNTING_CONFIGURATION_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        persisted_configuration = json.load(f)


    add_privacy_check(
        "Persisted accounting type is configured schedule",
        persisted_configuration.get(
            "accounting_type"
        )
        ==
        "configured_schedule",
    )

    add_privacy_check(
        "Persisted training status is false",
        persisted_configuration.get(
            "training_performed"
        )
        is False,
    )

    add_privacy_check(
        "Persisted achieved epsilon is deferred",
        persisted_configuration.get(
            "achieved_epsilon_status"
        )
        ==
        "DEFERRED_TO_NOTEBOOK_12",
    )

    add_privacy_check(
        "Persisted end-to-end privacy claim is false",
        persisted_configuration.get(
            "end_to_end_privacy_claim"
        )
        is False,
    )


# ==================================================================================================
# 18. VALIDATE PERSISTED CALIBRATED CONFIGURATION
# ==================================================================================================

if CALIBRATED_PRIVACY_CONFIGURATION_PATH.exists():

    with open(
        CALIBRATED_PRIVACY_CONFIGURATION_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        persisted_calibrated_configuration = json.load(f)


    add_privacy_check(
        "Persisted calibrated authority is Notebook 11",
        persisted_calibrated_configuration.get(
            "calibration_authority"
        )
        ==
        "Notebook 11",
    )

    add_privacy_check(
        "Persisted calibrated source is Section 7",
        persisted_calibrated_configuration.get(
            "calibration_source"
        )
        ==
        "Notebook 11 Section 7",
    )

    add_privacy_check(
        "Persisted calibrated accounting type exists",
        persisted_calibrated_configuration.get(
            "accounting_type"
        )
        ==
        "calibrated_schedule",
    )

    add_privacy_check(
        "Persisted calibrated accountant is RDP",
        persisted_calibrated_configuration.get(
            "accountant"
        )
        ==
        "RDP",
    )

    add_privacy_check(
        "Persisted calibrated sampling is Poisson",
        persisted_calibrated_configuration.get(
            "sampling"
        )
        ==
        "Poisson",
    )

    add_privacy_check(
        "Persisted calibrated clipping is flat L2",
        persisted_calibrated_configuration.get(
            "clipping"
        )
        ==
        "flat L2",
    )

    add_privacy_check(
        "Persisted calibrated training status is false",
        persisted_calibrated_configuration.get(
            "training_performed"
        )
        is False,
    )

    add_privacy_check(
        "Persisted calibrated achieved epsilon is deferred",
        persisted_calibrated_configuration.get(
            "achieved_epsilon_status"
        )
        ==
        "DEFERRED_TO_NOTEBOOK_12",
    )

    add_privacy_check(
        "Persisted calibrated end-to-end claim is false",
        persisted_calibrated_configuration.get(
            "end_to_end_privacy_claim"
        )
        is False,
    )


    persisted_calibrated_datasets = set(
        persisted_calibrated_configuration.get(
            "datasets",
            []
        )
    )

    add_privacy_check(
        "Persisted calibrated dataset coverage",
        persisted_calibrated_datasets
        ==
        set(DATASET_IDS),
    )


    persisted_sigma = (
        persisted_calibrated_configuration.get(
            "calibrated_noise_multipliers",
            {}
        )
    )

    add_privacy_check(
        "Persisted calibrated noise multipliers cover all datasets",
        set(
            persisted_sigma.keys()
        )
        ==
        set(DATASET_IDS),
    )


    persisted_calibrated_epsilon = (
        persisted_calibrated_configuration.get(
            "calibrated_epsilon",
            {}
        )
    )

    add_privacy_check(
        "Persisted calibrated epsilon covers all datasets",
        set(
            persisted_calibrated_epsilon.keys()
        )
        ==
        set(DATASET_IDS),
    )


    # ----------------------------------------------------------------------------------------------
    # Compare persisted sigma values with the authoritative Section 15 dataframe.
    # ----------------------------------------------------------------------------------------------

    sigma_match = True

    for dataset in DATASET_IDS:

        if dataset not in persisted_sigma:
            sigma_match = False
            break

        expected_sigma = float(
            CALIBRATED_NOISE_VALIDATION_DF.loc[
                CALIBRATED_NOISE_VALIDATION_DF[
                    "dataset"
                ].astype(str)
                ==
                dataset,
                "calibrated_noise_multiplier",
            ].iloc[0]
        )

        actual_sigma = float(
            persisted_sigma[dataset]
        )

        if not np.isclose(
            actual_sigma,
            expected_sigma,
            rtol=0.0,
            atol=1e-12,
        ):
            sigma_match = False
            break


    add_privacy_check(
        "Persisted calibrated noise multipliers match Section 15",
        sigma_match,
    )


    # ----------------------------------------------------------------------------------------------
    # Compare persisted calibrated epsilon values with Section 15.
    # ----------------------------------------------------------------------------------------------

    calibrated_epsilon_match = True

    for dataset in DATASET_IDS:

        if dataset not in persisted_calibrated_epsilon:
            calibrated_epsilon_match = False
            break

        expected_epsilon = float(
            CALIBRATED_NOISE_VALIDATION_DF.loc[
                CALIBRATED_NOISE_VALIDATION_DF[
                    "dataset"
                ].astype(str)
                ==
                dataset,
                "calibrated_epsilon",
            ].iloc[0]
        )

        actual_epsilon = float(
            persisted_calibrated_epsilon[dataset]
        )

        if not np.isclose(
            actual_epsilon,
            expected_epsilon,
            rtol=0.0,
            atol=1e-12,
        ):
            calibrated_epsilon_match = False
            break


    add_privacy_check(
        "Persisted calibrated epsilon matches Section 15",
        calibrated_epsilon_match,
    )


# ==================================================================================================
# 19. VALIDATE PERSISTED RDP ORDER GRID
# ==================================================================================================

if RDP_ORDER_PATH.exists():

    persisted_rdp_orders = pd.read_csv(
        RDP_ORDER_PATH
    )

    add_privacy_check(
        "Persisted RDP order column exists",
        "rdp_order"
        in
        persisted_rdp_orders.columns,
    )

    if "rdp_order" in persisted_rdp_orders.columns:

        add_privacy_check(
            "Persisted RDP order count matches configured grid",
            len(
                persisted_rdp_orders
            )
            ==
            len(RDP_ALPHAS),
        )

        add_privacy_check(
            "Persisted RDP orders match configured grid",
            np.allclose(
                persisted_rdp_orders[
                    "rdp_order"
                ].to_numpy(
                    dtype=float
                ),
                np.asarray(
                    RDP_ALPHAS,
                    dtype=float
                ),
                rtol=0.0,
                atol=1e-12,
            ),
        )


# ==================================================================================================
# 20. FINAL RESULT TABLE
# ==================================================================================================

FINAL_PRIVACY_CHECKS_DF = pd.DataFrame(
    FINAL_PRIVACY_CHECKS
)

all_checks_pass = (
    not FINAL_PRIVACY_CHECKS_DF.empty
    and
    FINAL_PRIVACY_CHECKS_DF[
        "status"
    ].eq("PASS").all()
)


if not all_checks_pass:

    display(
        FINAL_PRIVACY_CHECKS_DF
    )

    raise RuntimeError(
        "Notebook 11 final privacy verification failed."
    )


# ==================================================================================================
# 21. PERSIST FINAL VERIFICATION
# ==================================================================================================

FINAL_PRIVACY_VERIFICATION_PATH = (
    DIRS["validation"]
    /
    "sppgan_privacy_accounting_final_verification.csv"
)

FINAL_PRIVACY_CHECKS_DF.to_csv(
    FINAL_PRIVACY_VERIFICATION_PATH,
    index=False,
)


# ==================================================================================================
# 22. DISPLAY FINAL RESULTS
# ==================================================================================================

display(
    FINAL_PRIVACY_CHECKS_DF
)

print(
    f"✓ Final privacy verification saved:\n"
    f"  {FINAL_PRIVACY_VERIFICATION_PATH}"
)

print(
    f"✓ Total final checks: "
    f"{len(FINAL_PRIVACY_CHECKS_DF)}"
)

print(
    "✓ All final privacy checks: PASS"
)

print(
    "✓ Configured-schedule epsilon remains distinct from calibrated epsilon"
)

print(
    "✓ Calibrated privacy authority: NOTEBOOK 11 SECTION 7"
)

print(
    "✓ Calibrated privacy artifacts validated"
)

print(
    "✓ Achieved epsilon remains deferred to Notebook 12"
)

print(
    "✓ End-to-end privacy claim: FALSE"
)

print(
    "\nSECTION 17 STATUS: PASS"
)


17. FINAL PRIVACY VERIFICATION

----------------------------------------------------------------------------------------------------
CALIBRATION ARTIFACT VALIDATION
----------------------------------------------------------------------------------------------------


,check,status
0,Notebook 10 root exists,PASS
1,Notebook 10 privacy configuration exists,PASS
2,Notebook 10 privacy metadata exists,PASS
3,RDP accountant selected,PASS
4,Poisson sampling selected,PASS
...,...,...
83,Persisted calibrated noise multipliers match S...,PASS
84,Persisted calibrated epsilon matches Section 15,PASS
85,Persisted RDP order column exists,PASS
86,Persisted RDP order count matches configured grid,PASS


✓ Final privacy verification saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11/validation/sppgan_privacy_accounting_final_verification.csv
✓ Total final checks: 88
✓ All final privacy checks: PASS
✓ Configured-schedule epsilon remains distinct from calibrated epsilon
✓ Calibrated privacy authority: NOTEBOOK 11 SECTION 7
✓ Calibrated privacy artifacts validated
✓ Achieved epsilon remains deferred to Notebook 12
✓ End-to-end privacy claim: FALSE

SECTION 17 STATUS: PASS


In [28]:
# ==================================================================================================
# SECTION 18 — COMPLETION SUMMARY
# ==================================================================================================
#
# Purpose
# -------
# Generate the final Notebook 11 completion summary and reproducibility manifest.
#
# IMPORTANT PRIVACY SEMANTICS
# ---------------------------
# Configured-schedule epsilon:
#     Original Section 9 accounting using the configured noise multiplier.
#
# Calibrated epsilon:
#     Dataset-specific calibration produced by Notebook 11 Section 7.
#
# Achieved training epsilon:
#     Deferred to Notebook 12.
#
# Privacy boundary:
#     DP protection applies to the SPP-GAN discriminator / critic only.
#
# End-to-end privacy:
#     NOT established.
#
# STATE RECOVERY
# --------------
# This section reconstructs required state from persisted Notebook 11 artifacts.
#
# ==================================================================================================

print("=" * 100)
print("18. COMPLETION SUMMARY")
print("=" * 100)

from pathlib import Path
from datetime import datetime
import hashlib
import json
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. CANONICAL NOTEBOOK 11 PATHS
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NB11_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_11"
)

NB11_ACCOUNTING_ROOT = (
    NB11_ROOT
    / "accounting"
)

NB11_CONFIGURATION_ROOT = (
    NB11_ROOT
    / "configuration"
)

NB11_AUDIT_ROOT = (
    NB11_ROOT
    / "audit"
)

NB11_VALIDATION_ROOT = (
    NB11_ROOT
    / "validation"
)

print("✓ Canonical Notebook 11 paths established")
print(f"✓ Project root   : {PROJECT_ROOT}")
print(f"✓ Notebook 11    : {NB11_ROOT}")


# --------------------------------------------------------------------------------------------------
# 2. CANONICAL ARTIFACT PATHS
# --------------------------------------------------------------------------------------------------

CONFIGURED_ACCOUNTING_PATH = (
    NB11_ACCOUNTING_ROOT
    / "sppgan_configured_schedule_rdp_accounting.csv"
)

PER_EXPERIMENT_PRIVACY_RECORDS_PATH = (
    NB11_ACCOUNTING_ROOT
    / "per_experiment_privacy_records.csv"
)

DATASET_PRIVACY_SUMMARY_PATH = (
    NB11_ACCOUNTING_ROOT
    / "dataset_privacy_summary.csv"
)

MODEL_LEVEL_PRIVACY_SUMMARY_PATH = (
    NB11_ACCOUNTING_ROOT
    / "model_level_privacy_summary.csv"
)

PRIVACY_AUDIT_CSV_PATH = (
    NB11_AUDIT_ROOT
    / "sppgan_privacy_audit.csv"
)

PRIVACY_AUDIT_JSON_PATH = (
    NB11_AUDIT_ROOT
    / "sppgan_privacy_audit.json"
)

FINAL_PRIVACY_VERIFICATION_PATH = (
    NB11_VALIDATION_ROOT
    / "sppgan_privacy_accounting_final_verification.csv"
)

CALIBRATED_NOISE_VALIDATION_PATH = (
    NB11_VALIDATION_ROOT
    / "calibrated_noise_validation.csv"
)

CALIBRATED_PRIVACY_CONFIGURATION_PATH = (
    NB11_CONFIGURATION_ROOT
    / "sppgan_calibrated_privacy_configuration.json"
)

COMPLETION_MANIFEST_PATH = (
    NB11_VALIDATION_ROOT
    / "notebook_11_completion_manifest.json"
)


# --------------------------------------------------------------------------------------------------
# 3. VERIFY ROOT DIRECTORIES
# --------------------------------------------------------------------------------------------------

assert PROJECT_ROOT.exists(), (
    f"Project root does not exist:\n{PROJECT_ROOT}"
)

assert NB11_ROOT.exists(), (
    f"Notebook 11 root does not exist:\n{NB11_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED PERSISTED ARTIFACTS
# --------------------------------------------------------------------------------------------------

REQUIRED_ARTIFACTS = {
    "configured_schedule_accounting":
        CONFIGURED_ACCOUNTING_PATH,

    "per_experiment_privacy_records":
        PER_EXPERIMENT_PRIVACY_RECORDS_PATH,

    "dataset_privacy_summary":
        DATASET_PRIVACY_SUMMARY_PATH,

    "model_level_privacy_summary":
        MODEL_LEVEL_PRIVACY_SUMMARY_PATH,

    "privacy_audit_csv":
        PRIVACY_AUDIT_CSV_PATH,

    "privacy_audit_json":
        PRIVACY_AUDIT_JSON_PATH,

    "final_privacy_verification":
        FINAL_PRIVACY_VERIFICATION_PATH,

    "calibrated_noise_validation":
        CALIBRATED_NOISE_VALIDATION_PATH,

    "calibrated_privacy_configuration":
        CALIBRATED_PRIVACY_CONFIGURATION_PATH,
}


MISSING_ARTIFACTS = [
    name
    for name, path in REQUIRED_ARTIFACTS.items()
    if not path.exists()
]

if MISSING_ARTIFACTS:
    raise RuntimeError(
        "Required Notebook 11 artifacts are missing:\n"
        + "\n".join(
            f"  - {name}: {REQUIRED_ARTIFACTS[name]}"
            for name in MISSING_ARTIFACTS
        )
    )

print("✓ All required persisted Notebook 11 artifacts found")


# --------------------------------------------------------------------------------------------------
# 5. LOAD CONFIGURED-SCHEDULE ACCOUNTING
# --------------------------------------------------------------------------------------------------

CONFIGURED_ACCOUNTING_DF = pd.read_csv(
    CONFIGURED_ACCOUNTING_PATH
)

REQUIRED_CONFIGURED_COLUMNS = [
    "dataset",
    "n_train",
    "sample_rate",
    "noise_multiplier",
    "max_grad_norm",
    "epochs",
    "steps_per_epoch",
    "total_steps",
    "delta",
    "target_epsilon",
    "configured_schedule_epsilon",
    "optimal_rdp_order",
    "rdp_order_count",
    "accounting_type",
    "achieved_epsilon_status",
]

MISSING_COLUMNS = [
    column
    for column in REQUIRED_CONFIGURED_COLUMNS
    if column not in CONFIGURED_ACCOUNTING_DF.columns
]

if MISSING_COLUMNS:
    raise RuntimeError(
        "Configured accounting artifact is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in MISSING_COLUMNS
        )
    )

assert not CONFIGURED_ACCOUNTING_DF.empty
assert CONFIGURED_ACCOUNTING_DF["dataset"].is_unique

EXPECTED_DATASETS = (
    CONFIGURED_ACCOUNTING_DF["dataset"]
    .astype(str)
    .tolist()
)

assert len(EXPECTED_DATASETS) == 3

assert (
    CONFIGURED_ACCOUNTING_DF["accounting_type"]
    .astype(str)
    .str.lower()
    .eq("configured_schedule")
    .all()
)

assert (
    CONFIGURED_ACCOUNTING_DF["achieved_epsilon_status"]
    .astype(str)
    .eq("DEFERRED_TO_NOTEBOOK_12")
    .all()
)

print("✓ Configured-schedule accounting loaded and validated")


# --------------------------------------------------------------------------------------------------
# 6. RECONSTRUCT CONFIGURATION
# --------------------------------------------------------------------------------------------------

TARGET_EPSILON = float(
    CONFIGURED_ACCOUNTING_DF[
        "target_epsilon"
    ].iloc[0]
)

MAX_GRAD_NORM = float(
    CONFIGURED_ACCOUNTING_DF[
        "max_grad_norm"
    ].iloc[0]
)

DP_EPOCHS = int(
    CONFIGURED_ACCOUNTING_DF[
        "epochs"
    ].iloc[0]
)

DELTA_VALUES = (
    CONFIGURED_ACCOUNTING_DF[
        "delta"
    ].astype(float)
)

assert DELTA_VALUES.nunique() == 1

DELTA = float(
    DELTA_VALUES.iloc[0]
)

# DP batch size is recovered from N × sampling rate.
DP_BATCH_SIZE_VALUES = (
    CONFIGURED_ACCOUNTING_DF[
        "n_train"
    ].astype(float)
    *
    CONFIGURED_ACCOUNTING_DF[
        "sample_rate"
    ].astype(float)
)

DP_BATCH_SIZE = int(
    round(DP_BATCH_SIZE_VALUES.median())
)

FRAMEWORK_NAME = "SPP-GAN"
ACCOUNTANT_TYPE = "RDP"
SAMPLING = "Poisson"
CLIPPING = "flat L2"
LOSS_REDUCTION = "mean"

PRIVACY_BOUNDARY = {
    "generator_private": False,
    "statistical_guidance_private": False,
    "preprocessing_private": False,
    "end_to_end_privacy_claim": False,
}

print("✓ Configuration reconstructed from persisted accounting")


# --------------------------------------------------------------------------------------------------
# 7. CONFIGURATION VALIDATION
# --------------------------------------------------------------------------------------------------

assert TARGET_EPSILON == 5.0
assert MAX_GRAD_NORM == 1.0
assert DP_EPOCHS == 300
assert DELTA == 1e-5
assert ACCOUNTANT_TYPE == "RDP"
assert SAMPLING == "Poisson"
assert CLIPPING == "flat L2"
assert LOSS_REDUCTION == "mean"

print("✓ DP configuration validated")


# --------------------------------------------------------------------------------------------------
# 8. LOAD DATASET PRIVACY SUMMARY
# --------------------------------------------------------------------------------------------------

DATASET_PRIVACY_SUMMARY_DF = pd.read_csv(
    DATASET_PRIVACY_SUMMARY_PATH
)

assert set(
    DATASET_PRIVACY_SUMMARY_DF["dataset"]
) == set(EXPECTED_DATASETS)

assert len(DATASET_PRIVACY_SUMMARY_DF) == 3

print("✓ Dataset privacy summary loaded")


# --------------------------------------------------------------------------------------------------
# 9. LOAD MODEL-LEVEL PRIVACY SUMMARY
# --------------------------------------------------------------------------------------------------

MODEL_LEVEL_PRIVACY_SUMMARY = pd.read_csv(
    MODEL_LEVEL_PRIVACY_SUMMARY_PATH
)

assert not MODEL_LEVEL_PRIVACY_SUMMARY.empty

print("✓ Model-level privacy summary loaded")


# --------------------------------------------------------------------------------------------------
# 10. LOAD PRIVACY AUDIT
# --------------------------------------------------------------------------------------------------

PRIVACY_AUDIT_DF = pd.read_csv(
    PRIVACY_AUDIT_CSV_PATH
)

assert set(
    PRIVACY_AUDIT_DF["dataset"]
) == set(EXPECTED_DATASETS)

assert len(PRIVACY_AUDIT_DF) == 3

print("✓ Privacy audit loaded")


# --------------------------------------------------------------------------------------------------
# 11. LOAD FINAL PRIVACY VERIFICATION
# --------------------------------------------------------------------------------------------------

FINAL_PRIVACY_VERIFICATION_DF = pd.read_csv(
    FINAL_PRIVACY_VERIFICATION_PATH
)

assert not FINAL_PRIVACY_VERIFICATION_DF.empty

assert "status" in (
    FINAL_PRIVACY_VERIFICATION_DF.columns
)

FINAL_STATUS = (
    FINAL_PRIVACY_VERIFICATION_DF[
        "status"
    ]
    .astype(str)
    .str.upper()
)

FINAL_VERIFICATION_TOTAL = int(
    len(FINAL_PRIVACY_VERIFICATION_DF)
)

FINAL_VERIFICATION_PASS = int(
    FINAL_STATUS.eq("PASS").sum()
)

assert (
    FINAL_VERIFICATION_PASS
    == FINAL_VERIFICATION_TOTAL
)

print(
    f"✓ Final privacy verification loaded: "
    f"{FINAL_VERIFICATION_PASS}/"
    f"{FINAL_VERIFICATION_TOTAL} PASS"
)


# --------------------------------------------------------------------------------------------------
# 12. LOAD CALIBRATED PRIVACY VALIDATION
# --------------------------------------------------------------------------------------------------

CALIBRATED_NOISE_VALIDATION_DF = pd.read_csv(
    CALIBRATED_NOISE_VALIDATION_PATH
)

print()
print("CALIBRATED PRIVACY ARTIFACT SCHEMA")
print("-" * 100)
print(
    "Columns:"
)
for column in CALIBRATED_NOISE_VALIDATION_DF.columns:
    print(f"  - {column}")


# --------------------------------------------------------------------------------------------------
# 13. CALIBRATED SCHEMA RESOLUTION
# --------------------------------------------------------------------------------------------------
#
# Notebook 11 Section 15 is authoritative for the persisted artifact schema.
#
# The RDP-order field may be persisted under a different but semantically
# equivalent name. We resolve the existing column instead of requiring a
# fabricated column named 'calibrated_rdp_order'.
#
# --------------------------------------------------------------------------------------------------

CALIBRATED_DATASET_COLUMN = "dataset"
CALIBRATED_SIGMA_COLUMN = "calibrated_noise_multiplier"
CALIBRATED_EPSILON_COLUMN = "calibrated_epsilon"
CALIBRATED_DELTA_COLUMN = "delta"
CALIBRATED_STATUS_COLUMN = "status"

RDP_ORDER_CANDIDATES = [
    "calibrated_rdp_order",
    "optimal_rdp_order",
    "calibrated_optimal_rdp_order",
    "optimal_order",
    "rdp_order",
]

RDP_ORDER_COLUMN = next(
    (
        column
        for column in RDP_ORDER_CANDIDATES
        if column in CALIBRATED_NOISE_VALIDATION_DF.columns
    ),
    None,
)

if RDP_ORDER_COLUMN is None:
    raise RuntimeError(
        "The calibrated privacy artifact does not contain a recognized "
        "RDP-order column.\n\n"
        "Expected one of:\n"
        + "\n".join(
            f"  - {column}"
            for column in RDP_ORDER_CANDIDATES
        )
        + "\n\n"
        "Actual columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in CALIBRATED_NOISE_VALIDATION_DF.columns
        )
    )

print(
    f"✓ Calibrated RDP-order column resolved: "
    f"{RDP_ORDER_COLUMN}"
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATE CALIBRATED ARTIFACT
# --------------------------------------------------------------------------------------------------

REQUIRED_CALIBRATED_COLUMNS = [
    CALIBRATED_DATASET_COLUMN,
    CALIBRATED_SIGMA_COLUMN,
    CALIBRATED_EPSILON_COLUMN,
    CALIBRATED_DELTA_COLUMN,
    CALIBRATED_STATUS_COLUMN,
    RDP_ORDER_COLUMN,
]

MISSING_CALIBRATED_COLUMNS = [
    column
    for column in REQUIRED_CALIBRATED_COLUMNS
    if column not in CALIBRATED_NOISE_VALIDATION_DF.columns
]

if MISSING_CALIBRATED_COLUMNS:
    raise RuntimeError(
        "Calibrated privacy artifact is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in MISSING_CALIBRATED_COLUMNS
        )
    )

assert len(
    CALIBRATED_NOISE_VALIDATION_DF
) == 3

assert (
    CALIBRATED_NOISE_VALIDATION_DF[
        CALIBRATED_DATASET_COLUMN
    ].is_unique
)

assert set(
    CALIBRATED_NOISE_VALIDATION_DF[
        CALIBRATED_DATASET_COLUMN
    ]
) == set(EXPECTED_DATASETS)

assert (
    CALIBRATED_NOISE_VALIDATION_DF[
        CALIBRATED_STATUS_COLUMN
    ]
    .astype(str)
    .str.upper()
    .eq("PASS")
    .all()
)

print("✓ Calibrated privacy artifact schema validated")


# --------------------------------------------------------------------------------------------------
# 15. CREATE NORMALIZED CALIBRATED SERIES
# --------------------------------------------------------------------------------------------------

CALIBRATED_EPSILON_TOLERANCE = 1e-6

CALIBRATED_EPSILON_SERIES = (
    CALIBRATED_NOISE_VALIDATION_DF
    .set_index(CALIBRATED_DATASET_COLUMN)[
        CALIBRATED_EPSILON_COLUMN
    ]
    .astype(float)
)

CALIBRATED_SIGMA_SERIES = (
    CALIBRATED_NOISE_VALIDATION_DF
    .set_index(CALIBRATED_DATASET_COLUMN)[
        CALIBRATED_SIGMA_COLUMN
    ]
    .astype(float)
)

CALIBRATED_ALPHA_SERIES = (
    CALIBRATED_NOISE_VALIDATION_DF
    .set_index(CALIBRATED_DATASET_COLUMN)[
        RDP_ORDER_COLUMN
    ]
    .astype(float)
)

assert np.isfinite(
    CALIBRATED_EPSILON_SERIES
).all()

assert np.isfinite(
    CALIBRATED_SIGMA_SERIES
).all()

assert np.isfinite(
    CALIBRATED_ALPHA_SERIES
).all()

assert (
    CALIBRATED_EPSILON_SERIES
    <= TARGET_EPSILON
    + CALIBRATED_EPSILON_TOLERANCE
).all()

print("✓ Calibrated privacy values validated")


# --------------------------------------------------------------------------------------------------
# 16. CALIBRATION AUTHORITY VALIDATION
# --------------------------------------------------------------------------------------------------

CALIBRATION_AUTHORITY = "Notebook 11"
CALIBRATION_SOURCE = "Notebook 11 Section 7"

if (
    "calibration_authority"
    in CALIBRATED_NOISE_VALIDATION_DF.columns
):

    assert (
        CALIBRATED_NOISE_VALIDATION_DF[
            "calibration_authority"
        ]
        .astype(str)
        .eq(CALIBRATION_AUTHORITY)
        .all()
    )

if (
    "calibration_source"
    in CALIBRATED_NOISE_VALIDATION_DF.columns
):

    assert (
        CALIBRATED_NOISE_VALIDATION_DF[
            "calibration_source"
        ]
        .astype(str)
        .eq(CALIBRATION_SOURCE)
        .all()
    )

print("✓ Calibration authority validated")
print(f"  Authority : {CALIBRATION_AUTHORITY}")
print(f"  Source    : {CALIBRATION_SOURCE}")


# --------------------------------------------------------------------------------------------------
# 17. LOAD CALIBRATED PRIVACY CONFIGURATION
# --------------------------------------------------------------------------------------------------

with open(
    CALIBRATED_PRIVACY_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as file:

    CALIBRATED_PRIVACY_CONFIGURATION = json.load(
        file
    )

assert isinstance(
    CALIBRATED_PRIVACY_CONFIGURATION,
    dict
)

assert (
    CALIBRATED_PRIVACY_CONFIGURATION.get(
        "calibration_authority"
    )
    == CALIBRATION_AUTHORITY
)

assert (
    CALIBRATED_PRIVACY_CONFIGURATION.get(
        "calibration_source"
    )
    == CALIBRATION_SOURCE
)

assert (
    CALIBRATED_PRIVACY_CONFIGURATION.get(
        "training_performed"
    )
    is False
)

assert (
    CALIBRATED_PRIVACY_CONFIGURATION.get(
        "achieved_epsilon_status"
    )
    == "DEFERRED_TO_NOTEBOOK_12"
)

assert (
    CALIBRATED_PRIVACY_CONFIGURATION.get(
        "end_to_end_privacy_claim"
    )
    is False
)

print(
    "✓ Persisted calibrated privacy configuration validated"
)


# --------------------------------------------------------------------------------------------------
# 18. PRIVACY AUDIT SEMANTICS
# --------------------------------------------------------------------------------------------------

if "training_performed" in PRIVACY_AUDIT_DF.columns:

    TRAINING_STATUS_VALUES = (
        PRIVACY_AUDIT_DF[
            "training_performed"
        ]
        .astype(str)
        .str.upper()
    )

    assert TRAINING_STATUS_VALUES.isin(
        ["FALSE", "0"]
    ).all()

if "achieved_epsilon_status" in PRIVACY_AUDIT_DF.columns:

    assert (
        PRIVACY_AUDIT_DF[
            "achieved_epsilon_status"
        ]
        .astype(str)
        .eq("DEFERRED_TO_NOTEBOOK_12")
        .all()
    )

print("✓ Privacy audit semantics validated")


# --------------------------------------------------------------------------------------------------
# 19. CONFIGURED-SCHEDULE SUMMARY
# --------------------------------------------------------------------------------------------------

CONFIGURED_EPSILON_SERIES = (
    CONFIGURED_ACCOUNTING_DF
    .set_index("dataset")[
        "configured_schedule_epsilon"
    ]
    .astype(float)
)

CONFIGURED_ALPHA_SERIES = (
    CONFIGURED_ACCOUNTING_DF
    .set_index("dataset")[
        "optimal_rdp_order"
    ]
    .astype(float)
)

CONFIGURED_DELTA_SERIES = (
    CONFIGURED_ACCOUNTING_DF
    .set_index("dataset")[
        "delta"
    ]
    .astype(float)
)

CONFIGURED_TARGET_SERIES = (
    CONFIGURED_ACCOUNTING_DF
    .set_index("dataset")[
        "target_epsilon"
    ]
    .astype(float)
)

CONFIGURED_ABS_DIFF_SERIES = (
    CONFIGURED_EPSILON_SERIES
    - CONFIGURED_TARGET_SERIES
).abs()

CONFIGURED_WITHIN_TARGET = int(
    (
        CONFIGURED_EPSILON_SERIES
        <= CONFIGURED_TARGET_SERIES
    ).sum()
)

CONFIGURED_EXCEEDING_TARGET = int(
    (
        CONFIGURED_EPSILON_SERIES
        > CONFIGURED_TARGET_SERIES
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 20. CALIBRATED SUMMARY
# --------------------------------------------------------------------------------------------------

CALIBRATED_EPSILON_MIN = float(
    CALIBRATED_EPSILON_SERIES.min()
)

CALIBRATED_EPSILON_MAX = float(
    CALIBRATED_EPSILON_SERIES.max()
)

CALIBRATED_EPSILON_MEAN = float(
    CALIBRATED_EPSILON_SERIES.mean()
)

CALIBRATED_DATASETS_WITHIN_TARGET = int(
    (
        CALIBRATED_EPSILON_SERIES
        <= TARGET_EPSILON
        + CALIBRATED_EPSILON_TOLERANCE
    ).sum()
)

CALIBRATED_DATASETS_EXCEEDING_TARGET = int(
    (
        CALIBRATED_EPSILON_SERIES
        > TARGET_EPSILON
        + CALIBRATED_EPSILON_TOLERANCE
    ).sum()
)

assert CALIBRATED_DATASETS_WITHIN_TARGET == 3
assert CALIBRATED_DATASETS_EXCEEDING_TARGET == 0

print("✓ Privacy summary statistics calculated")


# --------------------------------------------------------------------------------------------------
# 21. SHA256 HELPER
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 22. ARTIFACT HASHES
# --------------------------------------------------------------------------------------------------

ARTIFACT_HASHES = {}

for artifact_name, artifact_path in REQUIRED_ARTIFACTS.items():

    ARTIFACT_HASHES[artifact_name] = {
        "path": str(artifact_path),
        "sha256": sha256_file(artifact_path),
        "size_bytes": int(
            artifact_path.stat().st_size
        ),
    }

print("✓ Artifact SHA256 hashes generated")


# --------------------------------------------------------------------------------------------------
# 23. COMPLETION MANIFEST
# --------------------------------------------------------------------------------------------------

COMPLETION_TIMESTAMP = (
    datetime.now().astimezone().isoformat()
)

COMPLETION_MANIFEST = {

    "manifest_version": "3.1",

    "framework": FRAMEWORK_NAME,

    "notebook":
        "Notebook 11 — Privacy Calibration, Accounting & Verification",

    "completion_timestamp":
        COMPLETION_TIMESTAMP,

    "datasets":
        EXPECTED_DATASETS,

    "dataset_count":
        len(EXPECTED_DATASETS),

    "privacy_mechanism":
        "DP-SGD",

    "protected_component":
        "SPP-GAN discriminator / critic",

    "accountant":
        ACCOUNTANT_TYPE,

    "sampling":
        SAMPLING,

    "clipping":
        CLIPPING,

    "loss_reduction":
        LOSS_REDUCTION,

    "target_epsilon":
        TARGET_EPSILON,

    "delta":
        DELTA,

    "maximum_gradient_norm":
        MAX_GRAD_NORM,

    "dp_batch_size":
        DP_BATCH_SIZE,

    "dp_epochs":
        DP_EPOCHS,

    "configured_schedule": {

        "accounting_type":
            "configured_schedule",

        "training_performed":
            False,

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "configured_epsilon_min":
            float(CONFIGURED_EPSILON_SERIES.min()),

        "configured_epsilon_max":
            float(CONFIGURED_EPSILON_SERIES.max()),

        "configured_epsilon_mean":
            float(CONFIGURED_EPSILON_SERIES.mean()),

        "datasets_within_target":
            CONFIGURED_WITHIN_TARGET,

        "datasets_exceeding_target":
            CONFIGURED_EXCEEDING_TARGET,
    },

    "noise_calibration_status":
        "COMPLETED",

    "calibration_authority":
        CALIBRATION_AUTHORITY,

    "calibration_source":
        CALIBRATION_SOURCE,

    "calibrated_rdp_order_column":
        RDP_ORDER_COLUMN,

    "calibrated_epsilon_available":
        True,

    "calibrated_noise_validation":
        str(CALIBRATED_NOISE_VALIDATION_PATH),

    "calibrated_privacy_configuration":
        str(CALIBRATED_PRIVACY_CONFIGURATION_PATH),

    "calibrated_epsilon_min":
        CALIBRATED_EPSILON_MIN,

    "calibrated_epsilon_max":
        CALIBRATED_EPSILON_MAX,

    "calibrated_epsilon_mean":
        CALIBRATED_EPSILON_MEAN,

    "calibrated_epsilon_budget_status":
        "WITHIN_TARGET",

    "calibrated_datasets_within_target":
        CALIBRATED_DATASETS_WITHIN_TARGET,

    "calibrated_datasets_exceeding_target":
        CALIBRATED_DATASETS_EXCEEDING_TARGET,

    "training_status":
        "NOT_PERFORMED",

    "achieved_training_epsilon":
        "DEFERRED_TO_NOTEBOOK_12",

    "synthetic_generation_status":
        "NOT_PERFORMED",

    "privacy_boundary": {

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,
    },

    "end_to_end_privacy_claim":
        False,

    "final_verification": {

        "total_checks":
            FINAL_VERIFICATION_TOTAL,

        "passed_checks":
            FINAL_VERIFICATION_PASS,

        "status":
            "PASS",
    },

    "artifacts": {
        name: str(path)
        for name, path in REQUIRED_ARTIFACTS.items()
    },

    "artifact_hashes":
        ARTIFACT_HASHES,

    "overall_status":
        "PASS",

    "next_notebook":
        "Notebook 12 — SPP-GAN DP Training",
}


# --------------------------------------------------------------------------------------------------
# 24. FINAL MANIFEST CONSISTENCY VALIDATION
# --------------------------------------------------------------------------------------------------

assert (
    COMPLETION_MANIFEST[
        "noise_calibration_status"
    ]
    == "COMPLETED"
)

assert (
    COMPLETION_MANIFEST[
        "calibration_authority"
    ]
    == "Notebook 11"
)

assert (
    COMPLETION_MANIFEST[
        "calibration_source"
    ]
    == "Notebook 11 Section 7"
)

assert (
    COMPLETION_MANIFEST[
        "calibrated_epsilon_budget_status"
    ]
    == "WITHIN_TARGET"
)

assert (
    COMPLETION_MANIFEST[
        "calibrated_datasets_exceeding_target"
    ]
    == 0
)

assert (
    COMPLETION_MANIFEST[
        "training_status"
    ]
    == "NOT_PERFORMED"
)

assert (
    COMPLETION_MANIFEST[
        "achieved_training_epsilon"
    ]
    == "DEFERRED_TO_NOTEBOOK_12"
)

assert (
    COMPLETION_MANIFEST[
        "end_to_end_privacy_claim"
    ]
    is False
)

assert (
    COMPLETION_MANIFEST[
        "overall_status"
    ]
    == "PASS"
)

print(
    "✓ Completion manifest consistency validation: PASS"
)


# --------------------------------------------------------------------------------------------------
# 25. SAVE COMPLETION MANIFEST
# --------------------------------------------------------------------------------------------------

with open(
    COMPLETION_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        COMPLETION_MANIFEST,
        file,
        indent=2,
        ensure_ascii=False,
    )

assert COMPLETION_MANIFEST_PATH.exists()

print("✓ Completion manifest saved:")
print(f"  {COMPLETION_MANIFEST_PATH}")


# --------------------------------------------------------------------------------------------------
# 26. RELOAD MANIFEST
# --------------------------------------------------------------------------------------------------

with open(
    COMPLETION_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:

    RELOADED_COMPLETION_MANIFEST = json.load(
        file
    )

assert (
    RELOADED_COMPLETION_MANIFEST[
        "manifest_version"
    ]
    == "3.1"
)

assert (
    RELOADED_COMPLETION_MANIFEST[
        "framework"
    ]
    == "SPP-GAN"
)

assert (
    RELOADED_COMPLETION_MANIFEST[
        "noise_calibration_status"
    ]
    == "COMPLETED"
)

assert (
    RELOADED_COMPLETION_MANIFEST[
        "calibration_authority"
    ]
    == "Notebook 11"
)

assert (
    RELOADED_COMPLETION_MANIFEST[
        "calibration_source"
    ]
    == "Notebook 11 Section 7"
)

assert (
    RELOADED_COMPLETION_MANIFEST[
        "achieved_training_epsilon"
    ]
    == "DEFERRED_TO_NOTEBOOK_12"
)

assert (
    RELOADED_COMPLETION_MANIFEST[
        "end_to_end_privacy_claim"
    ]
    is False
)

assert (
    RELOADED_COMPLETION_MANIFEST[
        "overall_status"
    ]
    == "PASS"
)

print(
    "✓ Completion manifest reload validation: PASS"
)


# --------------------------------------------------------------------------------------------------
# 27. FINAL DISPLAY
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("NOTEBOOK 11 — FINAL STATUS")
print("=" * 100)

print(f"Framework              : {FRAMEWORK_NAME}")
print(f"Datasets               : {len(EXPECTED_DATASETS)}")
print("Privacy mechanism      : DP-SGD")
print("Protected component    : SPP-GAN discriminator / critic")
print(f"Accountant             : {ACCOUNTANT_TYPE}")
print(f"Sampling               : {SAMPLING}")
print(f"Clipping               : {CLIPPING}")
print(f"Target epsilon         : {TARGET_EPSILON:.6f}")
print(f"Delta                  : {DELTA:.1e}")
print(f"Maximum gradient norm  : {MAX_GRAD_NORM:.6f}")
print(f"DP batch size          : {DP_BATCH_SIZE}")
print(f"DP epochs              : {DP_EPOCHS}")


# --------------------------------------------------------------------------------------------------
# 28. CONFIGURED-SCHEDULE DISPLAY
# --------------------------------------------------------------------------------------------------

print()
print("CONFIGURED-SCHEDULE EPSILON")
print("-" * 100)

for dataset in EXPECTED_DATASETS:

    epsilon = float(
        CONFIGURED_EPSILON_SERIES.loc[dataset]
    )

    delta = float(
        CONFIGURED_DELTA_SERIES.loc[dataset]
    )

    alpha = float(
        CONFIGURED_ALPHA_SERIES.loc[dataset]
    )

    abs_diff = float(
        CONFIGURED_ABS_DIFF_SERIES.loc[dataset]
    )

    status = (
        "WITHIN_TARGET"
        if epsilon <= TARGET_EPSILON
        else "EXCEEDS_TARGET"
    )

    print(
        f"{dataset:<20} "
        f"epsilon={epsilon:.6f} "
        f"delta={delta:.1e} "
        f"alpha={alpha:.2f} "
        f"|Δε|={abs_diff:.6f} "
        f"{status}"
    )


# --------------------------------------------------------------------------------------------------
# 29. CALIBRATED PRIVACY DISPLAY
# --------------------------------------------------------------------------------------------------

print()
print("CALIBRATED PRIVACY SCHEDULE")
print("-" * 100)

for dataset in EXPECTED_DATASETS:

    sigma = float(
        CALIBRATED_SIGMA_SERIES.loc[dataset]
    )

    epsilon = float(
        CALIBRATED_EPSILON_SERIES.loc[dataset]
    )

    alpha = float(
        CALIBRATED_ALPHA_SERIES.loc[dataset]
    )

    print(
        f"{dataset:<20} "
        f"sigma={sigma:.6f} "
        f"epsilon={epsilon:.10f} "
        f"alpha={alpha:.2f} "
        f"WITHIN_TARGET"
    )


# --------------------------------------------------------------------------------------------------
# 30. FINAL STATUS
# --------------------------------------------------------------------------------------------------

print()
print("Configured-schedule accounting : COMPLETED")
print("Target comparison              : COMPLETED")
print("Noise calibration              : COMPLETED")
print("Calibration authority          : NOTEBOOK 11 SECTION 7")
print("Training                       : NOT PERFORMED")
print("Achieved training epsilon      : DEFERRED TO NOTEBOOK 12")
print("Synthetic generation           : NOT PERFORMED")
print("End-to-end privacy claim       : NOT ESTABLISHED")

print()
print(
    f"Configured datasets within target    : "
    f"{CONFIGURED_WITHIN_TARGET}"
)

print(
    f"Configured datasets exceeding target : "
    f"{CONFIGURED_EXCEEDING_TARGET}"
)

print(
    f"Calibrated datasets within target    : "
    f"{CALIBRATED_DATASETS_WITHIN_TARGET}"
)

print(
    f"Calibrated datasets exceeding target : "
    f"{CALIBRATED_DATASETS_EXCEEDING_TARGET}"
)

print(
    f"Final verification checks            : "
    f"{FINAL_VERIFICATION_PASS}/"
    f"{FINAL_VERIFICATION_TOTAL} PASS"
)

print(
    f"Calibrated RDP-order column         : "
    f"{RDP_ORDER_COLUMN}"
)

print("Required artifacts              : VALIDATED")
print("Calibrated privacy artifacts    : VALIDATED")
print("Completion manifest             : SAVED")
print("Overall status                  : PASS")
print("Next Notebook                   : Notebook 12 — SPP-GAN DP Training")

print("=" * 100)
print("SECTION 18 STATUS: PASS")
print("=" * 100)

18. COMPLETION SUMMARY
✓ Canonical Notebook 11 paths established
✓ Project root   : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 11    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_11
✓ All required persisted Notebook 11 artifacts found
✓ Configured-schedule accounting loaded and validated
✓ Configuration reconstructed from persisted accounting
✓ DP configuration validated
✓ Dataset privacy summary loaded
✓ Model-level privacy summary loaded
✓ Privacy audit loaded
✓ Final privacy verification loaded: 88/88 PASS

CALIBRATED PRIVACY ARTIFACT SCHEMA
----------------------------------------------------------------------------------------------------
Columns:
  - dataset
  - n_train
  - batch_size
  - sample_rate
  - epochs
  - steps_per_epoch
  - total_steps
  - target_epsilon
  - calibrated_epsilon
  - optimal_rdp_order
  - delta
  - calibrated_noise_multiplier
  - max_grad_norm
  - accountant
  - sampling
  - clipping
  - loss_reduction
  - mechanism
  - prot